# 06 - Study 2: counterfactual ranking (main analysis)

Biographies are rewritten to carry less communal or less agentic wording, and
each rewritten version is ranked against real job advertisements. The question
is whether the wording alone moves a candidate's position in the ranking.

- **Question:** Does a biography move in the ranking of a job advertisement
  when only its agentic or communal wording is changed, and does the move
  depend on the wording of the advertisement?
- **Expectation:** removing communal wording raises the rank, removing agentic
  wording lowers it; T1 expects the gap between women and men in the original
  rankings to grow with the communal wording of the advertisement, T2 expects
  the slope of a biography over the advertisements to grow with the balance of
  that biography.
- **Status:** main analysis of Study 2, the final run. Dictionary A with
  substitution list v2 after the rater validation; A $\cup$ B and substitution
  list v1 are the variants of the earlier run that section 16 compares
  against.
- **Unit and test:** biography against advertisement; percentile rank of the
  cosine, Wilcoxon signed-rank for the change, and for T1 and T2 a Spearman
  correlation within the occupation with a permutation test stratified by
  occupation, 5,000 permutations in the full run, seed 42, alpha = 0.025 per
  test.
- **Inputs:** `outputs/df_features.parquet`, `outputs/df_features_lexAB.parquet`,
  `data/job_descriptions/job_postings.csv`, `config/substitution_rules_v2.csv`,
  `config/fixed_terms_v2.txt`, `config/substitution_rules_v2_validation.csv`,
  and for Arm 2 the embeddings in `outputs/study2_run5/arm2/`.
- **Outputs:** the tables and figures listed in the last section, under
  `outputs/study2_run5/`.
- **Run:** from a fresh kernel, top to bottom, from the project root.
  `SMOKE = True` writes a test run to `outputs/study2_run5/smoke13/`;
  `SMOKE = False` is the full run.
- **Decisions:** D1 to D7, fixed on 2026-10-11 (methodology appendix).
  Pronouns and modal verbs are never replaced (D1), and a rewritten biography
  only counts when it keeps its content above the gate of 0.90.


## 1. Settings

Every parameter of the run, with one comment per parameter.

In [1]:
# Settings of the run. One comment per parameter.
from pathlib import Path as _Path
import os as _os
if _Path.cwd().name == "notebooks":
    _os.chdir("..")
# The encoder is read from the local cache of the model hub. Without this the
# library asks the hub for a newer revision every time a model is loaded, which
# costs time and prints a warning when there is no connection.
_os.environ["HF_HUB_OFFLINE"] = "1"
_os.environ["TRANSFORMERS_OFFLINE"] = "1"
del _Path, _os

# Run 5 settings, all fixed before any result
SMOKE          = False          # smoke13: all 13 occupations, small case numbers
RUN_TAG        = "run5"
PRIMARY_MODEL  = "sentence-transformers/all-mpnet-base-v2"
JD_REPR        = "mean"
GATES          = [None, 0.85, 0.90, 0.95]
PRIMARY_GATE   = 0.90
FOCAL_N        = 250
FOCAL_MIN_COUNT   = 2
FOCAL_RATE_QUANTILE = 0.75
N_PERM, SEED   = 5000, 42

# Dictionary: A is the main analysis from 2026-10-10, A u B the robustness variant.
LEX_MAIN   = "A"
LEX_ROBUST = "AB"
# Manipulation: list v2, after the rater validation of D7.
MANIP      = "v2"
RULES_FILE       = "config/substitution_rules_v2.csv"
FIXED_TERMS_FILE = "config/fixed_terms_v2.txt"
VALIDATION_FILE  = "config/substitution_rules_v2_validation.csv"

KEPT_13 = [
    "nurse", "software_engineer", "accountant", "architect", "psychologist",
    "attorney", "teacher", "surgeon", "physician", "professor", "dentist",
    "paralegal", "dietitian",
]
N_BIOS_FULL = 250
N_JDS_FULL  = 250
RANDOM_STATE = 42
BATCH_SIZE   = 64
CP_THRESHOLD = PRIMARY_GATE

# Test run only: the candidates of the focal selection are drawn down to this
# number per occupation and arm, with the seed above. The full run ranks every
# candidate, which is what takes the time there.
FOCAL_CAND_SMOKE = 300

if SMOKE:
    # Test run: every occupation, few cases, few permutations.
    N_BIOS_FULL = 30
    N_JDS_FULL  = 20
    FOCAL_N     = 20
    N_PERM      = 200

OUT_BASE   = f"outputs/study2_{RUN_TAG}/smoke13" if SMOKE else f"outputs/study2_{RUN_TAG}"
# The test run keeps its embeddings apart. The tag of an array says which
# occupation and which arm it belongs to, not how many texts went into it, so a
# test run and a full run would write to the same file. The key would catch it,
# because it holds the hash of the texts, but the array of the full run would be
# gone and would have to be computed again, which is hours of work. The format
# of the key is unchanged.
CACHE_BASE = ("outputs/study2_cache/smoke13" if SMOKE
              else "outputs/study2_cache")
ARM2_DIR   = f"outputs/study2_{RUN_TAG}/arm2"
RUN4_BASE  = "outputs/study2_results/run4"

print(f"SMOKE={SMOKE}  model={PRIMARY_MODEL}  gate={PRIMARY_GATE}  jd_repr={JD_REPR}")
print(f"RUN_TAG={RUN_TAG}  lexicon={LEX_MAIN} (robustness {LEX_ROBUST})  manipulation={MANIP}")
print(f"OUT_BASE={OUT_BASE}")
print(f"Occupations: {len(KEPT_13)}  bios per occupation: {N_BIOS_FULL}  "
      f"jds per occupation: {N_JDS_FULL}  N_PERM={N_PERM}")

SMOKE=False  model=sentence-transformers/all-mpnet-base-v2  gate=0.9  jd_repr=mean
RUN_TAG=run5  lexicon=A (robustness AB)  manipulation=v2
OUT_BASE=outputs/study2_run5
Occupations: 13  bios per occupation: 250  jds per occupation: 250  N_PERM=5000


## 2. Imports

Libraries, the two dictionaries, and the spaCy pipelines: the full pipeline for the manipulation, a tokenizer-only pipeline for counting.

In [2]:
# Libraries, the two dictionaries, and the two spaCy pipelines.
import hashlib, importlib, json, os, sys, time, warnings
import numpy as np
import pandas as pd
import spacy
from pathlib import Path
from scipy.stats import pearsonr, spearmanr, wilcoxon
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
from sentence_transformers import SentenceTransformer
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# The tokenizer counts the tokens of a text before it is cut into windows and
# reports every text that is longer than the input of the model. That length is
# exactly what the windowing deals with, so this one message of the tokenizer is
# silenced; every other message of the library stays visible.
import logging as _logging
_logging.getLogger("transformers.tokenization_utils_base").setLevel(_logging.ERROR)
sys.path.insert(0, str(Path.cwd()))

import importlib.util
_core_path = Path("src/study2_core.py")
_spec = importlib.util.spec_from_file_location("study2_core", _core_path)
study2_core = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(study2_core)

from src.lexicon import load_lexicons
import src.substitution_v2 as _sub_mod
importlib.reload(_sub_mod)
from src.substitution_v2 import (UNTREATED_REASONS, SubstitutionEngine,
                                 apply_validation, check_rules, load_fixed_terms,
                                 load_rules)
from src.features import agentic_communal_ratio as _acr_fn  # for JD density

# The v2 list works on the parsed original text: part of speech, dependencies,
# named entities and sentence boundaries are all needed, so the full pipeline of
# en_core_web_lg is loaded. Counting lexicon hits only needs token surfaces, so a
# tokenizer-only pipeline is used for the job descriptions.
nlp       = spacy.load("en_core_web_lg")
nlp_count = spacy.load("en_core_web_lg", disable=["parser", "ner", "lemmatizer",
                                                  "tagger", "attribute_ruler"])

lex    = load_lexicons(version=LEX_MAIN)     # dictionary A, main analysis
lex_ab = load_lexicons(version=LEX_ROBUST)   # A u B, robustness variant

print(f"spaCy pipeline (manipulation): {nlp.pipe_names}")
print(f"spaCy pipeline (counting)    : {nlp_count.pipe_names}")
print(f"lexicons loaded: main={LEX_MAIN}, robustness={LEX_ROBUST}")

Path(OUT_BASE).mkdir(parents=True, exist_ok=True)
Path(CACHE_BASE).mkdir(parents=True, exist_ok=True)
print("Imports done.")

spaCy pipeline (manipulation): ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
spaCy pipeline (counting)    : ['tok2vec']
lexicons loaded: main=A, robustness=AB
Imports done.


## 3. Substitution list v2

The rule table is loaded, each rule is checked against the dictionary, and the rules that failed the rater validation are switched off.

In [3]:
# Substitution list v2: load, run the checks of apply D7.
rules_v2    = load_rules(RULES_FILE)
fixed_terms = load_fixed_terms(FIXED_TERMS_FILE)
print(f"rules loaded      : {len(rules_v2)} from {RULES_FILE}")
print(f"fixed terms loaded: {len(fixed_terms)} from {FIXED_TERMS_FILE}")

# Every source word matches dictionary A in its arm; every candidate is
# neutral under A u B, so the replacement is neutral under both versions.
_check = check_rules(rules_v2, lex, lex_ab)
print(f"source word not in dictionary {LEX_MAIN} in its arm (expected 0): "
      f"{len(_check['source_failures'])}")
for _rule in _check["source_failures"]:
    print(f"    {_rule.rule_id} {_rule.arm} {_rule.source_word}")
print(f"candidates rejected as not neutral under {LEX_ROBUST}: "
      f"{len(_check['candidate_failures'])}")
for _rule_id, _failure in _check["candidate_failures"]:
    print(f"    {_rule_id} {_failure}")
print(f"rules without a neutral candidate (occurrences untreated): "
      f"{len(_check['no_candidate'])}")
print(f"delete rules without a replacement fallback: "
      f"{len(_check['delete_without_fallback'])} "
      f"{[r.rule_id for r in _check['delete_without_fallback']]}")

# D7: rules that failed the rater validation are switched off.
_val_path = Path(VALIDATION_FILE)
if _val_path.exists():
    _validation = pd.read_csv(_val_path)
    _failed = set(_validation.loc[~_validation["passed"].astype(bool), "rule_id"])
    apply_validation(rules_v2, _failed)
    print(f"validation loaded : {len(_validation)} rated rules, "
          f"{len(_failed)} failed D7 and switched off {sorted(_failed)}")
else:
    raise FileNotFoundError(
        f"{VALIDATION_FILE} not found. The manipulation is fixed as v2 after "
        f"the validation of D7; run scripts/apply_validation_v2.py first.")

engine_v2 = SubstitutionEngine(rules_v2, fixed_terms, lex)
print(f"engine ready, source words covered: {len(engine_v2.by_word)}")

rules loaded      : 252 from config/substitution_rules_v2.csv
fixed terms loaded: 125 from config/fixed_terms_v2.txt
source word not in dictionary A in its arm (expected 0): 0
candidates rejected as not neutral under AB: 0
rules without a neutral candidate (occurrences untreated): 0
delete rules without a replacement fallback: 8 ['C126', 'C135', 'C137', 'A018', 'A044', 'A046', 'A068', 'A074']
validation loaded : 54 rated rules, 3 failed D7 and switched off ['A004', 'A007', 'C005']
engine ready, source words covered: 196


## 4. Samples

The pool is 250 biographies per occupation. The focal sample takes, per occupation and arm, the biographies with the most occurrences that the list actually treats (D4).

In [4]:
# Biography features under both dictionaries, and the pool of each occupation.
import re as _re

# Dictionary A is the main analysis and lives in df_features.parquet; A u B is the
# robustness variant and lives in df_features_lexAB.parquet, where the plain
# columns are A u B and the columns with suffix _A repeat dictionary A.
df_features = pd.read_parquet("outputs/df_features.parquet")
df_ab = pd.read_parquet("outputs/df_features_lexAB.parquet",
                        columns=["agency_count", "communion_count",
                                 "agency_count_A", "communion_count_A"])

# Control: the A columns of both files have to agree exactly.
_n_diff_ag = int((df_features["agency_count"] != df_ab["agency_count_A"]).sum())
_n_diff_cm = int((df_features["communion_count"] != df_ab["communion_count_A"]).sum())
print(f"dictionary {LEX_MAIN} counts identical in both files "
      f"(expected 0): agency {_n_diff_ag}, communion {_n_diff_cm}")
if _n_diff_ag or _n_diff_cm:
    raise AssertionError("The A columns of the two feature files disagree.")

df_s2 = df_features[df_features["occupation"].isin(KEPT_13)].copy()
df_s2["female"] = (df_s2["gender_label"] == "Female").astype(int)

# Rates per 100 words, for both versions. Suffix _A is the main analysis, _AB the
# robustness variant; the unsuffixed names stay on the main analysis so that the
# cells further down keep working.
df_s2["agency_count_AB"]    = df_ab.loc[df_s2.index, "agency_count"]
df_s2["communion_count_AB"] = df_ab.loc[df_s2.index, "communion_count"]
for _suffix, _ag, _cm in [("", "agency_count", "communion_count"),
                          ("_AB", "agency_count_AB", "communion_count_AB")]:
    df_s2[f"agency_rate{_suffix}"]    = df_s2[_ag] / df_s2["word_count"] * 100
    df_s2[f"communion_rate{_suffix}"] = df_s2[_cm] / df_s2["word_count"] * 100
    df_s2[f"bio_balance{_suffix}"] = (df_s2[f"communion_rate{_suffix}"]
                                      - df_s2[f"agency_rate{_suffix}"])
print(f"df_s2: {len(df_s2)} bios in {df_s2['occupation'].nunique()} occupations")
print(f"  mean communion_rate  {LEX_MAIN}: {df_s2['communion_rate'].mean():.3f}  "
      f"{LEX_ROBUST}: {df_s2['communion_rate_AB'].mean():.3f}")
print(f"  mean agency_rate     {LEX_MAIN}: {df_s2['agency_rate'].mean():.3f}  "
      f"{LEX_ROBUST}: {df_s2['agency_rate_AB'].mean():.3f}")

JD_RAW_PATH = "data/job_descriptions/job_postings.csv"
# The file holds 31 columns in 493 MB, and reading all of them costs several
# gigabytes. The list of columns is read from the header alone, so the report of
# the origin can print it in full, and the data are loaded with the columns that
# this notebook uses: the title for the match to an occupation, the text itself,
# four columns for the origin and the four time columns for the dates.
JD_RAW_COLUMNS = list(pd.read_csv(JD_RAW_PATH, nrows=0).columns)
JD_RAW_COLUMNS_USED = ["title", "description",
                       "location", "company_id", "zip_code", "fips",
                       "listed_time", "original_listed_time", "expiry",
                       "closed_time"]
_columns_missing = [c for c in JD_RAW_COLUMNS_USED if c not in JD_RAW_COLUMNS]
print(f"control columns of {JD_RAW_PATH}: {len(JD_RAW_COLUMNS)} in the file, "
      f"{len(JD_RAW_COLUMNS_USED)} loaded, missing {_columns_missing} "
      f"(expected [])")
if _columns_missing:
    raise AssertionError(
        f"{JD_RAW_PATH} is missing {_columns_missing}; the sections for the "
        f"match to an occupation, the origin and the dates need them.")
df_jd_raw = pd.read_csv(JD_RAW_PATH, usecols=JD_RAW_COLUMNS_USED,
                        low_memory=False)
OCC_KEYWORDS = {
    'nurse':             [r'\bnurse\b', r'\bnursing\b', r'\brn\b', r'\bnurse practitioner\b'],
    'software_engineer': [r'\bsoftware engineer\b', r'\bsoftware developer\b', r'\bprogrammer\b',
                         r'\bfull.?stack\b', r'\bbackend engineer\b', r'\bfrontend engineer\b'],
    'accountant':        [r'\baccountant\b', r'\bcpa\b', r'\bbookkeeper\b', r'\baccounting\b'],
    'architect':         [r'\barchitect\b', r'\barchitecture\b'],
    'psychologist':      [r'\bpsychologist\b', r'\btherapist\b', r'\bcounselor\b'],
    'attorney':          [r'\battorney\b', r'\blawyer\b', r'\bcounsel\b'],
    'teacher':           [r'\bteacher\b', r'\beducator\b', r'\bk-12\b'],
    'surgeon':           [r'\bsurgeon\b', r'\bsurgery\b', r'\bsurgical\b'],
    'physician':         [r'\bphysician\b', r'\bmedical doctor\b', r'\binternist\b'],
    'professor':         [r'\bprofessor\b', r'\bfaculty\b', r'\blecturer\b'],
    'dentist':           [r'\bdentist\b', r'\bdental\b', r'\bdds\b'],
    'paralegal':         [r'\bparalegal\b', r'\blegal assistant\b'],
    'dietitian':         [r'\bdietitian\b', r'\bnutritionist\b', r'\bregistered dietitian\b'],
}
title_lower = df_jd_raw["title"].fillna("").str.lower()
df_jd_raw["occupation"] = None
for occ, patterns in OCC_KEYWORDS.items():
    mask = title_lower.str.contains("|".join(patterns), regex=True, na=False)
    df_jd_raw.loc[mask, "occupation"] = occ
df_jd_matched = df_jd_raw[df_jd_raw["occupation"].notna()].copy()

# Reference counts from notebook 05 (title matched, before the N_JDS_FULL cap).
JD_REF_AVAIL = {
    'nurse': 6999, 'software_engineer': 2004, 'accountant': 2210,
    'architect': 1308, 'psychologist': 1762, 'attorney': 958,
    'teacher': 908, 'surgeon': 700, 'physician': 686,
    'professor': 148, 'dentist': 415, 'paralegal': 446, 'dietitian': 77,
}

bio_pools = {}
jd_pools  = {}
print()
print(f"{'Occupation':<22}  {'#bios_avail':>12}  {'#jds_avail':>10}  "
      f"{'n_bio':>6}  {'n_jd':>6}")
print("-" * 65)
jd_avail = {}
for occ in KEPT_13:
    bio_sub = df_s2[df_s2["occupation"] == occ].copy()
    jd_sub  = df_jd_matched[df_jd_matched["occupation"] == occ][["description"]].dropna().copy()
    jd_avail[occ] = len(jd_sub)
    n_bio = min(N_BIOS_FULL, len(bio_sub))
    n_jd  = min(N_JDS_FULL,  len(jd_sub))
    bio_pools[occ] = bio_sub.sample(n_bio, random_state=RANDOM_STATE).reset_index(drop=True)
    jd_pools[occ]  = jd_sub.sample(n_jd,  random_state=RANDOM_STATE).reset_index(drop=True)
    print(f"  {occ:<22}  {len(bio_sub):>12,}  {len(jd_sub):>10,}  "
          f"{n_bio:>6}  {n_jd:>6}")

# Validate JD counts in full mode
if not SMOKE:
    mismatches = [(o, jd_avail[o], JD_REF_AVAIL[o])
                  for o in KEPT_13 if jd_avail[o] != JD_REF_AVAIL[o]]
    if mismatches:
        for o, got, exp in mismatches:
            print(f"  MISMATCH {o}: got {got}, expected {exp}")
        raise AssertionError("JD available counts do not match Notebook 05 - check OCC_KEYWORDS")
    print("JD counts validated against Notebook 05 reference.")

dictionary A counts identical in both files (expected 0): agency 0, communion 0
df_s2: 301109 bios in 13 occupations
  mean communion_rate  A: 3.827  AB: 4.131
  mean agency_rate     A: 1.643  AB: 2.627
control columns of data/job_descriptions/job_postings.csv: 31 in the file, 10 loaded, missing [] (expected [])

Occupation               #bios_avail  #jds_avail   n_bio    n_jd
-----------------------------------------------------------------
  nurse                         18,928       6,999     250     250
  software_engineer              6,886       2,004     250     250
  accountant                     5,619       2,210     250     250
  architect                     10,081       1,308     250     250
  psychologist                  18,354       1,762     250     250
  attorney                      32,460         958     250     250
  teacher                       16,169         908     250     250
  surgeon                       13,565         700     250     250
  physician       

In [5]:
# D4: per occupation and arm the biographies with the most occurrences that v2
# actually treats (replaced or deleted), with the same case number as Run 4.
# The lexical pre-filter of Run 4 selects the candidates, so that v2 only has to
# run on bios with a real signal in the target dimension; the ranking inside the
# candidate set is then by treated occurrences.
focal_bios = {}
_focal_rows = []

print(f"{'Occupation':<22}  {'Arm':>9}  {'cand':>6}  {'focal':>6}  "
      f"{'mean_treated':>12}")
print("-" * 62)
for occ in KEPT_13:
    _pool_sample = df_s2[df_s2["occupation"] == occ].sample(
        min(N_BIOS_FULL, len(df_s2[df_s2["occupation"] == occ])),
        random_state=RANDOM_STATE
    )
    non_pool = df_s2[df_s2["occupation"] == occ].drop(index=_pool_sample.index)
    focal_bios[occ] = {}
    for arm, rate_col, count_col in [
        ("communion", "communion_rate", "communion_count"),
        ("agency",    "agency_rate",    "agency_count"),
    ]:
        q = non_pool[rate_col].quantile(FOCAL_RATE_QUANTILE)
        cand = non_pool[
            (non_pool[count_col] >= FOCAL_MIN_COUNT) &
            (non_pool[rate_col]  >= q)
        ].copy()
        if cand.empty:
            focal_bios[occ][arm] = cand.reset_index(drop=True)
            print(f"  {occ:<22}  {arm:>9}  {0:>6}  {0:>6}  {0:>12.2f}")
            continue
        # The test run only has to exercise the code, so it ranks a random draw
        # of the candidates. The full run ranks all of them.
        if SMOKE and len(cand) > FOCAL_CAND_SMOKE:
            cand = cand.sample(FOCAL_CAND_SMOKE, random_state=RANDOM_STATE)
            print(f"  {occ:<22}  {arm:>9}  candidates drawn down to "
                  f"{len(cand)} for the test run")
        _treated = []
        for _doc in nlp.pipe(cand["text_clean"].tolist(), batch_size=BATCH_SIZE):
            _res = engine_v2.apply(_doc, arm)
            _treated.append(_res.n_replaced + _res.n_deleted)
        cand["n_treated_v2"] = _treated
        cand = cand.sort_values("n_treated_v2", ascending=False)
        take = min(FOCAL_N, len(cand)) if FOCAL_N > 0 else len(cand)
        chosen = cand.head(take).reset_index(drop=True)
        focal_bios[occ][arm] = chosen
        print(f"  {occ:<22}  {arm:>9}  {len(cand):>6}  {len(chosen):>6}  "
              f"{chosen['n_treated_v2'].mean():>12.2f}")
        _focal_rows.append({"occ": occ, "arm": arm, "n_candidates": len(cand),
                            "n_focal": len(chosen),
                            "mean_treated": float(chosen["n_treated_v2"].mean()),
                            "min_treated": int(chosen["n_treated_v2"].min()),
                            "max_treated": int(chosen["n_treated_v2"].max())})
df_focal_selection = pd.DataFrame(_focal_rows)

Occupation                    Arm    cand   focal  mean_treated
--------------------------------------------------------------
  nurse                   communion    4693     250          5.40
  nurse                      agency    2444     250          3.26
  software_engineer       communion    1578     250          3.43
  software_engineer          agency    1008     250          2.26
  accountant              communion    1363     250          3.19
  accountant                 agency    1066     250          2.85
  architect               communion    2280     250          3.83
  architect                  agency    1859     250          3.10
  psychologist            communion    4539     250          5.72
  psychologist               agency    3556     250          3.52
  attorney                communion    8116     250          5.34
  attorney                   agency    6679     250          4.02
  teacher                 communion    4016     250          6.70
  teacher      

In [6]:
# % female computed from df_s2 (not hardcoded)
PCT_FEMALE = {
    occ: df_s2[df_s2["occupation"] == occ]["female"].mean() * 100
    for occ in KEPT_13
}

rows = []
for occ in KEPT_13:
    for arm, count_col, rate_col in [
        ("communion", "communion_count", "communion_rate"),
        ("agency",    "agency_count",    "agency_rate"),
    ]:
        pool_df  = bio_pools[occ]
        focal_df = focal_bios[occ][arm]
        rows.append({
            "occ": occ, "arm": arm,
            "pool_n":  len(pool_df),   "focal_n":  len(focal_df),
            "pool_pct_female":  pool_df["female"].mean()  * 100 if "female" in pool_df.columns  else np.nan,
            "focal_pct_female": focal_df["female"].mean() * 100 if "female" in focal_df.columns else np.nan,
            "pool_mean_hits":   pool_df[count_col].mean() if count_col in pool_df.columns  else np.nan,
            "focal_mean_hits":  focal_df[count_col].mean()if count_col in focal_df.columns else np.nan,
            "pool_mean_rate":   pool_df[rate_col].mean()  if rate_col  in pool_df.columns  else np.nan,
            "focal_mean_rate":  focal_df[rate_col].mean() if rate_col  in focal_df.columns else np.nan,
        })
df_pool_focal_cmp = pd.DataFrame(rows)
print("% female per occupation (from df_s2):")
for occ in KEPT_13:
    print(f"  {occ:<22}: {PCT_FEMALE[occ]:.1f}%")
print()
print(df_pool_focal_cmp[["occ","arm","pool_n","focal_n",
    "pool_pct_female","focal_pct_female",
    "pool_mean_hits","focal_mean_hits"]].to_string(index=False))

% female per occupation (from df_s2):
  nurse                 : 90.8%
  software_engineer     : 15.8%
  accountant            : 36.7%
  architect             : 23.7%
  psychologist          : 62.1%
  attorney              : 38.2%
  teacher               : 60.3%
  surgeon               : 14.8%
  physician             : 49.4%
  professor             : 45.1%
  dentist               : 35.3%
  paralegal             : 84.8%
  dietitian             : 92.8%

              occ       arm  pool_n  focal_n  pool_pct_female  focal_pct_female  pool_mean_hits  focal_mean_hits
            nurse communion     250      250             92.8              93.6           2.720            8.472
            nurse    agency     250      250             92.8              94.8           0.608            4.688
software_engineer communion     250      250             16.8              29.6           1.300            4.984
software_engineer    agency     250      250             16.8              14.8           0.6

## 5. Job descriptions

- **Question:** How are communal and agentic words spread over the job
  advertisements of the 13 occupations, and does that go with the field and the
  share of women?
- **Test:** job description; Kruskal-Wallis with $\eta^2$ across occupations, an
  exact permutation test between the two fields, and correlations with the share
  of women.
- **Status:** descriptive, fixed 2026-10-05.
- **Output:** `study2_jd_origin.csv`, `study2_jd_eda_by_occupation.csv`,
  `study2_jd_eda_tests.csv`, `study2_jd_top_lexicon_words.csv`, the figures in
  `outputs/figures/study2/`

Densities are computed here once, for dictionary A and for A $\cup$ B, each with
the split into the part without the pronouns and modal verbs of D1 (`_cw`) and
the part that was taken out (`_fw`).

For orientation, Pietraszkiewicz et al. (2019, Study 4; 1,291 Monster.com
advertisements from 2016) report, with their dictionaries, agency at 5.39% and
communion at 3.40% of words for male-dominated professions (engineer, computer
programmer), and 4.28% and 6.43% for female-dominated professions (early
childhood educator, registered nurse).

In [7]:
# Occupation metadata, and the job descriptions drawn again as whole rows.
_meta_path = Path("config/occupation_metadata.csv")
if not _meta_path.exists():
    raise FileNotFoundError(
        f"{_meta_path} not found. The job description section needs the field and "
        f"the share of women per occupation; there is no placeholder.")
_occ_meta = pd.read_csv(_meta_path)
_missing_occ = [o for o in KEPT_13 if o not in _occ_meta["occ"].values]
_needed_cols = ["field", "pct_women_bls_2025"]
_absent_cols = [c for c in _needed_cols if c not in _occ_meta.columns]
_na_cols = [c for c in _needed_cols if c in _occ_meta.columns
            and _occ_meta.loc[_occ_meta["occ"].isin(KEPT_13), c].isna().any()]
print(f"occupation metadata: {len(_occ_meta)} rows, columns {list(_occ_meta.columns)}")
print(f"control occupations missing from the metadata: {_missing_occ} (expected [])")
print(f"control columns absent: {_absent_cols} (expected [])")
print(f"control columns with a missing value: {_na_cols} (expected [])")
if _missing_occ or _absent_cols or _na_cols:
    raise AssertionError(
        f"The occupation metadata is incomplete: missing occupations "
        f"{_missing_occ}, absent columns {_absent_cols}, columns with a missing "
        f"value {_na_cols}.")

jd_pools_full = {}
_resample_mismatches = 0
for occ in KEPT_13:
    _jd_sub_full = df_jd_matched[df_jd_matched["occupation"] == occ].dropna(subset=["description"])
    _n_jd = min(N_JDS_FULL, len(_jd_sub_full))
    jd_pools_full[occ] = _jd_sub_full.sample(_n_jd, random_state=RANDOM_STATE)
    _desc_new  = jd_pools_full[occ]["description"].reset_index(drop=True)
    _desc_orig = jd_pools[occ]["description"]
    _n_mismatch = int((_desc_new != _desc_orig).sum())
    print(f"control {occ}: mismatches: {_n_mismatch} of {len(_desc_orig)} "
          f"(expected 0)")
    _resample_mismatches += _n_mismatch
if _resample_mismatches:
    raise AssertionError(
        f"The repeated draw of the job descriptions differs in "
        f"{_resample_mismatches} rows; check random_state.")
print("job descriptions drawn again as whole rows, no mismatch")

occupation metadata: 13 rows, columns ['occ', 'field', 'field_detail', 'bls_title', 'pct_women_bls_2025', 'note', 'source']
control occupations missing from the metadata: [] (expected [])
control columns absent: [] (expected [])
control columns with a missing value: [] (expected [])
control nurse: mismatches: 0 of 250 (expected 0)
control software_engineer: mismatches: 0 of 250 (expected 0)
control accountant: mismatches: 0 of 250 (expected 0)
control architect: mismatches: 0 of 250 (expected 0)
control psychologist: mismatches: 0 of 250 (expected 0)
control attorney: mismatches: 0 of 250 (expected 0)
control teacher: mismatches: 0 of 250 (expected 0)
control surgeon: mismatches: 0 of 250 (expected 0)
control physician: mismatches: 0 of 250 (expected 0)
control professor: mismatches: 0 of 148 (expected 0)
control dentist: mismatches: 0 of 250 (expected 0)
control paralegal: mismatches: 0 of 250 (expected 0)
control dietitian: mismatches: 0 of 77 (expected 0)
job descriptions drawn agai

In [8]:
# Origin of the job descriptions: columns, dates, location and company seat.
print(f"source file: data/job_descriptions/job_postings.csv")
print(f"columns ({len(JD_RAW_COLUMNS)}): {JD_RAW_COLUMNS}")
print(f"columns loaded ({len(JD_RAW_COLUMNS_USED)}): {JD_RAW_COLUMNS_USED}")

# The time columns hold milliseconds since 1970. They are converted on
# jd_pools_full, which is the frame this section reads.
TIME_COLUMNS = ["listed_time", "original_listed_time", "expiry", "closed_time"]
_present = [c for c in TIME_COLUMNS if c in JD_RAW_COLUMNS]
_needed = ["location", "company_id", "zip_code", "fips"]
_absent = [c for c in _needed + ["listed_time"] if c not in JD_RAW_COLUMNS]
print(f"control time columns present: {_present}")
print(f"control columns absent: {_absent} (expected [])")
if _absent:
    raise AssertionError(
        f"job_postings.csv is missing {_absent}; the date and location report "
        f"needs them.")

print()
print("first five raw values of the time columns:")
for _column in _present:
    print(f"  {_column:<22} {df_jd_raw[_column].head(5).tolist()}")

DATE_COLUMN = "listed_time_dt"
for occ in KEPT_13:
    for _column in _present:
        jd_pools_full[occ][_column + "_dt"] = pd.to_datetime(
            jd_pools_full[occ][_column], unit="ms", errors="coerce")
print(f"dates are read from {DATE_COLUMN}, formed on jd_pools_full")

# The rule by which a location counts as the United States.
US_ABBREVS = {
    'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN','IA','KS',
    'KY','LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ','NM','NY',
    'NC','ND','OH','OK','OR','PA','RI','SC','SD','TN','TX','UT','VT','VA','WA','WV',
    'WI','WY','DC'
}
def _is_us(loc):
    if not isinstance(loc, str):
        return False
    last = loc.strip().split(",")[-1].strip()
    return last in US_ABBREVS or "United States" in loc
print()
print("a location counts as the United States when the part after the last comma "
      "is one of the 51 state abbreviations, or the text contains 'United States'")

# The seat of the company. companies.csv leaves some countries as "0" or empty;
# those count as unknown rather than as a country.
_companies_path = Path("data/job_descriptions/companies/companies.csv")
if not _companies_path.exists():
    raise FileNotFoundError(
        f"{_companies_path} not found; the report of the company seat needs it.")
_companies = pd.read_csv(_companies_path,
                         usecols=["company_id", "country", "state", "city"])
_country_raw = _companies["country"].astype("string").str.strip()
_unknown_mask = _country_raw.isna() | _country_raw.isin(["", "0"])
_companies["country_clean"] = _country_raw.where(~_unknown_mask, "unknown")
print(f"company records: {len(_companies)}")
print(f"  country given as '0' or empty, counted as unknown: "
      f"{int(_unknown_mask.sum())} ({_unknown_mask.mean():.1%})")

_origin_rows = []
for occ in KEPT_13 + ["TOTAL"]:
    _df_o = (pd.concat([jd_pools_full[o] for o in KEPT_13], ignore_index=True)
             if occ == "TOTAL" else jd_pools_full[occ])
    _n = len(_df_o)
    _pct_us = float(_df_o["location"].apply(_is_us).mean())
    _dates = pd.to_datetime(_df_o[DATE_COLUMN], errors="coerce").dropna()
    _date_min = str(_dates.min().date()) if len(_dates) else None
    _date_max = str(_dates.max().date()) if len(_dates) else None

    _joined = _df_o[["company_id"]].merge(_companies, on="company_id", how="left")
    _no_record = float(_joined["country_clean"].isna().mean())
    _country = _joined["country_clean"].fillna("no record")
    _country_us = float((_country == "US").mean())
    _country_unknown = float((_country == "unknown").mean())

    _origin_rows.append({"occ": occ, "n_jds": _n, "pct_us_location": _pct_us,
                         "pct_company_country_us": _country_us,
                         "pct_company_country_unknown": _country_unknown,
                         "pct_company_no_record": _no_record,
                         "date_min": _date_min, "date_max": _date_max,
                         "has_zip": float(_df_o["zip_code"].notna().mean()),
                         "has_fips": float(_df_o["fips"].notna().mean())})
    print()
    print(f"  {occ}: n={_n}, location in the United States {_pct_us:.1%}, "
          f"dates {_date_min} to {_date_max}")
    print(f"    company seat in the United States {_country_us:.1%}, "
          f"country unknown {_country_unknown:.1%}, "
          f"without a company record {_no_record:.1%}")
    print(f"    five most frequent countries: "
          f"{dict(_country.value_counts().head(5))}")
    print(f"    five most frequent locations: "
          f"{dict(_df_o['location'].value_counts().head(5))}")
df_jd_origin = pd.DataFrame(_origin_rows)

# Job descriptions per month, over all occupations.
_all_pools = pd.concat([jd_pools_full[o] for o in KEPT_13], ignore_index=True)
_all_dates = pd.to_datetime(_all_pools[DATE_COLUMN], errors="coerce").dropna()
print()
print(f"control job descriptions with a date: {len(_all_dates)} of "
      f"{len(_all_pools)} (expected {len(_all_pools)})")
if len(_all_dates) != len(_all_pools):
    raise AssertionError(
        f"{len(_all_pools) - len(_all_dates)} job descriptions have no date; "
        f"check the unit of the time columns.")
print("job descriptions per month:")
for _period, _count in _all_dates.dt.to_period("M").value_counts().sort_index().items():
    print(f"  {_period}  {_count}")
_earliest_year = int(_all_dates.min().year)
print(f"control earliest year: {_earliest_year} (expected 2023 or later)")
if _earliest_year < 2023:
    raise AssertionError(
        f"The earliest year is {_earliest_year}, which is before 2023; check the "
        f"unit of the time columns.")

# The twenty most frequent locations that do not count as the United States.
_non_us = _all_pools.loc[~_all_pools["location"].apply(_is_us), "location"]
print()
print(f"locations not counted as the United States: {len(_non_us)} of "
      f"{len(_all_pools)}; the twenty most frequent:")
for _value, _count in _non_us.value_counts().head(20).items():
    print(f"  {_count:>5}  {_value}")

print()
print(f"study2_jd_origin.csv: {len(df_jd_origin)} rows")

source file: data/job_descriptions/job_postings.csv
columns (31): ['job_id', 'company_name', 'title', 'description', 'max_salary', 'pay_period', 'location', 'company_id', 'views', 'med_salary', 'min_salary', 'formatted_work_type', 'applies', 'original_listed_time', 'remote_allowed', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type', 'normalized_salary', 'zip_code', 'fips']
columns loaded (10): ['title', 'description', 'location', 'company_id', 'zip_code', 'fips', 'listed_time', 'original_listed_time', 'expiry', 'closed_time']
control time columns present: ['listed_time', 'original_listed_time', 'expiry', 'closed_time']
control columns absent: [] (expected [])

first five raw values of the time columns:
  listed_time            [1713397508000.0, 1712857887000.0, 1713277614000.0, 1712895812000.0, 1713451943000.0]
  origin

In [9]:
# Job description densities, under dictionary A (main) and A u B
# (robustness), each with the split of into the part without the D1
# function words (_cw) and the part that was taken out (_fw).
from src.lexicon import count_multiword_communion as _count_mw
from src.substitution_v2 import is_function_word_kept as _is_fw

print("Job description densities")
_no_pct = [o for o in KEPT_13 if o not in PCT_FEMALE]
print(f"control occupations with a share of women in the pool: "
      f"{len(KEPT_13) - len(_no_pct)} of {len(KEPT_13)} (expected all)")
if _no_pct:
    raise AssertionError(
        f"No share of women in the pool for {_no_pct}. The pool section has to "
        f"run before this one.")

def _jd_hits(doc, text, lexicon):
    """Agency and communion hits, and how many of them are D1 function words."""
    _ag = [t.lower_ for t in doc if lexicon.match_agency(t.lower_)]
    _cm = [t.lower_ for t in doc if lexicon.match_communion(t.lower_)]
    _mw = _count_mw(text.lower(), lexicon.multi_word_communion)
    return (len(_ag), len(_cm) + _mw,
            sum(1 for t in _ag if _is_fw(t)),
            sum(1 for t in _cm if _is_fw(t)))

jd_comm_dens, jd_ag_dens, jd_n_words = {}, {}, {}
jd_comm_dens_cw, jd_ag_dens_cw = {}, {}
jd_comm_dens_ab, jd_ag_dens_ab = {}, {}
_eda_rows = []
_control_mismatch = 0

for occ in KEPT_13:
    jd_texts = jd_pools[occ]["description"].tolist()
    _cd, _ad, _nw = [], [], []
    _cd_cw, _ad_cw, _cd_ab, _ad_ab = [], [], [], []
    for j, (text, doc) in enumerate(zip(jd_texts,
                                        nlp_count.pipe(jd_texts, batch_size=BATCH_SIZE))):
        wc = max(len(text.split()), 1)
        _ag_a, _cm_a, _ag_fw, _cm_fw = _jd_hits(doc, text, lex)
        _ag_b, _cm_b, _, _           = _jd_hits(doc, text, lex_ab)

        # Control: the same counts as agentic_communal_ratio under dictionary A.
        _ref_ag, _ref_cm, _ = _acr_fn(doc, text, lex)
        if (_ref_ag, _ref_cm) != (_ag_a, _cm_a):
            _control_mismatch += 1

        _cd.append(_cm_a / wc * 100);        _ad.append(_ag_a / wc * 100)
        _cd_cw.append((_cm_a - _cm_fw) / wc * 100)
        _ad_cw.append((_ag_a - _ag_fw) / wc * 100)
        _cd_ab.append(_cm_b / wc * 100);     _ad_ab.append(_ag_b / wc * 100)
        _nw.append(wc)

        _row = {"occ": occ, "jd_idx": j, "n_words": wc,
                "comm_count": _cm_a, "ag_count": _ag_a,
                "comm_dens": _cm_a / wc * 100, "ag_dens": _ag_a / wc * 100,
                "comm_dens_cw": (_cm_a - _cm_fw) / wc * 100,
                "ag_dens_cw": (_ag_a - _ag_fw) / wc * 100,
                "comm_dens_fw": _cm_fw / wc * 100,
                "ag_dens_fw": _ag_fw / wc * 100,
                "comm_dens_AB": _cm_b / wc * 100,
                "ag_dens_AB": _ag_b / wc * 100}
        _row["balance"]    = _row["comm_dens"] - _row["ag_dens"]
        _row["balance_cw"] = _row["comm_dens_cw"] - _row["ag_dens_cw"]
        _row["balance_fw"] = _row["comm_dens_fw"] - _row["ag_dens_fw"]
        _row["balance_AB"] = _row["comm_dens_AB"] - _row["ag_dens_AB"]
        _eda_rows.append(_row)

    jd_comm_dens[occ]    = np.array(_cd);    jd_ag_dens[occ]    = np.array(_ad)
    jd_comm_dens_cw[occ] = np.array(_cd_cw); jd_ag_dens_cw[occ] = np.array(_ad_cw)
    jd_comm_dens_ab[occ] = np.array(_cd_ab); jd_ag_dens_ab[occ] = np.array(_ad_ab)
    jd_n_words[occ]      = np.array(_nw)

df_jd_eda = pd.DataFrame(_eda_rows)
print(f"  job descriptions: {len(df_jd_eda)}")
print(f"  control against agentic_communal_ratio under {LEX_MAIN} "
      f"(expected 0 mismatches): {_control_mismatch}")
if _control_mismatch:
    raise AssertionError(
        f"{_control_mismatch} job descriptions counted differently here than by "
        f"agentic_communal_ratio.")

# Control of the full balance is the sum of its two parts.
_split_dev = float((df_jd_eda["balance"]
                    - (df_jd_eda["balance_cw"] + df_jd_eda["balance_fw"])).abs().max())
print(f"  largest deviation of balance from balance_cw + balance_fw: {_split_dev:.2e} "
      f"(expected below 1e-9)")
if _split_dev > 1e-9:
    raise AssertionError("The function-word split of the balance does not add up.")

# Distribution per occupation
print()
print(f"Distribution per occupation")
print(f"  {'Occ':<22}  {'n_jds':>5}  {'med_wds':>7}  "
      f"{'mu_comm':>7}  {'sd_comm':>7}  {'med_comm':>8}  "
      f"{'mu_ag':>6}  {'sd_ag':>6}  {'med_ag':>7}  "
      f"{'mu_bal':>7}  {'mu_bal_cw':>9}  {'mu_bal_fw':>9}  "
      f"{'pct_com>0':>9}  {'field':<14}  {'pct_F_bls':>9}")
print("  " + "-" * 140)
_occ_eda_rows = []
for occ in KEPT_13 + ["TOTAL"]:
    _meta_row = _occ_meta[_occ_meta["occ"] == occ].iloc[0] \
                if occ in _occ_meta["occ"].values else None
    _field = _meta_row["field"] if _meta_row is not None else "?"
    _pctf_bls = float(_meta_row["pct_women_bls_2025"]) if _meta_row is not None else float("nan")
    _sub = df_jd_eda if occ == "TOTAL" else df_jd_eda[df_jd_eda["occ"] == occ]
    _n = len(_sub)
    _med_w = float(_sub["n_words"].median())
    _mu_c, _sd_c = float(_sub["comm_dens"].mean()), float(_sub["comm_dens"].std())
    _med_c = float(_sub["comm_dens"].median())
    _mu_a, _sd_a = float(_sub["ag_dens"].mean()), float(_sub["ag_dens"].std())
    _med_a = float(_sub["ag_dens"].median())
    _mu_b = float(_sub["balance"].mean())
    _mu_b_cw = float(_sub["balance_cw"].mean())
    _mu_b_fw = float(_sub["balance_fw"].mean())
    _pct_b = float((_sub["balance"] > 0).mean())
    # The share of women is defined per occupation; the TOTAL row has none.
    _pct_f_pool = float("nan") if occ == "TOTAL" else PCT_FEMALE[occ]
    print(f"  {occ:<22}  {_n:>5}  {_med_w:>7.0f}  "
          f"{_mu_c:>7.3f}  {_sd_c:>7.3f}  {_med_c:>8.3f}  "
          f"{_mu_a:>6.3f}  {_sd_a:>6.3f}  {_med_a:>7.3f}  "
          f"{_mu_b:>+7.3f}  {_mu_b_cw:>+9.3f}  {_mu_b_fw:>+9.3f}  "
          f"{_pct_b:>9.1%}  {str(_field):<14}  {_pctf_bls:>9.1f}")
    if occ != "TOTAL":
        _occ_eda_rows.append({
            "occ": occ, "field": _field, "pct_women_bls_2025": _pctf_bls,
            "pct_women_pool": _pct_f_pool, "n_jds": _n, "median_n_words": _med_w,
            "mean_comm_dens": _mu_c, "sd_comm_dens": _sd_c, "med_comm_dens": _med_c,
            "p25_comm_dens": float(_sub["comm_dens"].quantile(0.25)),
            "p75_comm_dens": float(_sub["comm_dens"].quantile(0.75)),
            "mean_ag_dens": _mu_a, "sd_ag_dens": _sd_a, "med_ag_dens": _med_a,
            "p25_ag_dens": float(_sub["ag_dens"].quantile(0.25)),
            "p75_ag_dens": float(_sub["ag_dens"].quantile(0.75)),
            "mean_balance": _mu_b, "med_balance": float(_sub["balance"].median()),
            "p25_balance": float(_sub["balance"].quantile(0.25)),
            "p75_balance": float(_sub["balance"].quantile(0.75)),
            "pct_balance_pos": _pct_b,
            "mean_comm_dens_cw": float(_sub["comm_dens_cw"].mean()),
            "mean_ag_dens_cw": float(_sub["ag_dens_cw"].mean()),
            "mean_balance_cw": _mu_b_cw, "mean_balance_fw": _mu_b_fw,
            "mean_comm_dens_AB": float(_sub["comm_dens_AB"].mean()),
            "mean_ag_dens_AB": float(_sub["ag_dens_AB"].mean()),
            "mean_balance_AB": float(_sub["balance_AB"].mean()),
        })
df_jd_eda_by_occ = pd.DataFrame(_occ_eda_rows)

# Spearman between the full balance and its two parts, per occupation.
print()
print("Rank correlation of balance with its two parts, per occupation:")
print(f"  {'occ':<22}  {'rho(bal, bal_cw)':>17}  {'rho(bal, bal_fw)':>17}")
for occ in KEPT_13:
    _sub = df_jd_eda[df_jd_eda["occ"] == occ]
    if len(_sub) < 5:
        continue
    _r_cw, _ = spearmanr(_sub["balance"], _sub["balance_cw"])
    _r_fw, _ = spearmanr(_sub["balance"], _sub["balance_fw"])
    print(f"  {occ:<22}  {_r_cw:>+17.3f}  {_r_fw:>+17.3f}")

print(f"\n  study2_jd_eda.csv: {len(df_jd_eda)} rows  "
      f"study2_jd_eda_by_occupation.csv: {len(df_jd_eda_by_occ)} rows")

Job description densities
control occupations with a share of women in the pool: 13 of 13 (expected all)
  job descriptions: 2975
  control against agentic_communal_ratio under A (expected 0 mismatches): 0
  largest deviation of balance from balance_cw + balance_fw: 4.44e-15 (expected below 1e-9)

Distribution per occupation
  Occ                     n_jds  med_wds  mu_comm  sd_comm  med_comm   mu_ag   sd_ag   med_ag   mu_bal  mu_bal_cw  mu_bal_fw  pct_com>0  field           pct_F_bls
  --------------------------------------------------------------------------------------------------------------------------------------------
  nurse                     250      316    7.146    2.421     7.154   4.896   1.903    4.962   +2.250     +1.930     +0.320      83.6%  care_education       87.3
  software_engineer         250      480    4.752    2.375     4.865   5.027   1.849    4.979   -0.275     -0.235     -0.040      50.4%  tech_law_business       20.3
  accountant                250      3

In [10]:
# The occupation means of the job description densities
# under A u B have to match the full run of 2026-10-05, tolerance 0.001. The cell
# sits before the embeddings, so an abort costs nothing here. The reference values
# come from A u B, so the check runs on the A u B densities.
JD_DENS_REF_AB = {
    "architect":        (5.114, 8.410), "software_engineer": (5.377, 7.799),
    "accountant":       (6.006, 7.871), "paralegal":         (6.685, 7.445),
    "physician":        (7.018, 7.186), "attorney":          (7.401, 7.394),
    "professor":        (6.367, 6.275), "psychologist":      (7.808, 7.316),
    "dietitian":        (7.287, 6.700), "surgeon":           (7.028, 6.334),
    "dentist":          (8.143, 7.046), "nurse":             (7.816, 6.633),
    "teacher":          (8.025, 6.789),
}
if SMOKE:
    print("control job description densities: skipped in the test run; "
          "the reference values belong to the full run of 2026-10-05.")
else:
    _rows = []
    print(f"  {'occ':<20} {'comm_dens':>10} {'ref':>8} {'ag_dens':>9} {'ref':>8}  status")
    print("  " + "-" * 68)
    _worst = 0.0
    for occ in KEPT_13:
        _sub = df_jd_eda[df_jd_eda["occ"] == occ]
        _cd = float(_sub["comm_dens_AB"].mean())
        _ad = float(_sub["ag_dens_AB"].mean())
        _ref_cd, _ref_ad = JD_DENS_REF_AB[occ]
        _dev = max(abs(_cd - _ref_cd), abs(_ad - _ref_ad))
        _worst = max(_worst, _dev)
        print(f"  {occ:<20} {_cd:>10.3f} {_ref_cd:>8.3f} {_ad:>9.3f} {_ref_ad:>8.3f}  "
              f"deviation {_dev:.4f}")
        _rows.append({"occ": occ, "comm_dens": _cd, "comm_dens_ref": _ref_cd,
                      "ag_dens": _ad, "ag_dens_ref": _ref_ad, "deviation": _dev})
    df_ctrl_jd_dens = pd.DataFrame(_rows)
    print(f"  largest deviation: {_worst:.4f} (tolerance 0.001)")
    if _worst > 0.001:
        raise AssertionError(
            f"The job description densities deviate from the full run of "
            f"2026-10-05 by up to {_worst:.4f}; expected at most 0.001.")
    print("  control job description densities passed")

  occ                   comm_dens      ref   ag_dens      ref  status
  --------------------------------------------------------------------
  nurse                     7.816    7.816     6.633    6.633  deviation 0.0003
  software_engineer         5.377    5.377     7.799    7.799  deviation 0.0004
  accountant                6.006    6.006     7.871    7.871  deviation 0.0004
  architect                 5.114    5.114     8.410    8.410  deviation 0.0005
  psychologist              7.808    7.808     7.316    7.316  deviation 0.0004
  attorney                  7.401    7.401     7.394    7.394  deviation 0.0001
  teacher                   8.025    8.025     6.789    6.789  deviation 0.0005
  surgeon                   7.028    7.028     6.334    6.334  deviation 0.0005
  physician                 7.018    7.018     7.186    7.186  deviation 0.0001
  professor                 6.367    6.367     6.275    6.275  deviation 0.0003
  dentist                   8.143    8.143     7.046    7.0

In [11]:
# Between / within occupations (Kruskal-Wallis + eta^2)
from scipy.stats import kruskal as _kw_e3, f_oneway as _fow_e3
_eda_test_rows = []
print("Kruskal-Wallis and eta squared by occupation")
print(f"  {'Variable':<12}  {'H':>8}  {'df':>4}  {'p':>8}  {'eta2':>7}")
print("  " + "-" * 45)
for _var in ("comm_dens","ag_dens","balance"):
    _groups = [df_jd_eda[df_jd_eda["occ"]==occ][_var].values for occ in KEPT_13
               if len(df_jd_eda[df_jd_eda["occ"]==occ]) > 0]
    if len(_groups) < 2: continue
    _H, _p = _kw_e3(*_groups)
    _grand_mean = df_jd_eda[_var].mean()
    _SST = ((df_jd_eda[_var] - _grand_mean)**2).sum()
    _SSB = sum(len(g) * (g.mean() - _grand_mean)**2 for g in _groups)
    _eta2 = float(_SSB / _SST) if _SST > 0 else float("nan")
    _df = len(_groups) - 1
    print(f"  {_var:<12}  {_H:>8.2f}  {_df:>4}  {_p:>8.4f}  {_eta2:>7.3f}")
    _eda_test_rows.append({"test":"kruskal_wallis","variable":_var,"H":float(_H),
                            "df_between":_df,"p":float(_p),"eta2":_eta2})

Kruskal-Wallis and eta squared by occupation
  Variable             H    df         p     eta2
  ---------------------------------------------
  comm_dens       440.99    12    0.0000    0.132
  ag_dens          90.25    12    0.0000    0.030
  balance         379.60    12    0.0000    0.112


In [12]:
# Correlation of the densities with the share of women, over the occupations.
from scipy.stats import spearmanr as _sp_e4, pearsonr as _pe_e4
print(f"control occupations available: {len(KEPT_13)} (expected at least 10)")
if len(KEPT_13) < 10:
    raise AssertionError(
        f"The correlation over occupations needs at least 10 of them, "
        f"{len(KEPT_13)} are present.")
print("Correlation with the share of women")
print(f"  {'x':<20}  {'y':<14}  {'r':>6}  {'p_r':>7}  {'rho':>6}  {'p_rho':>7}  status")
print("  " + "-" * 70)
_x_bls = [float(_occ_meta[_occ_meta["occ"]==o]["pct_women_bls_2025"].values[0])
          if o in _occ_meta["occ"].values else float("nan") for o in KEPT_13]
_x_pool = [PCT_FEMALE[o] for o in KEPT_13]
_y_vals = {_var: df_jd_eda_by_occ.set_index("occ").reindex(KEPT_13)[f"mean_{_var}"].values
           for _var in ("comm_dens","ag_dens","balance")}
for _xi, _xlabel in [(_x_bls,"pct_women_bls"),(_x_pool,"pct_women_pool")]:
    for _var, _yv in _y_vals.items():
        _pairs = [(x,y) for x,y in zip(_xi,_yv) if not (pd.isna(x) or pd.isna(y))]
        if len(_pairs) < 5: continue
        _xp, _yp = zip(*_pairs)
        _r, _pr = _pe_e4(_xp, _yp); _rho, _prho = _sp_e4(_xp, _yp)
        _exp = "comm>0" if _var in ("comm_dens","balance") else "ag<0"
        _status = "aligned" if ((_var in ("comm_dens","balance") and _r > 0) or
                                 (_var == "ag_dens" and _r < 0)) else "opposite"
        print(f"  {_xlabel:<20}  {_var:<14}  {_r:>+6.3f}  {_pr:>7.4f}  "
              f"{_rho:>+6.3f}  {_prho:>7.4f}  {_status}")
        _eda_test_rows.append({"test":"corr_pct_women","x":_xlabel,"y":_var,
                                "n":len(_pairs),"r":float(_r),"p_r":float(_pr),
                                "rho":float(_rho),"p_rho":float(_prho),
                                "expected":_exp,"status":_status})

control occupations available: 13 (expected at least 10)
Correlation with the share of women
  x                     y                    r      p_r     rho    p_rho  status
  ----------------------------------------------------------------------
  pct_women_bls         comm_dens       +0.507   0.0772  +0.374   0.2086  aligned
  pct_women_bls         ag_dens         -0.057   0.8520  -0.247   0.4154  aligned
  pct_women_bls         balance         +0.500   0.0819  +0.445   0.1275  aligned
  pct_women_pool        comm_dens       +0.509   0.0759  +0.390   0.1876  aligned
  pct_women_pool        ag_dens         -0.036   0.9058  -0.148   0.6286  aligned
  pct_women_pool        balance         +0.494   0.0863  +0.396   0.1809  aligned


In [13]:
# Comparison of the two occupational fields, with an exact permutation test.
from itertools import combinations as _comb_e5
print(f"control occupations available: {len(KEPT_13)} (expected at least 10)")
print(f"control field column present: {'field' in _occ_meta.columns} "
      f"(expected True)")
if len(KEPT_13) < 10 or "field" not in _occ_meta.columns:
    raise AssertionError(
        f"The field comparison needs at least 10 occupations and a field column; "
        f"occupations {len(KEPT_13)}, field column "
        f"{'field' in _occ_meta.columns}.")
_fields = {row["occ"]: row["field"] for _, row in _occ_meta.iterrows()}
_tech_set = {o for o, f in _fields.items() if f == "tech_law_business" and o in KEPT_13}
_care_set = {o for o in KEPT_13 if o not in _tech_set}
print(f"care_education n={len(_care_set)}, tech_law_business n={len(_tech_set)}")
print(f"  tech_law_business: {sorted(_tech_set)}")
for _var in ("comm_dens","ag_dens","balance"):
    _occ_means = df_jd_eda_by_occ.set_index("occ")[f"mean_{_var}"].to_dict()
    _care_m = np.mean([_occ_means.get(o, float("nan")) for o in _care_set])
    _tech_m = np.mean([_occ_means.get(o, float("nan")) for o in _tech_set])
    _obs_diff = float(_care_m - _tech_m)
    # Exact permutation: all C(13, n_tech) assignments
    _all_occs = [o for o in KEPT_13 if o in _occ_means]
    _n_tech = len(_tech_set)
    _n_perms = 0; _n_extreme = 0
    for _tech_perm in _comb_e5(_all_occs, _n_tech):
        _care_perm = [o for o in _all_occs if o not in _tech_perm]
        _d = np.mean([_occ_means[o] for o in _care_perm]) - \
             np.mean([_occ_means[o] for o in _tech_perm])
        _n_perms += 1
        if abs(_d) >= abs(_obs_diff):
            _n_extreme += 1
    _p_perm = float(_n_extreme / _n_perms)
    print(f"  {_var:<12}: care={_care_m:>+8.3f}  tech={_tech_m:>+8.3f}  "
          f"diff={_obs_diff:>+8.3f}  p={_p_perm:.4f}  n_perms={_n_perms}")
    _eda_test_rows.append({"test":"field_perm","variable":_var,
                            "mean_care":float(_care_m),"mean_tech":float(_tech_m),
                            "diff_care_minus_tech":_obs_diff,"p_perm":_p_perm,
                            "n_perms":_n_perms})
df_jd_eda_tests = pd.DataFrame(_eda_test_rows)
print(f"\n  study2_jd_eda_tests.csv: {len(df_jd_eda_tests)} rows")

control occupations available: 13 (expected at least 10)
control field column present: True (expected True)
care_education n=8, tech_law_business n=5
  tech_law_business: ['accountant', 'architect', 'attorney', 'paralegal', 'software_engineer']
  comm_dens   : care=  +6.863  tech=  +5.571  diff=  +1.292  p=0.0155  n_perms=1287
  ag_dens     : care=  +4.868  tech=  +5.029  diff=  -0.160  p=0.5470  n_perms=1287
  balance     : care=  +1.995  tech=  +0.543  diff=  +1.452  p=0.0093  n_perms=1287

  study2_jd_eda_tests.csv: 12 rows


In [14]:
# Top lexicon words per JD dimension
print("Most frequent lexicon words in the job descriptions")
_word_counts_comm = {}
_word_counts_ag   = {}

def _get_lex_matches(doc, text, lex):
    # Lex is a Lexicons dataclass with .match_communion(tok), .match_agency(tok)
    _c_words, _a_words = [], []
    for t in doc:
        _tl = t.lower_
        if lex.match_communion(_tl): _c_words.append(_tl)
        if lex.match_agency(_tl):   _a_words.append(_tl)
    # Multi-word communion (same as agentic_communal_ratio in src/features.py)
    import re as _re_e6
    _text_lower = text.lower()
    for _mw, _is_wild in lex.multi_word_communion:
        _pat = r'\b' + _re_e6.escape(_mw) + (r'\w*' if _is_wild else r'') + r'\b'
        _mc = len(_re_e6.findall(_pat, _text_lower))
        _c_words.extend([_mw] * _mc)
    return _c_words, _a_words

_jd_match_totals = {}  # {(occ,jd_idx): (n_comm_hits, n_ag_hits)}
_total_hits_comm = 0; _total_hits_ag = 0
_control_mismatch = 0

# Counting lexicon hits needs token surfaces only, so the tokenizer-only
# pipeline runs here, the same one the density section uses. The control below
# compares the counts with agentic_communal_ratio and has to stay at zero.
for occ in KEPT_13:
    jd_texts = jd_pools[occ]["description"].tolist()
    for j, (text, doc) in enumerate(zip(jd_texts,
                                        nlp_count.pipe(jd_texts,
                                                       batch_size=BATCH_SIZE))):
        _cw, _aw = _get_lex_matches(doc, text, lex)
        for w in _cw: _word_counts_comm[w] = _word_counts_comm.get(w, 0) + 1
        for w in _aw: _word_counts_ag[w]   = _word_counts_ag.get(w, 0) + 1
        _jd_match_totals[(occ, j)] = (len(_cw), len(_aw))
        # Control: verify total vs agentic_communal_ratio count
        _ag_c, _c_c, _ = _acr_fn(doc, text, lex)
        if abs(len(_cw) - _c_c) + abs(len(_aw) - _ag_c) > 0:
            _control_mismatch += 1
        _total_hits_comm += len(_cw); _total_hits_ag += len(_aw)

print(f"  Control: mismatches vs agentic_communal_ratio: {_control_mismatch} (expected 0)")
print(f"  Total communion hits: {_total_hits_comm}, agency hits: {_total_hits_ag}")

_top20_rows = []
for _dim, _wc, _tot in [("communion", _word_counts_comm, _total_hits_comm),
                          ("agency",    _word_counts_ag,   _total_hits_ag)]:
    _sorted = sorted(_wc.items(), key=lambda x: -x[1])[:20]
    print(f"\n  Top 20 {_dim} words across all JDs:")
    for _w, _n in _sorted:
        print(f"    {_w:<25} n={_n:>5}  share={_n/_tot:.3%}")
        _top20_rows.append({"dimension":_dim,"word":_w,"count":_n,
                             "share":float(_n/_tot),"field":"ALL"})
df_jd_top_words = pd.DataFrame(_top20_rows)
print(f"\n  study2_jd_top_lexicon_words.csv: {len(df_jd_top_words)} rows")

Most frequent lexicon words in the job descriptions
  Control: mismatches vs agentic_communal_ratio: 0 (expected 0)
  Total communion hits: 89323, agency hits: 67342

  Top 20 communion words across all JDs:
    our                       n= 9394  share=10.517%
    we                        n= 8563  share=9.587%
    care                      n= 6192  share=6.932%
    team                      n= 4804  share=5.378%
    support                   n= 2950  share=3.303%
    education                 n= 2813  share=3.149%
    services                  n= 2553  share=2.858%
    service                   n= 1719  share=1.924%
    law                       n= 1687  share=1.889%
    us                        n= 1679  share=1.880%
    communication             n= 1572  share=1.760%
    community                 n= 1543  share=1.727%
    learning                  n= 1501  share=1.680%
    family                    n= 1161  share=1.300%
    teaching                  n= 1122  share=1.256%
    members

In [15]:
# Figures (3 plots)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

_fig_dir = Path("outputs/figures/study2")
_fig_dir.mkdir(parents=True, exist_ok=True)

_occ_eda = df_jd_eda_by_occ.set_index("occ")
_sorted_occ = _occ_eda.sort_values("med_balance").index.tolist()
_field_colors = {"care_education": "#4C72B0", "tech_law_business": "#DD8452",
                 "unknown": "#8C8C8C"}
_colors = [_field_colors.get(str(_occ_eda.loc[o, "field"]) if o in _occ_eda.index else "unknown",
                              "#8C8C8C") for o in _sorted_occ]

# Fig 1: Boxplots of balance by occupation
_fig1, _ax1 = plt.subplots(figsize=(10, 5))
_bal_data  = [df_jd_eda[df_jd_eda["occ"]==o]["balance"].values for o in _sorted_occ]
_bp = _ax1.boxplot(_bal_data, vert=True, patch_artist=True, medianprops=dict(color="black"))
for _patch, _col in zip(_bp["boxes"], _colors):
    _patch.set_facecolor(_col); _patch.set_alpha(0.7)
_ax1.axhline(0, color="black", linewidth=0.7, linestyle="--")
_ax1.set_xticks(range(1, len(_sorted_occ)+1))
_ax1.set_xticklabels([o.replace("_"," ") for o in _sorted_occ], rotation=45, ha="right")
_ax1.set_ylabel("Balance (comm_dens - ag_dens, pp)")
_ax1.set_xlabel("Occupation (sorted by median balance)")
plt.tight_layout()
_p1_png = str(_fig_dir / "jd_balance_by_occupation.png")
_p1_pdf = str(_fig_dir / "jd_balance_by_occupation.pdf")
_fig1.savefig(_p1_png, dpi=300, bbox_inches="tight")
_fig1.savefig(_p1_pdf, bbox_inches="tight")
plt.close(_fig1)
print(f"  Fig 1: {_p1_png}")

# Fig 2: Occupation means - ag_dens vs comm_dens with diagonal
_fig2, _ax2 = plt.subplots(figsize=(6,5))
_shown = [o for o in KEPT_13 if o in _occ_eda.index]
_xvals = [float(_occ_eda.loc[o, "mean_ag_dens"]) for o in _shown]
_yvals = [float(_occ_eda.loc[o, "mean_comm_dens"]) for o in _shown]
_fcols = [_field_colors.get(str(_occ_eda.loc[o,"field"]),"#8C8C8C") for o in _shown]
_ax2.scatter(_xvals, _yvals, c=_fcols, s=60, zorder=3)
for _o, _x, _y in zip(_shown, _xvals, _yvals):
    _ax2.annotate(_o.replace("_","\n"), (_x,_y), fontsize=6.5, ha="center", va="bottom")
_lim = max(max(_xvals), max(_yvals)) * 1.05
_ax2.plot([0, _lim], [0, _lim], "k--", linewidth=0.7)
_ax2.set_xlabel("Agency density (%)"); _ax2.set_ylabel("Communion density (%)")
plt.tight_layout()
_p2_png = str(_fig_dir / "jd_agency_vs_communion.png")
_p2_pdf = str(_fig_dir / "jd_agency_vs_communion.pdf")
_fig2.savefig(_p2_png, dpi=300, bbox_inches="tight")
_fig2.savefig(_p2_pdf, bbox_inches="tight")
plt.close(_fig2)
print(f"  Fig 2: {_p2_png}")

# Fig 3: balance vs pct_women_bls_2025
_fig3, _ax3 = plt.subplots(figsize=(6,5))
_xv3 = [float(_occ_eda.loc[o,"pct_women_bls_2025"]) if o in _occ_eda.index and
         not pd.isna(_occ_eda.loc[o,"pct_women_bls_2025"]) else float("nan") for o in _shown]
_yv3 = [float(_occ_eda.loc[o,"mean_balance"]) if o in _occ_eda.index else float("nan") for o in _shown]
_pairs3 = [(x,y,o,_field_colors.get(str(_occ_eda.loc[o,"field"]),"#8C8C8C"))
            for x,y,o in zip(_xv3,_yv3,_shown) if not (pd.isna(x) or pd.isna(y))]
if _pairs3:
    _ax3.scatter([p[0] for p in _pairs3],[p[1] for p in _pairs3],
                  c=[p[3] for p in _pairs3], s=60, zorder=3)
    for _x,_y,_o,_ in _pairs3:
        _ax3.annotate(_o.replace("_","\n"),(_x,_y),fontsize=6.5,ha="center",va="bottom")
_ax3.axhline(0, color="black", linewidth=0.7, linestyle="--")
_ax3.set_xlabel("Women (%, BLS 2025)"); _ax3.set_ylabel("Mean balance (pp)")
plt.tight_layout()
_p3_png = str(_fig_dir / "jd_balance_vs_pct_women.png")
_p3_pdf = str(_fig_dir / "jd_balance_vs_pct_women.pdf")
_fig3.savefig(_p3_png, dpi=300, bbox_inches="tight")
_fig3.savefig(_p3_pdf, bbox_inches="tight")
plt.close(_fig3)
print(f"  Fig 3: {_p3_png}")

  Fig 1: outputs\figures\study2\jd_balance_by_occupation.png
  Fig 2: outputs\figures\study2\jd_agency_vs_communion.png
  Fig 3: outputs\figures\study2\jd_balance_vs_pct_women.png


## 6. Counterfactuals

The list is applied per occurrence on the parsed original text. Every lexicon hit gets a protocol row, so coverage and the reason for every untreated occurrence can be reported. The control at the end recounts the lexicon hits of the pool against the stored counts.

In [16]:
# Manipulation with the substitution list v2. The engine works on the
# parsed original text and returns one protocol row per lexicon hit, so coverage
# and the reasons for untreated occurrences can be reported.
manip_pool  = {}
manip_focal = {}
_occurrence_rows = []

print(f"{'Occ':<22}  {'Arm':>9}  {'pool_treated':>12}  {'focal_treated':>13}")
print("-" * 62)
_t_manip = time.time()
for occ in KEPT_13:
    manip_pool[occ]  = {}
    manip_focal[occ] = {}
    pool_texts = bio_pools[occ]["text_clean"].tolist()
    pool_docs  = list(nlp.pipe(pool_texts, batch_size=BATCH_SIZE))
    for arm in ("communion", "agency"):
        focal_texts = focal_bios[occ][arm]["text_clean"].tolist() \
                      if "text_clean" in focal_bios[occ][arm].columns else []
        rp = [engine_v2.apply(doc, arm, bio_id=i) for i, doc in enumerate(pool_docs)]
        rf = [engine_v2.apply(doc, arm, bio_id=j) for j, doc
              in enumerate(nlp.pipe(focal_texts, batch_size=BATCH_SIZE))]
        manip_pool[occ][arm]  = rp
        manip_focal[occ][arm] = rf
        for sample, results in (("pool", rp), ("focal", rf)):
            for bio_index, result in enumerate(results):
                for occurrence in result.occurrences:
                    _occurrence_rows.append({
                        "occ": occ, "sample": sample, "arm": arm,
                        "bio_idx": bio_index, "token": occurrence.token,
                        "position": occurrence.position,
                        "rule_id": occurrence.rule_id,
                        "word_class": occurrence.word_class,
                        "action_taken": occurrence.action_taken,
                        "reason_untreated": occurrence.reason_untreated,
                        "target": occurrence.target,
                        "verb_reading": occurrence.verb_reading,
                        "side_effect": occurrence.side_effect,
                        "context_original": occurrence.context_original,
                        "context_manipulated": occurrence.context_manipulated,
                    })
        avg_p = np.mean([r.n_replaced + r.n_deleted for r in rp]) if rp else 0
        avg_f = np.mean([r.n_replaced + r.n_deleted for r in rf]) if rf else 0
        print(f"  {occ:<22}  {arm:>9}  {avg_p:>12.2f}  {avg_f:>13.2f}")

df_occurrences = pd.DataFrame(_occurrence_rows)
print(f"\nmanipulation done in {time.time() - _t_manip:.0f}s; "
      f"{len(df_occurrences)} occurrences logged")

# Coverage over the pool per arm, with the reason for every untreated
# occurrence.
_cov_rows = []
for arm in ("communion", "agency"):
    subset = df_occurrences[(df_occurrences["arm"] == arm)
                            & (df_occurrences["sample"] == "pool")]
    entry = {"arm": arm, "lexicon_hits": len(subset),
             "replaced": int((subset["action_taken"] == "replaced").sum()),
             "deleted": int((subset["action_taken"] == "deleted").sum()),
             "untreated": int((subset["action_taken"] == "untreated").sum())}
    entry["treated_share"] = ((entry["replaced"] + entry["deleted"]) / len(subset)
                              if len(subset) else float("nan"))
    for reason in UNTREATED_REASONS:
        entry[f"untreated_{reason}"] = int((subset["reason_untreated"] == reason).sum())
    _cov_rows.append(entry)
    print(f"  {arm:<10} hits={entry['lexicon_hits']:>6}  "
          f"replaced={entry['replaced']:>6}  deleted={entry['deleted']:>5}  "
          f"untreated={entry['untreated']:>6}  treated={entry['treated_share']:.1%}")
    for reason in UNTREATED_REASONS:
        if entry[f"untreated_{reason}"]:
            print(f"      {reason:<26} {entry[f'untreated_{reason}']:>6}")
df_coverage_v2 = pd.DataFrame(_cov_rows)

# Sentence-initial nouns read as verbs, and phrase replacements that also remove
# A word of the other dimension.
print(f"\nsentence-initial nouns read as verbs: "
      f"{int(df_occurrences['verb_reading'].sum())}")
_side = df_occurrences[df_occurrences["side_effect"].astype(str) != ""]
print(f"replacements that also remove a word of the other dimension: {len(_side)}")
if len(_side):
    print(_side.groupby(["arm", "rule_id", "side_effect"]).size()
          .rename("n").reset_index().to_string(index=False))

Occ                           Arm  pool_treated  focal_treated
--------------------------------------------------------------
  nurse                   communion          1.38           5.40
  nurse                      agency          0.26           3.26
  software_engineer       communion          0.51           3.43
  software_engineer          agency          0.22           2.26
  accountant              communion          0.72           3.19
  accountant                 agency          0.52           2.85
  architect               communion          0.70           3.83
  architect                  agency          0.45           3.10
  psychologist            communion          1.28           5.72
  psychologist               agency          0.52           3.52
  attorney                communion          0.94           5.34
  attorney                   agency          0.60           4.02
  teacher                 communion          1.72           6.70
  teacher                    

In [17]:
# The recounted communion and agency hits of the pool biographies under
# dictionary A equal the stored values.
_rows = []
_worst = 0
for occ in KEPT_13:
    _frame = bio_pools[occ]
    for _doc, (_, _row) in zip(nlp_count.pipe(_frame["text_clean"].tolist(),
                                              batch_size=BATCH_SIZE),
                               _frame.iterrows()):
        _ag, _cm, _ = _acr_fn(_doc, _row["text_clean"], lex)
        _d = max(abs(_ag - _row["agency_count"]), abs(_cm - _row["communion_count"]))
        _worst = max(_worst, _d)
        if _d:
            _rows.append({"occ": occ, "agency_recounted": _ag,
                          "agency_stored": int(_row["agency_count"]),
                          "communion_recounted": _cm,
                          "communion_stored": int(_row["communion_count"])})
print(f"POOL bios recounted under dictionary {LEX_MAIN}: "
      f"{sum(len(bio_pools[o]) for o in KEPT_13)}")
print(f"bios with a deviation (expected 0): {len(_rows)}")
for _r in _rows[:10]:
    print(f"    {_r}")
if _rows:
    raise AssertionError(
        f"{len(_rows)} POOL bios have lexicon counts that differ from "
        f"df_features.parquet; largest deviation {_worst}.")
print("Control passed: the recounted values match the stored ones.")

POOL bios recounted under dictionary A: 3250
bios with a deviation (expected 0): 0
Control passed: the recounted values match the stored ones.


## 7. Manipulation check

- **Question:** How far does the manipulation move the measured construct,
  and does the gate remove the biographies that carry the most signal?
- **Expectation:** the rate of the manipulated dimension falls clearly, the rate
  of the other dimension stays near zero.
- **Test:** biography; rates per 100 words before and after, and the share of the
  hits of that dimension that were removed.
- **Status:** control, fixed 2026-10-11.
- **Output:** `manipulation_check.csv`

The parts that need the change in rank follow in section 13.

In [18]:
# Manipulation check. Per biography, arm and sample, with dictionary A:
# Rates before and after, the change, the cross effect on the other dimension,
# and how much of the dimension's hits were removed.
_mc_rows = []
for occ in KEPT_13:
    for sample, frame, results in (
            ("pool", bio_pools[occ], manip_pool[occ]),
            ("focal", None, manip_focal[occ])):
        for arm in ("communion", "agency"):
            _res = results[arm]
            if not _res:
                continue
            _frame = frame if frame is not None else focal_bios[occ][arm]
            _orig_texts  = [r.original for r in _res]
            _manip_texts = [r.manipulated for r in _res]
            _docs_o = list(nlp_count.pipe(_orig_texts,  batch_size=BATCH_SIZE))
            _docs_m = list(nlp_count.pipe(_manip_texts, batch_size=BATCH_SIZE))
            for i, (doc_o, doc_m, result) in enumerate(zip(_docs_o, _docs_m, _res)):
                ag_o, cm_o, _ = _acr_fn(doc_o, _orig_texts[i],  lex)
                ag_m, cm_m, _ = _acr_fn(doc_m, _manip_texts[i], lex)
                # The denominator before the manipulation is the stored word
                # count, so the rate matches the one Study 1 reports. After the
                # manipulation it moves by the change in length.
                words_o = max(int(_frame.iloc[i]["word_count"]), 1)
                words_m = max(words_o - (len(_orig_texts[i].split())
                                         - len(_manip_texts[i].split())), 1)
                rate_o = {"communion": cm_o / words_o * 100, "agency": ag_o / words_o * 100}
                rate_m = {"communion": cm_m / words_m * 100, "agency": ag_m / words_m * 100}
                other = "agency" if arm == "communion" else "communion"
                hits_o = {"communion": cm_o, "agency": ag_o}[arm]
                hits_m = {"communion": cm_m, "agency": ag_m}[arm]
                _mc_rows.append({
                    "occ": occ, "sample": sample, "arm": arm, "bio_idx": i,
                    "female": int(_frame.iloc[i]["female"])
                              if "female" in _frame.columns else np.nan,
                    "rate_before": rate_o[arm], "rate_after": rate_m[arm],
                    "delta_rate": rate_m[arm] - rate_o[arm],
                    "rate_other_before": rate_o[other],
                    "rate_other_after": rate_m[other],
                    "delta_rate_other": rate_m[other] - rate_o[other],
                    "hits_before": hits_o, "hits_after": hits_m,
                    "share_hits_removed": (hits_o - hits_m) / hits_o if hits_o else np.nan,
                    "n_replaced": result.n_replaced, "n_deleted": result.n_deleted,
                    "words_before": words_o, "words_after": words_m,
                    "delta_words": words_m - words_o,
                    "changed": (result.n_replaced + result.n_deleted) > 0,
                })
df_manip_check = pd.DataFrame(_mc_rows)

# Rate_before has to equal the stored rate of the biography.
_stored = pd.concat(
    [bio_pools[o].reset_index(drop=True)
     .assign(occ=o, bio_idx=range(len(bio_pools[o])))
     [["occ", "bio_idx", "communion_rate", "agency_rate"]]
     for o in KEPT_13], ignore_index=True)
_pool_check = df_manip_check[df_manip_check["sample"] == "pool"].merge(
    _stored, on=["occ", "bio_idx"], how="inner")
_pool_check["stored_rate"] = np.where(
    _pool_check["arm"] == "communion",
    _pool_check["communion_rate"], _pool_check["agency_rate"])
_dev_rate = float((_pool_check["rate_before"] - _pool_check["stored_rate"]).abs().max())
print(f"control rate_before against the stored rate: {_dev_rate:.2e} "
      f"(expected below 1e-9)")
if _dev_rate > 1e-9:
    raise AssertionError(
        f"rate_before differs from the stored rate by up to {_dev_rate:.3e}.")

print(f"{'sample':<7} {'arm':<10} {'n':>5} {'changed':>8} {'d_rate':>8} "
      f"{'d_other':>8} {'removed':>8}")
print("-" * 58)
for sample in ("pool", "focal"):
    for arm in ("communion", "agency"):
        _s = df_manip_check[(df_manip_check["sample"] == sample)
                            & (df_manip_check["arm"] == arm)]
        if _s.empty:
            raise AssertionError(
                f"The manipulation check has no rows for {sample}/{arm}; the "
                f"manipulation section has to run for both samples and arms.")
        _c = _s[_s["changed"]]
        print(f"{sample:<7} {arm:<10} {len(_s):>5} {len(_c):>8} "
              f"{_c['delta_rate'].median():>8.3f} "
              f"{_c['delta_rate_other'].median():>8.3f} "
              f"{_c['share_hits_removed'].median():>8.1%}")

control rate_before against the stored rate: 0.00e+00 (expected below 1e-9)
sample  arm            n  changed   d_rate  d_other  removed
----------------------------------------------------------
pool    communion   3250     1878   -2.703    0.000    60.0%
pool    agency      3250     1093   -2.151    0.000   100.0%
focal   communion   3250     3250   -5.470    0.000    66.7%
focal   agency      3250     3208   -3.704    0.000    75.0%


## 8. Function words in the biographies

- **Question:** What share of the lexicon hits falls on the pronouns and
  modal verbs of D1, and does that share differ between women and men?
- **Test:** biography; counting only.
- **Status:** descriptive, fixed 2026-10-11. The main tests keep the full
  dictionary; the manipulation never touches these words (D1).
- **Output:** `study2_function_words_bios.csv`

The same pass writes the biography rates without those hits, which the fit test
of section 14 uses as a variant.

In [19]:
# Share of the D1 function words among the lexicon hits of the POOL biographies,
# separately for women and men, and the biography rates without those hits (_cw)
# and the part taken out (_fw).
from src.substitution_v2 import is_function_word_kept
from src.lexicon import count_multiword_communion as _count_mw_bio

_fw_rows, _bio_fw_rows = [], []
_hit_mismatches = 0
for occ in KEPT_13:
    _frame = bio_pools[occ].reset_index(drop=True)
    for _idx, (_doc, _text) in enumerate(
            zip(nlp_count.pipe(_frame["text_clean"].tolist(), batch_size=BATCH_SIZE),
                _frame["text_clean"].tolist())):
        _row = _frame.iloc[_idx]
        # The denominator is the stored word count, the same one Study 1 and the
        # rates in df_s2 use.
        _words = max(int(_row["word_count"]), 1)
        _per_dim = {}
        for dimension, matcher in (("communion", lex.match_communion),
                                   ("agency", lex.match_agency)):
            _hits = [t.lower_ for t in _doc if matcher(t.lower_)]
            if dimension == "communion":
                # Multi-word communion matches hold no function words.
                _n_hits = len(_hits) + _count_mw_bio(str(_text).lower(),
                                                     lex.multi_word_communion)
            else:
                _n_hits = len(_hits)
            _n_fw = sum(1 for t in _hits if is_function_word_kept(t))
            _per_dim[dimension] = (_n_hits, _n_fw)
            _fw_rows.append({"occ": occ, "bio_idx": _idx, "dimension": dimension,
                             "female": int(_row["female"]),
                             "n_hits": _n_hits, "n_function_words": _n_fw,
                             "word_count": _words})
        _cm_hits, _cm_fw = _per_dim["communion"]
        _ag_hits, _ag_fw = _per_dim["agency"]
        if (_cm_hits != int(_row["communion_count"])
                or _ag_hits != int(_row["agency_count"])):
            _hit_mismatches += 1
        _bio_fw_rows.append({
            "occ": occ, "bio_idx": _idx,
            "communion_rate_cw": (_cm_hits - _cm_fw) / _words * 100,
            "agency_rate_cw": (_ag_hits - _ag_fw) / _words * 100,
            "communion_rate_fw": _cm_fw / _words * 100,
            "agency_rate_fw": _ag_fw / _words * 100,
        })
df_function_words = pd.DataFrame(_fw_rows)
df_bio_fw = pd.DataFrame(_bio_fw_rows)
df_bio_fw["bio_balance_cw"] = (df_bio_fw["communion_rate_cw"]
                               - df_bio_fw["agency_rate_cw"])
df_bio_fw["bio_balance_fw"] = (df_bio_fw["communion_rate_fw"]
                               - df_bio_fw["agency_rate_fw"])

print(f"control hits against stored counts: {_hit_mismatches} mismatches "
      f"(expected 0)")
if _hit_mismatches:
    raise AssertionError(
        f"{_hit_mismatches} biographies have lexicon hits that differ from the "
        f"stored communion_count or agency_count.")

print()
print(f"{'dimension':<10} {'group':<6} {'hits':>7} {'function words':>15} {'share':>8}")
print("-" * 50)
_summary = []
for dimension in ("communion", "agency"):
    for female, label in ((1, "women"), (0, "men"), (None, "all")):
        _s = df_function_words[df_function_words["dimension"] == dimension]
        if female is not None:
            _s = _s[_s["female"] == female]
        _hits, _fw = int(_s["n_hits"].sum()), int(_s["n_function_words"].sum())
        _share = _fw / _hits if _hits else float("nan")
        print(f"{dimension:<10} {label:<6} {_hits:>7} {_fw:>15} {_share:>8.1%}")
        _summary.append({"dimension": dimension, "group": label, "n_hits": _hits,
                         "n_function_words": _fw, "share_function_words": _share})
df_function_words_summary = pd.DataFrame(_summary)

# The rate without the function words plus the part taken out has to equal the
# stored rate of the biography.
_check = df_bio_fw.merge(
    pd.concat([bio_pools[o].reset_index(drop=True)
               .assign(occ=o, bio_idx=range(len(bio_pools[o])))
               [["occ", "bio_idx", "communion_rate", "agency_rate"]]
               for o in KEPT_13], ignore_index=True),
    on=["occ", "bio_idx"], how="inner")
_dev_cm = float((_check["communion_rate"]
                 - (_check["communion_rate_cw"] + _check["communion_rate_fw"])).abs().max())
_dev_ag = float((_check["agency_rate"]
                 - (_check["agency_rate_cw"] + _check["agency_rate_fw"])).abs().max())
print()
print(f"control rate = rate_cw + rate_fw: largest deviation communion "
      f"{_dev_cm:.2e}, agency {_dev_ag:.2e} (expected below 1e-9)")
if max(_dev_cm, _dev_ag) > 1e-9:
    raise AssertionError("The function-word split of the biography rates does not add up.")

control hits against stored counts: 0 mismatches (expected 0)

dimension  group     hits  function words    share
--------------------------------------------------
communion  women     4039               2     0.0%
communion  men       3393               0     0.0%
communion  all       7432               2     0.0%
agency     women     1549              40     2.6%
agency     men       1746              48     2.7%
agency     all       3295              88     2.7%

control rate = rate_cw + rate_fw: largest deviation communion 0.00e+00, agency 1.78e-15 (expected below 1e-9)


## 9. Encoder and embeddings

The encoder of Arm 1 is loaded and the embedding cache is prepared. The cache test that closes the section is required before any run: five manipulated biographies have to be embedded afresh and must not be served from the cache of the originals.

In [20]:
# Embedding cache. The key holds the hash of the embedded text, the name of the
# model and the width of its embeddings, and the prefix. The job description
# representation is recorded with it; it is applied to the embeddings afterwards
# and does not change them, so it is reported rather than used for validity.
#
# A hash file written before this key existed is not accepted: it does not say
# which model wrote it, and two encoders of different width once shared a folder.
# Such a file is replaced by a fresh computation.

# What this run read and what it computed. Every call appends one entry, and
# the last section reports them per part of the analysis.
CACHE_LOG = []

def _text_hash(texts):
    h = hashlib.sha256()
    for t in texts:
        h.update(t.encode("utf-8"))
    return h.hexdigest()[:16]

def _cache_path(model_name, tag):
    safe = str(model_name).replace("/", "_").replace("-", "_")
    return Path(CACHE_BASE) / safe / f"{tag}.npy"

def _mid_str(model):
    """The name of the model behind a SentenceTransformer."""
    card = getattr(model, "model_card_data", None)
    for candidate in (getattr(card, "base_model", None),
                      getattr(card, "model_id", None)):
        if candidate:
            return str(candidate)
    try:
        name = model[0].auto_model.name_or_path
        if name:
            return str(name)
    except (AttributeError, IndexError, TypeError):
        pass
    raise RuntimeError(
        "The name of the encoder cannot be read, so the cache key would not "
        "separate one model from another. Pass the name explicitly.")

def _model_width(model):
    try:
        return int(model.get_sentence_embedding_dimension())
    except AttributeError:
        return int(model.get_embedding_dimension())

def _cache_key(texts, model_id, prefix, width):
    return {"text_hash": _text_hash(texts), "model": str(model_id),
            "prefix": prefix, "width": int(width), "jd_repr": JD_REPR}

def _cache_valid(hash_path, key):
    """True when the stored key matches on text, model, prefix and width."""
    if not hash_path.exists():
        return False
    try:
        stored = json.loads(hash_path.read_text().strip())
    except json.JSONDecodeError:
        return False
    return all(stored.get(field) == key[field]
               for field in ("text_hash", "model", "prefix", "width"))

def _check_width(array, width, cpath):
    """Every vector handed back has to be as wide as the model makes them."""
    found = int(array.shape[1]) if getattr(array, "ndim", 0) == 2 else None
    if found is not None and found != int(width):
        raise AssertionError(
            f"{cpath} holds vectors of width {found} while the encoder makes "
            f"{width}. The cache of another model was read; delete that folder "
            f"and run again.")
    return array

def encode_plain_cached(texts, model, tag):
    model_id = f"{_mid_str(model)}_plain"
    width = _model_width(model)
    cpath = _cache_path(model_id, tag)
    hash_path = cpath.with_suffix(".hash")
    key = _cache_key(texts, model_id, "", width)
    if cpath.exists() and _cache_valid(hash_path, key):
        CACHE_LOG.append({"tag": tag, "model": model_id, "prefix": "",
                          "n_texts": len(texts), "source": "cache"})
        return _check_width(np.load(cpath), width, cpath)
    cpath.parent.mkdir(parents=True, exist_ok=True)
    out = model.encode(texts, normalize_embeddings=True,
                       batch_size=BATCH_SIZE, show_progress_bar=False)
    np.save(cpath, out)
    hash_path.write_text(json.dumps(key))
    CACHE_LOG.append({"tag": tag, "model": model_id, "prefix": "",
                      "n_texts": len(texts), "source": "computed"})
    return _check_width(out, width, cpath)

def load_or_encode(texts, model, tag, prefix="", batch_size=None):
    model_id = _mid_str(model)
    width = _model_width(model)
    cpath = _cache_path(model_id, tag)
    hash_path = cpath.with_suffix(".hash")
    key = _cache_key(texts, model_id, prefix, width)
    if cpath.exists() and _cache_valid(hash_path, key):
        out = np.load(cpath, allow_pickle=True).item()
        _check_width(out["mean"], width, cpath)
        CACHE_LOG.append({"tag": tag, "model": model_id, "prefix": prefix,
                          "n_texts": len(texts), "source": "cache"})
        return out
    cpath.parent.mkdir(parents=True, exist_ok=True)
    _bs = batch_size or BATCH_SIZE
    out = study2_core.encode_long(texts, model, prefix=prefix, batch_size=_bs)
    np.save(cpath, out)
    hash_path.write_text(json.dumps(key))
    _check_width(out["mean"], width, cpath)
    CACHE_LOG.append({"tag": tag, "model": model_id, "prefix": prefix,
                      "n_texts": len(texts), "source": "computed"})
    return out

In [21]:
# Load the encoder of Arm 1 and report its window.
print(f"Loading primary model: {PRIMARY_MODEL}")
main_model = SentenceTransformer(PRIMARY_MODEL)
print(f"  max_seq_length = {main_model.max_seq_length}")
_window = main_model.max_seq_length - 2 - 4   # n_special=2, margin=4 (encode_long default)
print(f"  encode_long window = {_window} tokens  "
      f"(plain model.encode truncates at {main_model.max_seq_length})")
print(f"control model name used in the cache key: {_mid_str(main_model)}")
print(f"control embedding width: {_model_width(main_model)}")
print(f"cache folder: {_cache_path(_mid_str(main_model), 'TAG').parent}")

Loading primary model: sentence-transformers/all-mpnet-base-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  max_seq_length = 384
  encode_long window = 378 tokens  (plain model.encode truncates at 384)
control model name used in the cache key: sentence-transformers/all-mpnet-base-v2
control embedding width: 768
cache folder: outputs\study2_cache\sentence_transformers_all_mpnet_base_v2


In [22]:
# Cache test: manipulated bios must not be served from the cache of the originals.
_cache_occ    = KEPT_13[0]
_cache_orig   = bio_pools[_cache_occ]["text_clean"].tolist()[:5]
_cache_manip  = [manip_pool[_cache_occ]["communion"][i].manipulated for i in range(5)]
_n_changed    = sum(1 for a, b in zip(_cache_orig, _cache_manip) if a != b)
print(f"test bios: 5 from {_cache_occ}; changed by the manipulation: {_n_changed}")
if _n_changed == 0:
    raise AssertionError(
        "None of the five test bios was changed, so the cache test cannot "
        "distinguish originals from manipulated texts. Pick an occupation with "
        "changed bios.")

_emb_orig  = encode_plain_cached(_cache_orig,  main_model, "cachetest_orig")
_emb_manip = encode_plain_cached(_cache_manip, main_model, "cachetest_manip")

# The key of the originals must not validate for the manipulated texts. It is
# built exactly as encode_plain_cached builds it, the width included.
_plain_id    = f"{_mid_str(main_model)}_plain"
_plain_width = _model_width(main_model)
_key_orig  = _cache_key(_cache_orig,  _plain_id, "", _plain_width)
_key_manip = _cache_key(_cache_manip, _plain_id, "", _plain_width)
print(f"text hash originals  : {_key_orig['text_hash']}")
print(f"text hash manipulated: {_key_manip['text_hash']}")
if _key_orig["text_hash"] == _key_manip["text_hash"]:
    raise AssertionError("The two text hashes are equal; the cache key does not "
                         "separate originals from manipulated texts.")

_hash_path_orig = _cache_path(_plain_id, "cachetest_orig").with_suffix(".hash")
if _cache_valid(_hash_path_orig, _key_manip):
    raise AssertionError("The cache of the originals validates for the manipulated "
                         "texts; the key is not specific enough.")
print("the cache of the originals does not validate for the manipulated texts")

_diff = float(np.abs(_emb_orig - _emb_manip).max())
_rowwise = [float(np.abs(_emb_orig[i] - _emb_manip[i]).max()) for i in range(5)]
print(f"largest difference between the two embedding sets: {_diff:.6f}")
print(f"per bio: {[round(d, 6) for d in _rowwise]}")
if _diff == 0.0:
    raise AssertionError("The manipulated texts produced identical embeddings; "
                         "they were loaded from the cache of the originals.")
print("cache test passed: manipulated bios are embedded afresh")

test bios: 5 from nurse; changed by the manipulation: 4
text hash originals  : d087fc16c355162e
text hash manipulated: 2f4c260bbfeb5d6b
the cache of the originals does not validate for the manipulated texts
largest difference between the two embedding sets: 0.024369
per bio: [0.024369, 0.0, 0.013233, 0.00365, 0.014186]
cache test passed: manipulated bios are embedded afresh


## 10. Content-preservation gate

The similarity between the original and the manipulated biography says how much of the content survived the rewriting. The main analysis keeps the biographies above 0.90.

In [23]:
# Content-preservation similarity between original and manipulated biography.
_t_cp = time.time()
cp_emb_orig  = {}
cp_emb_manip = {}
cp_sims_pool = {}
cp_kept_pool = {}
cp_sims_focal= {}
cp_gate_rows = []
_man_cache_sfx = f"_{RUN_TAG}" if RUN_TAG != "run4" else ""

print(f"CP gate = {PRIMARY_GATE}  (plain model.encode, cached, batch={BATCH_SIZE})")
print(f"{'Occ':<22}  {'Arm':>9}  {'n':>5}  {'n_excl':>6}  {'pct_excl':>8}  {'mean_sim':>9}")
print("-" * 66)

for occ in KEPT_13:
    pool_texts = bio_pools[occ]["text_clean"].tolist()
    emb_o = encode_plain_cached(pool_texts, main_model, f"{occ}_pool_orig")
    cp_emb_orig[occ]  = emb_o
    cp_emb_manip[occ] = {}
    cp_sims_pool[occ] = {}
    cp_kept_pool[occ] = {}
    cp_sims_focal[occ]= {}
    for arm in ("communion", "agency"):
        mans = [r.manipulated for r in manip_pool[occ][arm]]
        emb_m = encode_plain_cached(mans, main_model,
                                    f"{occ}_{arm}_pool_manip{_man_cache_sfx}")
        cp_emb_manip[occ][arm] = emb_m
        sims  = np.diag(cos_sim(emb_o, emb_m))
        cp_sims_pool[occ][arm] = sims
        cp_kept_pool[occ][arm] = sims >= PRIMARY_GATE
        n_excl = (sims < PRIMARY_GATE).sum()
        print(f"  {occ:<22}  {arm:>9}  {len(pool_texts):>5}  "
              f"{n_excl:>6}  {n_excl/len(pool_texts)*100:>7.1f}%  {sims.mean():>9.4f}")
        cp_gate_rows.append({"sample": "pool", "occ": occ, "arm": arm,
                              "n": len(pool_texts), "n_excl": int(n_excl),
                              "mean_sim": float(sims.mean()),
                              "gate": PRIMARY_GATE})
        focal_texts = focal_bios[occ][arm]["text_clean"].tolist() \
                      if "text_clean" in focal_bios[occ][arm].columns else []
        if focal_texts:
            emb_fo = encode_plain_cached(focal_texts, main_model, f"{occ}_{arm}_focal_orig")
            mans_f = [r.manipulated for r in manip_focal[occ][arm]]
            emb_fm = encode_plain_cached(mans_f, main_model,
                                         f"{occ}_{arm}_focal_manip{_man_cache_sfx}")
            fsims  = np.diag(cos_sim(emb_fo, emb_fm))
            cp_sims_focal[occ][arm] = fsims
            fn_excl = (fsims < PRIMARY_GATE).sum()
            cp_gate_rows.append({"sample": "focal", "occ": occ, "arm": arm,
                                  "n": len(focal_texts), "n_excl": int(fn_excl),
                                  "mean_sim": float(fsims.mean()),
                                  "gate": PRIMARY_GATE})
        else:
            cp_sims_focal[occ][arm] = np.array([])

df_cp_gate = pd.DataFrame(cp_gate_rows)
print(f"\nCP filter done in {time.time()-_t_cp:.0f}s")

CP gate = 0.9  (plain model.encode, cached, batch=64)
Occ                           Arm      n  n_excl  pct_excl   mean_sim
------------------------------------------------------------------
  nurse                   communion    250       0      0.0%     0.9948
  nurse                      agency    250       0      0.0%     0.9972
  software_engineer       communion    250       0      0.0%     0.9972
  software_engineer          agency    250       0      0.0%     0.9985
  accountant              communion    250       0      0.0%     0.9973
  accountant                 agency    250       0      0.0%     0.9980
  architect               communion    250       1      0.4%     0.9969
  architect                  agency    250       0      0.0%     0.9984
  psychologist            communion    250       3      1.2%     0.9930
  psychologist               agency    250       2      0.8%     0.9961
  attorney                communion    250       0      0.0%     0.9965
  attorney       

## 11. Job description representation

Job descriptions are embedded whole, with a sliding window, so nothing is cut at the length limit of the encoder. The comparison at the end shows what the truncated representation would have given instead.

In [24]:
# Embed the job descriptions whole, with a sliding window.
_t_enc = time.time()
print(f"Embedding pools, focal bios, JDs with encode_long ...")
emb_orig_pool  = {}
emb_manip_pool = {}
emb_focal_orig = {}
emb_focal_manip= {}
emb_jd         = {}
_manip_cache_sfx = f"_{RUN_TAG}" if RUN_TAG != "run4" else ""

for occ in KEPT_13:
    pool_texts = bio_pools[occ]["text_clean"].tolist()
    jd_texts   = jd_pools[occ]["description"].tolist()
    emb_orig_pool[occ] = load_or_encode(pool_texts, main_model, f"{occ}_pool_orig")
    emb_jd[occ]        = load_or_encode(jd_texts,   main_model, f"{occ}_jds")
    emb_manip_pool[occ] = {}
    emb_focal_orig[occ] = {}
    emb_focal_manip[occ]= {}
    for arm in ("communion", "agency"):
        mans_pool  = [r.manipulated for r in manip_pool[occ][arm]]
        mans_focal = [r.manipulated for r in manip_focal[occ][arm]]
        emb_manip_pool[occ][arm] = load_or_encode(
            mans_pool, main_model, f"{occ}_{arm}_manip_pool{_manip_cache_sfx}")
        focal_texts = focal_bios[occ][arm]["text_clean"].tolist() \
                      if "text_clean" in focal_bios[occ][arm].columns else []
        if focal_texts:
            emb_focal_orig[occ][arm]  = load_or_encode(
                focal_texts, main_model, f"{occ}_{arm}_focal_orig")
            emb_focal_manip[occ][arm] = load_or_encode(
                mans_focal, main_model, f"{occ}_{arm}_manip_focal{_manip_cache_sfx}")
        else:
            emb_focal_orig[occ][arm]  = None
            emb_focal_manip[occ][arm] = None
    print(f"  {occ}  pool={len(pool_texts)}  jds={len(jd_texts)}")
print(f"Done. encode_long elapsed: {time.time()-_t_enc:.0f}s")

Embedding pools, focal bios, JDs with encode_long ...
  nurse  pool=250  jds=250
  software_engineer  pool=250  jds=250
  accountant  pool=250  jds=250
  architect  pool=250  jds=250
  psychologist  pool=250  jds=250
  attorney  pool=250  jds=250
  teacher  pool=250  jds=250
  surgeon  pool=250  jds=250
  physician  pool=250  jds=250
  professor  pool=250  jds=148
  dentist  pool=250  jds=250
  paralegal  pool=250  jds=250
  dietitian  pool=250  jds=77
Done. encode_long elapsed: 8083s


In [25]:
# Length of the job descriptions against the window of the encoder.
print(f"Window size: {list(emb_jd.values())[0]['window']} tokens")
print(f"{'Occ':<22}  {'n_jds':>6}  {'pct_multi':>9}  {'med_win':>7}  "
      f"{'med_tok':>8}  {'pct_seen_trunc':>14}")
print("-" * 75)
jd_len_rows = []
for occ in KEPT_13:
    jd = emb_jd[occ]
    nc = jd["n_chunks"]
    nt = jd["n_tokens"]
    totals    = np.array([nt[int(nc[:i].sum()):int(nc[:i].sum())+nc[i]].sum()
                          for i in range(len(nc))])
    first_win = np.array([nt[int(nc[:i].sum())] for i in range(len(nc))])
    pct_seen  = first_win / np.maximum(totals, 1) * 100
    print(f"  {occ:<22}  {len(nc):>6}  {(nc>1).mean()*100:>8.1f}%"
          f"  {np.median(nc):>7.1f}  {np.median(totals):>8.0f}"
          f"  {np.median(pct_seen):>13.1f}%")
    jd_len_rows.append({"occ": occ, "n_jds": int(len(nc)),
                         "pct_multi_chunk": float((nc>1).mean()*100),
                         "median_chunks": float(np.median(nc)),
                         "median_tokens": float(np.median(totals)),
                         "median_pct_seen_trunc": float(np.median(pct_seen))})
df_jd_lengths = pd.DataFrame(jd_len_rows)

Window size: 378 tokens
Occ                      n_jds  pct_multi  med_win   med_tok  pct_seen_trunc
---------------------------------------------------------------------------
  nurse                      250      61.2%      2.0       458           82.5%
  software_engineer          250      74.0%      2.0       656           57.4%
  accountant                 250      61.2%      2.0       494           76.4%
  architect                  250      69.6%      2.0       536           70.4%
  psychologist               250      80.8%      2.0       570           66.3%
  attorney                   250      50.0%      1.5       383           99.2%
  teacher                    250      75.2%      2.0       534           70.5%
  surgeon                    250      78.0%      2.0       598           63.2%
  physician                  250      61.2%      2.0       462           81.8%
  professor                  148      86.5%      3.0       907           41.7%
  dentist                    250 

In [26]:
# Delta from truncated job description embeddings, that is plain encoding cut
# at max_seq_length. The comparison below and the sensitivity section use it.
delta_trunc = {}
_t_trunc = time.time()
print(f"{'Occ':<22}  {'Arm':>9}  {'median_delta_trunc':>19}")
print("-" * 54)
for occ in KEPT_13:
    jd_texts  = jd_pools[occ]["description"].tolist()
    emb_jd_tr = encode_plain_cached(jd_texts, main_model, f"{occ}_jds_plain")
    sim_orig  = cp_emb_orig[occ] @ emb_jd_tr.T
    pct_orig  = study2_core.percentile_ranks(sim_orig)
    delta_trunc[occ] = {}
    for arm in ("communion", "agency"):
        sim_m     = cp_emb_manip[occ][arm] @ emb_jd_tr.T
        pct_indiv = study2_core.individual_cf_ranks(sim_orig, sim_m)
        delta_trunc[occ][arm] = pct_indiv - pct_orig
        print(f"  {occ:<22}  {arm:>9}  "
              f"{float(np.median(delta_trunc[occ][arm])):>19.3f}")
_all_zero = [occ for occ in KEPT_13 for arm in ("communion", "agency")
             if not np.any(delta_trunc[occ][arm])]
print(f"control occupation and arm combinations whose delta is all zero: "
      f"{len(_all_zero)} (expected 0)")
if _all_zero:
    raise AssertionError(
        f"The truncated-embedding delta is all zero for {_all_zero}; a "
        f"placeholder must not reach a result.")
print(f"truncated-embedding deltas computed in {time.time() - _t_trunc:.0f}s")

Occ                           Arm   median_delta_trunc
------------------------------------------------------
  nurse                   communion                0.044
  nurse                      agency                0.000
  software_engineer       communion                0.000
  software_engineer          agency                0.000
  accountant              communion                0.000
  accountant                 agency                0.000
  architect               communion                0.000
  architect                  agency                0.000
  psychologist            communion                0.000
  psychologist               agency                0.000
  attorney                communion                0.000
  attorney                   agency                0.000
  teacher                 communion                0.000
  teacher                    agency                0.000
  surgeon                 communion                0.124
  surgeon                    agency

In [27]:
# Truncated against full job description embeddings. Both use the plain
# biography embedding, so only the job description representation differs.
from scipy.stats import spearmanr as _spear10a
print("Truncated against full job description embeddings")
print(f"  {'Occ':<22}  {'cos_med':>8}  {'cos_p05':>8}  {'cos_p95':>8}  "
      f"{'rho_jd_med':>11}  {'rho_comm_ch':>11}  {'rho_ag_ch':>11}")
print("  " + "-" * 85)
_trf_rows = []
for occ in KEPT_13:
    jd_texts  = jd_pools[occ]["description"].tolist()
    n_jd      = len(jd_texts)
    _ejd_tr   = encode_plain_cached(jd_texts, main_model, f"{occ}_jds_plain")
    _ejd_full = emb_jd[occ]["mean"]
    # (a) cosine between truncated and full JD embeddings
    _nt   = np.linalg.norm(_ejd_tr,   axis=1, keepdims=True)
    _nf   = np.linalg.norm(_ejd_full, axis=1, keepdims=True)
    _cos  = ((_ejd_tr/_nt) * (_ejd_full/_nf)).sum(axis=1)
    _med_cos = float(np.median(_cos))
    _p05_cos = float(np.percentile(_cos, 5))
    _p95_cos = float(np.percentile(_cos, 95))
    # (b) per-JD Spearman: 250 bios ranked by sim_trunc vs sim_full
    _str  = cp_emb_orig[occ] @ _ejd_tr.T
    _sfl  = cp_emb_orig[occ] @ _ejd_full.T
    _rho_jd = [float(_spear10a(_str[:, j], _sfl[:, j])[0]) for j in range(n_jd)]
    _med_rho = float(np.median(_rho_jd))
    # (c) per arm: Spearman(delta_trunc, delta_full_approx) over changed bios
    _row = {"occ": occ, "median_cos": _med_cos, "p05_cos": _p05_cos,
            "p95_cos": _p95_cos, "median_spearman_per_jd": _med_rho}
    _rho_arm = {}
    for arm in ("communion", "agency"):
        _dt = delta_trunc[occ][arm]
        _smo = study2_core.percentile_ranks(_sfl)
        _smm = study2_core.individual_cf_ranks(
            _sfl, cp_emb_manip[occ][arm] @ _ejd_full.T)
        _dfl = _smm - _smo
        _changed = [i for i in range(len(_dt))
                    if manip_pool[occ][arm][i].n_replaced
                       + manip_pool[occ][arm][i].n_deleted >= 1]
        if len(_changed) > 5:
            _rc = float(_spear10a(_dt[_changed], _dfl[_changed])[0])
        else:
            _rc = float("nan")
        _row[f"spearman_{arm}_changed"] = _rc
        _rho_arm[arm] = _rc
    _trf_rows.append(_row)
    print(f"  {occ:<22}  {_med_cos:>8.4f}  {_p05_cos:>8.4f}  {_p95_cos:>8.4f}  "
          f"{_med_rho:>11.4f}  "
          f"{_rho_arm.get('communion',float('nan')):>11.4f}  "
          f"{_rho_arm.get('agency',float('nan')):>11.4f}")
df_trunc_vs_full = pd.DataFrame(_trf_rows)
print()

Truncated against full job description embeddings
  Occ                      cos_med   cos_p05   cos_p95   rho_jd_med  rho_comm_ch    rho_ag_ch
  -------------------------------------------------------------------------------------
  nurse                     0.9903    0.8806    1.0000       0.9966       0.9989       0.9931
  software_engineer         0.9389    0.8219    1.0000       0.9767       0.9878       0.9873
  accountant                0.9793    0.8554    1.0000       0.9925       0.9910       0.9866
  architect                 0.9736    0.8442    1.0000       0.9907       0.9968       0.9938
  psychologist              0.9664    0.8734    1.0000       0.9885       0.9955       0.9882
  attorney                  0.9999    0.8917    1.0000       1.0000       0.9972       0.9982
  teacher                   0.9736    0.8532    1.0000       0.9935       0.9949       0.9868
  surgeon                   0.9562    0.8462    1.0000       0.9723       0.9962       0.9924
  physician     

## 12. Main delta computation

For every biography, arm and job description: the percentile rank of the original, the rank of the manipulated version, and the change between them.

In [28]:
# Percentile rank of every biography before and after the manipulation.
_t_main = time.time()
records_r5_pool  = []
records_r5_focal = []

print(f"Primary run: JD_REPR={JD_REPR}  gate applied at analysis stage")
for occ in KEPT_13:
    sim_orig_pool = study2_core.jd_similarity(
        emb_orig_pool[occ]["mean"], emb_jd[occ], how=JD_REPR)
    pct_orig_pool = study2_core.percentile_ranks(sim_orig_pool)

    for arm in ("communion", "agency"):
        sim_m     = study2_core.jd_similarity(
            emb_manip_pool[occ][arm]["mean"], emb_jd[occ], how=JD_REPR)
        pct_indiv = study2_core.individual_cf_ranks(sim_orig_pool, sim_m)
        delta_ind = pct_indiv - pct_orig_pool
        dz        = study2_core.delta_z_per_jd(
            sim_orig_pool, sim_orig_pool, sim_m).mean(axis=1)

        # The v2 result counts replacements and deletions separately. Both are
        # kept, and n_subs is their sum: the number of occurrences that the
        # manipulation touched in this biography.
        for i in range(len(bio_pools[occ])):
            r = manip_pool[occ][arm][i]
            records_r5_pool.append({
                "occ": occ, "bio_idx": int(i), "arm": arm,
                "female":  int(bio_pools[occ].iloc[i]["female"])
                           if "female" in bio_pools[occ].columns else np.nan,
                "delta":   float(delta_ind[i]),
                "delta_z": float(dz[i]),
                "pct_orig": float(pct_orig_pool[i]),
                "pct_indiv": float(pct_indiv[i]),
                "n_subs":  int(r.n_replaced + r.n_deleted),
                "n_replaced": int(r.n_replaced),
                "n_deleted":  int(r.n_deleted),
                "sim_cp":  float(cp_sims_pool[occ][arm][i]),
            })

        if emb_focal_orig[occ][arm] is not None:
            focal_df = focal_bios[occ][arm]
            sim_fo   = study2_core.jd_similarity(
                emb_focal_orig[occ][arm]["mean"], emb_jd[occ], how=JD_REPR)
            sim_fm   = study2_core.jd_similarity(
                emb_focal_manip[occ][arm]["mean"], emb_jd[occ], how=JD_REPR)
            pct_fb   = study2_core.focal_pct_per_jd(sim_orig_pool, sim_fo).mean(axis=1)
            pct_fa   = study2_core.focal_pct_per_jd(sim_orig_pool, sim_fm).mean(axis=1)
            df_f     = pct_fa - pct_fb
            dz_f     = study2_core.delta_z_per_jd(
                sim_orig_pool, sim_fo, sim_fm).mean(axis=1)
            for j in range(len(focal_df)):
                rf = manip_focal[occ][arm][j]
                records_r5_focal.append({
                    "occ": occ, "focal_idx": int(j), "arm": arm,
                    "female": int(focal_df.iloc[j]["female"])
                              if "female" in focal_df.columns else np.nan,
                    "delta":  float(df_f[j]),
                    "delta_z": float(dz_f[j]),
                    "pct_orig": float(pct_fb[j]), "pct_after": float(pct_fa[j]),
                    "n_subs":  int(rf.n_replaced + rf.n_deleted),
                    "n_replaced": int(rf.n_replaced),
                    "n_deleted":  int(rf.n_deleted),
                    "sim_cp":  float(cp_sims_focal[occ][arm][j])
                               if j < len(cp_sims_focal[occ][arm]) else np.nan,
                })
    print(f"  {occ} done.")

df_pool  = pd.DataFrame(records_r5_pool)
df_focal = pd.DataFrame(records_r5_focal)
print(f"\nPool records:  {len(df_pool)}  focal records: {len(df_focal)}")
print(f"Main delta elapsed: {time.time()-_t_main:.0f}s")

Primary run: JD_REPR=mean  gate applied at analysis stage
  nurse done.
  software_engineer done.
  accountant done.
  architect done.
  psychologist done.
  attorney done.
  teacher done.
  surgeon done.
  physician done.
  professor done.
  dentist done.
  paralegal done.
  dietitian done.

Pool records:  6500  focal records: 6500
Main delta elapsed: 8s


## 13. Controls against Run 4

- **Question:** Does the new code reproduce the ranking of Run 4 where the
  inputs are the same, and do the changes in rank follow the change in the rate?
- **Test:** biography; the original ranks against Run 4, and the rank
  correlation between the change in rate and the change in rank.
- **Status:** control, fixed 2026-10-11; the regression test of the v1
  manipulation removed 2026-10-12.
- **Output:** `control_original_ranks.csv`, `manip_quintiles.csv`,
  `manip_spearman.csv`, `manipulation_rates.png`

The regression test of the v1 manipulation under A $\cup$ B is no longer part
of this notebook. The v1 texts cannot be produced identically in today's
environment: all 26 arrays of the pool differ from those of Run 4 in the hash of
their texts, which `scripts/reuse_run4_v1_embeddings.py` records. A test on
different texts does not test what it was meant to test, which is whether the
new code reproduces the old result. What the ranking does with identical inputs
is tested by the control of the original ranks below.

In [29]:
# Control: the original ranks of the pool biographies against Run 4. The inputs
# are the same in both runs, the biographies and the job descriptions of the
# pool, so the ranks have to agree. The criterion of 2026-10-04: a deviation
# reaches at most five steps of the rank grid, and at most ten per cent of the
# biographies deviate at all.
#
# The comparison is defined on the full pool, because the percentile ranks of
# Run 4 sit on the grid of 250 biographies per occupation.
MAX_RANK_STEPS = 5.0
MAX_SHARE_DEVIATING = 0.10

if SMOKE:
    print("control original ranks against Run 4: skipped in the test run; it "
          "draws fewer biographies per occupation, so its percentile ranks sit "
          "on a different grid. The saving cell writes "
          "control_original_ranks.csv in the full run only.")
else:
    from scipy.stats import spearmanr as _sp_ranks
    _r4_pool_path = Path(f"{RUN4_BASE}/study2_delta_r4_pool.csv")
    if not _r4_pool_path.exists():
        raise FileNotFoundError(
            f"{_r4_pool_path} not found; the control compares Run 5 against the "
            f"saved Run 4 results.")
    _r4 = pd.read_csv(_r4_pool_path)
    _r5 = pd.DataFrame(records_r5_pool)
    print(f"control Run 5 pool records: {len(_r5)} (expected more than 0)")
    if _r5.empty:
        raise AssertionError(
            "There are no pool records, so there is nothing to compare with "
            "Run 4; the main delta section has to run before this one.")
    _merged = _r5.merge(_r4[["occ", "bio_idx", "arm", "pct_orig"]],
                        on=["occ", "bio_idx", "arm"], how="inner",
                        suffixes=("", "_run4"))
    print(f"bios compared: {len(_merged)} of {len(_r5)} in Run 5 "
          f"({_merged['occ'].nunique()} occupations)")
    if _merged.empty:
        raise AssertionError("No overlap with Run 4; check occ and bio_idx.")

    # One step of the rank grid, and what counts as a deviation: anything above
    # the numerical noise of the percentile computation.
    _step = 100.0 / (len(bio_pools[KEPT_13[0]]) - 1)
    _merged["dev"] = (_merged["pct_orig"] - _merged["pct_orig_run4"]).abs()
    _rows = []
    print(f"one step of the rank grid: {_step:.4f} percentile points")
    print(f"  {'occ':<20} {'n':>5} {'identical':>10} {'deviating':>10} "
          f"{'max_dev':>9} {'steps':>7} {'rho':>7} {'p':>9}")
    print("  " + "-" * 86)
    for occ in sorted(_merged["occ"].unique()):
        _s = _merged[_merged["occ"] == occ]
        _identical = int((_s["dev"] < 1e-9).sum())
        _n_dev = int((_s["dev"] >= 1e-9).sum())
        _max_dev = float(_s["dev"].max())
        _share_dev = _n_dev / len(_s)
        # The rank correlation needs both ranks to vary; on a pool of 250
        # biographies they do, and a constant column would be a failure.
        if _s["pct_orig"].nunique() < 2 or _s["pct_orig_run4"].nunique() < 2:
            raise AssertionError(
                f"The original ranks of {occ} do not vary in one of the two "
                f"runs, so the rank correlation is not defined.")
        _rho, _p_rho = _sp_ranks(_s["pct_orig"], _s["pct_orig_run4"])
        print(f"  {occ:<20} {len(_s):>5} {_identical:>10} {_n_dev:>10} "
              f"{_max_dev:>9.4f} {_max_dev / _step:>7.2f} {float(_rho):>7.4f} "
              f"{float(_p_rho):>9.2e}")
        _rows.append({"occ": occ, "n": len(_s), "n_identical": _identical,
                      "n_deviating": _n_dev,
                      "share_deviating": float(_share_dev),
                      "max_deviation": _max_dev,
                      "max_rank_steps": _max_dev / _step,
                      "spearman_rho": float(_rho),
                      "spearman_p": float(_p_rho)})
    df_ctrl_ranks = pd.DataFrame(_rows)

    _worst_steps = float(df_ctrl_ranks["max_rank_steps"].max())
    _worst_share = float(df_ctrl_ranks["share_deviating"].max())
    _lowest_rho = float(df_ctrl_ranks["spearman_rho"].min())
    print(f"largest deviation: {df_ctrl_ranks['max_deviation'].max():.4f} "
          f"percentile points, {_worst_steps:.2f} steps "
          f"(criterion {MAX_RANK_STEPS:.0f})")
    print(f"largest share of deviating biographies: {_worst_share:.1%} "
          f"(criterion {MAX_SHARE_DEVIATING:.0%})")
    print(f"lowest rank correlation over the occupations: {_lowest_rho:.4f}")
    _failed = df_ctrl_ranks[
        (df_ctrl_ranks["max_rank_steps"] > MAX_RANK_STEPS + 1e-9)
        | (df_ctrl_ranks["share_deviating"] > MAX_SHARE_DEVIATING + 1e-9)]
    if len(_failed):
        print(_failed.to_string(index=False))
        raise AssertionError(
            f"{len(_failed)} occupations are outside the criterion of "
            f"2026-10-04: at most {MAX_RANK_STEPS:.0f} steps of the rank grid "
            f"and at most {MAX_SHARE_DEVIATING:.0%} deviating biographies.")
    print("control passed: the original ranks reproduce Run 4 within the "
          "criterion")

control Run 5 pool records: 6500 (expected more than 0)
bios compared: 6500 of 6500 in Run 5 (13 occupations)
one step of the rank grid: 0.4016 percentile points
  occ                      n  identical  deviating   max_dev   steps     rho         p
  --------------------------------------------------------------------------------------
  accountant             500        500          0    0.0000    0.00  1.0000  0.00e+00
  architect              500        500          0    0.0000    0.00  1.0000  0.00e+00
  attorney               500        500          0    0.0000    0.00  1.0000  0.00e+00
  dentist                500        500          0    0.0000    0.00  1.0000  0.00e+00
  dietitian              500        500          0    0.0000    0.00  1.0000  0.00e+00
  nurse                  500        500          0    0.0000    0.00  1.0000  0.00e+00
  paralegal              500        500          0    0.0000    0.00  1.0000  0.00e+00
  physician              500        500          0   

In [30]:
# Manipulation check, continued: fifths of the starting rate within occupation,
# the rank correlation with the change in rank, and the length model.
_mc = df_manip_check.merge(
    pd.DataFrame(records_r5_pool)[["occ", "bio_idx", "arm", "delta", "sim_cp"]],
    on=["occ", "bio_idx", "arm"], how="left")
if "delta" not in _mc.columns:
    raise AssertionError(
        "The change in rank is missing, so the manipulation check cannot be "
        "related to it; the main delta section has to run first.")
_pool_rows = _mc[_mc["sample"] == "pool"]
_missing_delta = int(_pool_rows["delta"].isna().sum())
print(f"control pool rows without a change in rank: {_missing_delta} of "
      f"{len(_pool_rows)} (expected 0)")
if _missing_delta:
    raise AssertionError(
        f"{_missing_delta} biographies of the pool have no change in rank, "
        f"so the manipulation check and the main computation do not cover "
        f"the same biographies.")
_mc["quintile"] = (_mc.groupby(["occ", "arm", "sample"])["rate_before"]
                      .transform(lambda s: pd.qcut(s.rank(method="first"), 5,
                                                   labels=False,
                                                   duplicates="drop") + 1))
print("Fifths of the starting rate (within occupation), POOL:")
print(f"  {'arm':<10} {'q':>2} {'n':>5} {'changed':>8} {'d_rate':>8} "
      f"{'sim_cp':>8} {'gated_out':>10}")
_q_rows = []
for arm in ("communion", "agency"):
    for q in sorted(_mc.loc[(_mc["arm"] == arm) & (_mc["sample"] == "pool"),
                            "quintile"].dropna().unique()):
        _s = _mc[(_mc["arm"] == arm) & (_mc["sample"] == "pool")
                 & (_mc["quintile"] == q)]
        _gated = float((_s["sim_cp"] < PRIMARY_GATE).mean())
        print(f"  {arm:<10} {int(q):>2} {len(_s):>5} "
              f"{int(_s['changed'].sum()):>8} {_s['delta_rate'].mean():>8.3f} "
              f"{_s['sim_cp'].mean():>8.4f} {_gated:>10.1%}")
        _q_rows.append({"arm": arm, "quintile": int(q), "n": len(_s),
                        "n_changed": int(_s["changed"].sum()),
                        "mean_delta_rate": float(_s["delta_rate"].mean()),
                        "mean_sim_cp": float(_s["sim_cp"].mean()),
                        "share_gated_out": _gated})
df_manip_quintiles = pd.DataFrame(_q_rows)

print()
print("Spearman between the change in rate and the change in percentile rank "
      "(changed bios only):")
_sp_rows = []
for sample in ("pool",):
    for arm in ("communion", "agency"):
        _s = _mc[(_mc["sample"] == sample) & (_mc["arm"] == arm) & _mc["changed"]]
        _s = _s.dropna(subset=["delta_rate", "delta"])
        if len(_s) < 10:
            continue
        _rho, _p = spearmanr(_s["delta_rate"], _s["delta"])
        print(f"  {sample:<6} {arm:<10} n={len(_s):>5}  rho={_rho:>+6.3f}  p={_p:.4g}")
        _sp_rows.append({"sample": sample, "arm": arm, "n": len(_s),
                         "rho": float(_rho), "p": float(_p)})
df_manip_spearman = pd.DataFrame(_sp_rows)

print()
print("Change in percentile rank by kind of change, and the length model:")
for arm in ("communion", "agency"):
    _s = _mc[(_mc["sample"] == "pool") & (_mc["arm"] == arm) & _mc["changed"]].copy()
    if _s.empty:
        raise AssertionError(
            f"No biography of the pool was changed for {arm}, so the kind of "
            f"change cannot be related to the change in rank.")
    _s["kind"] = np.where(_s["n_deleted"] == 0, "replaced only",
                  np.where(_s["n_replaced"] == 0, "deleted only", "both"))
    for kind, group in _s.groupby("kind"):
        print(f"  {arm:<10} {kind:<14} n={len(group):>5}  "
              f"median delta rank {group['delta'].median():>+7.3f}  "
              f"median delta words {group['delta_words'].median():>+5.1f}")
    _fit = smf.ols("delta ~ n_replaced + n_deleted + delta_words + C(occ)",
                   data=_s).fit(cov_type="HC1")
    for term in ("n_replaced", "n_deleted", "delta_words"):
        print(f"  {arm:<10} OLS {term:<13} "
              f"beta={_fit.params[term]:>+8.4f}  se={_fit.bse[term]:>7.4f}  "
              f"p={_fit.pvalues[term]:.4g}")

control pool rows without a change in rank: 0 of 6500 (expected 0)
Fifths of the starting rate (within occupation), POOL:
  arm         q     n  changed   d_rate   sim_cp  gated_out
  communion   1   650       34   -0.075   0.9995       0.0%
  communion   2   650      317   -0.927   0.9970       0.2%
  communion   3   650      449   -1.735   0.9949       0.2%
  communion   4   650      530   -2.427   0.9930       0.5%
  communion   5   650      548   -3.420   0.9896       0.5%
  agency      1   650        0    0.000   1.0000       0.0%
  agency      2   650       45   -0.085   0.9997       0.0%
  agency      3   650      214   -0.522   0.9980       0.2%
  agency      4   650      379   -1.270   0.9944       0.3%
  agency      5   650      455   -2.239   0.9919       0.9%

Spearman between the change in rate and the change in percentile rank (changed bios only):
  pool   communion  n= 1878  rho=-0.044  p=0.05507
  pool   agency     n= 1093  rho=-0.168  p=2.391e-08

Change in percentile 

In [31]:
# Figure of the rate of the manipulated dimension before and after the
# manipulation, per arm and sample.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

_fig_dir = Path("outputs/figures/study2")
_fig_dir.mkdir(parents=True, exist_ok=True)

_samples = [s for s in ("pool", "focal")
            if not df_manip_check[df_manip_check["sample"] == s].empty]
_fig, _axes = plt.subplots(len(_samples), 2,
                           figsize=(11, 4 * len(_samples)), squeeze=False)
for _row, _sample in enumerate(_samples):
    for _col, _arm in enumerate(("communion", "agency")):
        _ax = _axes[_row][_col]
        _s = df_manip_check[(df_manip_check["sample"] == _sample)
                            & (df_manip_check["arm"] == _arm)]
        if _s.empty:
            raise AssertionError(
                f"The manipulation check has no rows for {_sample}/{_arm}, so "
                f"the figure would hold an empty panel.")
        _bins = np.histogram_bin_edges(
            pd.concat([_s["rate_before"], _s["rate_after"]]), bins=30)
        _ax.hist(_s["rate_before"], bins=_bins, alpha=0.6, label="before",
                 color="#4C72B0")
        _ax.hist(_s["rate_after"], bins=_bins, alpha=0.6, label="after",
                 color="#DD8452")
        _ax.set_title(f"{_arm}, {_sample} (n={len(_s)})")
        _ax.set_xlabel(f"{_arm} rate (hits per 100 words)")
        _ax.set_ylabel("biographies")
        _ax.legend()
plt.tight_layout()
_p_png = str(_fig_dir / "manipulation_rates.png")
_p_pdf = str(_fig_dir / "manipulation_rates.pdf")
_fig.savefig(_p_png, dpi=300, bbox_inches="tight")
_fig.savefig(_p_pdf, bbox_inches="tight")
plt.close(_fig)
print(f"figure: {_p_png}")
print(f"figure: {_p_pdf}")

figure: outputs\figures\study2\manipulation_rates.png
figure: outputs\figures\study2\manipulation_rates.pdf


## 14. Results

### 14.1 Treatment table

- **Question:** Does the manipulation move the ranking at all?
- **Expectation:** the rank falls when the matching dimension is lowered.
- **Test:** biography; Wilcoxon on the change in rank, per arm and sample.
- **Status:** pre-specified test, fixed 2026-10-05.
- **Output:** `study2_treatment_pool.csv`, `study2_treatment_focal.csv`

In [32]:
# Treatment table: median change and Wilcoxon per arm and sample.
BONF = 0.05 / 26
print(f"Bonferroni threshold: p < {BONF:.5f}")

def _gated(df, gate):
    if gate is None:
        return df
    return df[df["sim_cp"] >= gate]

treatment_tables = {}
for label, df_src in [("POOL", df_pool), ("FOCAL", df_focal)]:
    dg = _gated(df_src, PRIMARY_GATE)
    if dg.empty:
        raise AssertionError(
            f"No biography of {label} passes the gate of {PRIMARY_GATE}. With "
            f"{len(df_src)} rows in the sample that means the similarity "
            f"before the manipulation was not formed.")
    print(f"\n=== {label} (gate={PRIMARY_GATE}) - delta (rank) ===")
    t_occ = study2_core.treatment_table(dg, ["occ","arm"])
    t_all = study2_core.treatment_table(dg, ["arm"])
    t_all.insert(0, "occ", "POOLED")
    t_full = pd.concat([t_occ, t_all], ignore_index=True)
    cols = [c for c in ["occ","arm","n","share_unchanged","share_delta_neg",
            "share_delta_pos","median_all","p_wilcoxon_all","n_treated",
            "median_treated","p_wilcoxon_treated","rho_dose_signed","p_dose_signed"]
            if c in t_full.columns]
    print(t_full[cols].to_string(index=False))
    treatment_tables[label.lower()] = t_full

    print(f"\n=== {label} (gate={PRIMARY_GATE}) - delta_z ===")
    if "delta_z" in dg.columns:
        tz_all = study2_core.treatment_table(dg, ["arm"], delta_col="delta_z")
        tz_all.insert(0, "occ", "POOLED")
        tz_cols = [c for c in ["occ","arm","n","share_unchanged","median_all",
                   "p_wilcoxon_all","n_treated","median_treated","p_wilcoxon_treated"]
                   if c in tz_all.columns]
        print(tz_all[tz_cols].to_string(index=False))
        treatment_tables[f"{label.lower()}_z"] = tz_all

Bonferroni threshold: p < 0.00192

=== POOL (gate=0.9) - delta (rank) ===
              occ       arm    n  share_unchanged  share_delta_neg  share_delta_pos  median_all  p_wilcoxon_all  n_treated  median_treated  p_wilcoxon_treated  rho_dose_signed  p_dose_signed
            nurse communion  250         0.304000         0.188000         0.504000    0.041767    5.598296e-13        174        1.844980        5.598296e-13         0.061391   4.209774e-01
            nurse    agency  250         0.812000         0.068000         0.116000    0.000000    1.634403e-01         47        0.441767        1.634403e-01         0.333262   2.207469e-02
software_engineer communion  250         0.632000         0.164000         0.204000    0.000000    4.054145e-02         92        0.559839        4.054145e-02         0.203782   5.136922e-02
software_engineer    agency  250         0.804000         0.120000         0.076000    0.000000    1.909283e-01         49       -0.269880        1.909283e-01    

### 14.2 By author gender

- **Question:** Does the change differ between biographies of women and of
  men?
- **Test:** biography; Wilcoxon per group and Mann-Whitney between the groups.
- **Status:** pre-specified test, fixed 2026-10-05.
- **Output:** `study2_gender_pool.csv`, `study2_gender_focal.csv`

In [33]:
# The change split by the gender of the author.
gender_frames = {}
from scipy.stats import mannwhitneyu as _mwu_12b
for label, df_src in [("POOL", df_pool), ("FOCAL", df_focal)]:
    dg = _gated(df_src, PRIMARY_GATE)
    if dg.empty or "female" not in dg.columns:
        continue
    print(f"=== {label} (gate={PRIMARY_GATE}) - by gender ===")
    tg = study2_core.treatment_table(dg, ["arm","female"])
    print(tg[["arm","female","n","share_unchanged","median_all","p_wilcoxon_all",
               "median_treated","p_wilcoxon_treated"]].to_string(index=False))
    # Mann-Whitney U: women vs men on delta of changed bios (n_subs >= 1)
    print(f"\n  Mann-Whitney U (women vs men, changed bios, gate={PRIMARY_GATE}):")
    mw_rows = []
    for arm in ("communion","agency"):
        sub = dg[(dg["arm"] == arm) & (dg["n_subs"] >= 1)]
        f_vals = sub[sub["female"] == 1]["delta"].values
        m_vals = sub[sub["female"] == 0]["delta"].values
        if len(f_vals) >= 5 and len(m_vals) >= 5:
            _U, _p = _mwu_12b(f_vals, m_vals, alternative="two-sided")
            print(f"  {arm:<10}  nF={len(f_vals):>4}  nM={len(m_vals):>4}  "
                  f"U={_U:.0f}  p={_p:.4f}")
            mw_rows.append({"sample": label.lower(), "arm": arm,
                             "n_female": len(f_vals), "n_male": len(m_vals),
                             "U": float(_U), "p_mwu": float(_p)})
        else:
            print(f"  {arm:<10}  insufficient data (nF={len(f_vals)}, nM={len(m_vals)})")
    if mw_rows:
        tg["mwu_p"] = tg.apply(
            lambda r: next((x["p_mwu"] for x in mw_rows
                             if x["arm"]==r["arm"]), float("nan")), axis=1)
    print()
    gender_frames[label.lower()] = tg

=== POOL (gate=0.9) - by gender ===
      arm  female    n  share_unchanged  median_all  p_wilcoxon_all  median_treated  p_wilcoxon_treated
communion       0 1660         0.472289         0.0    9.228118e-11        0.330739        9.228118e-11
communion       1 1582         0.371681         0.0    1.110282e-09        0.374297        1.110282e-09
   agency       0 1660         0.642169         0.0    3.041805e-03        0.063454        3.041805e-03
   agency       1 1581         0.690070         0.0    1.498935e-06        0.341365        1.498935e-06

  Mann-Whitney U (women vs men, changed bios, gate=0.9):
  communion   nF= 994  nM= 876  U=431639  p=0.7487
  agency      nF= 490  nM= 594  U=153670  p=0.1126

=== FOCAL (gate=0.9) - by gender ===
      arm  female    n  share_unchanged  median_all  p_wilcoxon_all  median_treated  p_wilcoxon_treated
communion       1 1864         0.000000    0.449600    6.235764e-13        0.449600        6.235764e-13
communion       0 1305         0.00000

### 14.3 Dimension separation

- **Question:** Does lowering one dimension move the ranking through that
  dimension rather than through the other one?
- **Expectation:** the matching dimension carries the change.
- **Test:** biography; OLS with occupation effects, HC1.
- **Status:** pre-specified test, fixed 2026-10-05.
- **Output:** `study2_ols.csv`

In [34]:
# Does the manipulated dimension carry the change? OLS with occupation effects.
ols_rows = []
for label, df_src in [("POOL", df_pool)]:
    dg = _gated(df_src, PRIMARY_GATE)
    if dg.empty:
        raise AssertionError(
            f"No biography of the pool passes the gate of {PRIMARY_GATE}, so "
            f"the regression has no data.")
    print(f"=== {label} (gate={PRIMARY_GATE}) - OLS ===")
    for arm in ("communion", "agency"):
        sub = dg[dg["arm"] == arm].copy()
        # The model holds the occupation fixed and needs enough biographies.
        if len(sub) < 30:
            print(f"  {arm}: left out, {len(sub)} biographies pass the gate, "
                  f"30 needed")
            continue
        for occ in KEPT_13:
            idx = sub[sub["occ"] == occ]["bio_idx"].values
            bp  = bio_pools[occ]
            for col in ("communion_rate","agency_rate","word_count"):
                if col in bp.columns:
                    sub.loc[sub["occ"]==occ, col] = bp[col].values[idx]
        sub["log_wc"]  = np.log(sub["word_count"].clip(lower=1))
        other_col = "agency_rate" if arm == "communion" else "communion_rate"
        _fit_data = sub.dropna(subset=["n_subs", other_col, "log_wc"])
        _constant = [_c for _c in ("n_subs", other_col, "log_wc")
                     if _fit_data[_c].nunique() < 2]
        print(f"  control {arm}: {len(_fit_data)} rows with all predictors, "
              f"occupations {_fit_data['occ'].nunique()}, "
              f"predictors without variation {_constant}")
        if len(_fit_data) < 30 or _constant or _fit_data["occ"].nunique() < 2:
            raise AssertionError(
                f"The regression for {arm} cannot be fitted: {len(_fit_data)} "
                f"rows, {_fit_data['occ'].nunique()} occupations, predictors "
                f"without variation {_constant}.")
        res = smf.ols(f"delta ~ n_subs + {other_col} + log_wc + C(occ)",
                      data=_fit_data).fit(cov_type="HC1")
        print(f"  {arm.upper()}: n={len(sub)}  R^2={res.rsquared:.3f}  "
              f"b_nsubs={res.params['n_subs']:+.4f} p={res.pvalues['n_subs']:.4f}  "
              f"b_{other_col}={res.params[other_col]:+.4f} p={res.pvalues[other_col]:.4f}")
        for pname in res.params.index:
            ols_rows.append({"sample": label.lower(), "arm": arm,
                              "param": pname, "coef": float(res.params[pname]),
                              "se": float(res.bse[pname]), "p": float(res.pvalues[pname])})
df_ols = pd.DataFrame(ols_rows)

=== POOL (gate=0.9) - OLS ===
  control communion: 3242 rows with all predictors, occupations 13, predictors without variation []
  COMMUNION: n=3242  R^2=0.081  b_nsubs=+0.3018 p=0.0000  b_agency_rate=-0.0393 p=0.0770
  control agency: 3241 rows with all predictors, occupations 13, predictors without variation []
  AGENCY: n=3241  R^2=0.075  b_nsubs=+0.3789 p=0.0000  b_communion_rate=-0.0009 p=0.9502


### 14.4 Language of the job description

- **Question:** Is the change larger against advertisements written in the
  manipulated dimension?
- **Expectation:** the change grows with the density of that dimension.
- **Test:** job description; Spearman within occupation with a pooled
  permutation test.
- **Status:** pre-specified test, fixed 2026-10-05; the rows without the
  function words of D1 are a robustness variant.
- **Output:** `study2_jd_language_test.csv`,
  `study2_jd_language_test_per_occ.csv`

In [35]:
# Density statistics of the job descriptions, and their length.
# N_tokens correlation added here because emb_jd (encode_long) is needed
from scipy.stats import spearmanr as _spear_12d

print("Job description densities and their length")
print(f"  {'Occ':<22}  {'mean_comm':>9}  {'mean_ag':>8}  "
      f"{'rho_comm_ag':>12}  {'p':>7}  {'rho_comm_ntok':>14}  {'p':>7}  "
      f"{'rho_ag_ntok':>12}  {'p':>7}")
print("  " + "-" * 100)
for occ in KEPT_13:
    _cd = jd_comm_dens[occ]; _ad = jd_ag_dens[occ]
    _nc = emb_jd[occ]["n_chunks"]
    _nt_arr = emb_jd[occ]["n_tokens"]
    _nt = np.array([int(_nt_arr[int(_nc[:i].sum()): int(_nc[:i].sum()) + _nc[i]].sum())
                    for i in range(len(_nc))])
    _rca, _pca   = _spear_12d(_cd, _ad)
    _rcnt, _pcnt = _spear_12d(_cd, _nt)
    _rant, _pant = _spear_12d(_ad, _nt)
    print(f"  {occ:<22}  {_cd.mean():>9.3f}  {_ad.mean():>8.3f}  "
          f"{_rca:>+12.3f}  {_pca:>7.4f}  {_rcnt:>+14.3f}  {_pcnt:>7.4f}  "
          f"{_rant:>+12.3f}  {_pant:>7.4f}")

Job description densities and their length
  Occ                     mean_comm   mean_ag   rho_comm_ag        p   rho_comm_ntok        p   rho_ag_ntok        p
  ----------------------------------------------------------------------------------------------------
  nurse                       7.146     4.896        +0.366   0.0000          +0.094   0.1386        +0.048   0.4454
  software_engineer           4.752     5.027        +0.193   0.0022          +0.551   0.0000        -0.050   0.4357
  accountant                  5.389     5.161        +0.163   0.0100          +0.293   0.0000        +0.106   0.0942
  architect                   4.472     5.227        +0.121   0.0557          +0.535   0.0000        +0.067   0.2924
  psychologist                7.195     5.388        +0.107   0.0928          +0.087   0.1698        +0.077   0.2264
  attorney                    6.988     4.943        +0.161   0.0106          +0.136   0.0315        +0.126   0.0465
  teacher                     7.476

In [36]:
# Change against the density of the job description, within occupation.
_t_jdtest = time.time()
jd_test_results = {}   # key -> {"tab": df, "pool": dict}

def _run_jd_lang(sample_label, df_src, _tag, delta_key, get_delta_j):
    # Run JD language test for one delta variant (delta or delta_z)
    dg = _gated(df_src, PRIMARY_GATE)
    dg = dg[dg["n_subs"] >= 1]
    if dg.empty:
        raise AssertionError(
            f"No changed biography of {sample_label} passes the gate of "
            f"{PRIMARY_GATE}, so the language test has no data.")
    for arm in ("communion", "agency"):
        # The same test also on the densities without the
        # D1 function words, as extra rows; the main rows stay unchanged.
        for jd_dim, dens_dict in [("communion", jd_comm_dens),
                                  ("agency", jd_ag_dens),
                                  ("communion_cw", jd_comm_dens_cw),
                                  ("agency_cw", jd_ag_dens_cw)]:
            cl = "MATCHED" if arm == jd_dim.replace("_cw", "") else "CROSSED"
            print(f"\n=== {sample_label}  arm={arm}  JD-dim={jd_dim} ({cl})  [{delta_key}] ===")
            print(f"  {'Occ':<22}  {'rho':>6}  {'p_rho':>7}  "
                  f"{'diff_hi_lo':>10}  {'p_diff':>7}  {'n_jds':>6}")
            print("  " + "-" * 65)
            delta_j_by_occ = {}
            for occ in KEPT_13:
                sub = dg[(dg["occ"] == occ) & (dg["arm"] == arm)]
                if len(sub) < 5:
                    continue
                idx = sub["bio_idx"].values.astype(int) if _tag == "pool" \
                      else sub["focal_idx"].values.astype(int)
                dj = get_delta_j(occ, arm, _tag, idx)
                if dj is None:
                    continue
                delta_j_by_occ[occ] = dj
            dens_sub  = {o: dens_dict[o] for o in delta_j_by_occ if o in dens_dict}
            delta_sub = {o: delta_j_by_occ[o] for o in delta_j_by_occ
                         if o in dens_sub and len(dens_sub[o]) == len(delta_j_by_occ[o])}
            for occ in sorted(delta_sub):
                # A rank correlation needs at least three job descriptions.
                if len(delta_sub[occ]) < 3:
                    print(f"  {occ:<22}  left out: {len(delta_sub[occ])} job "
                          f"descriptions, 3 needed")
                    continue
                _, po = study2_core.jd_language_test(
                    {occ: delta_sub[occ]}, {occ: dens_sub[occ]},
                    n_perm=N_PERM, seed=SEED)
                print(f"  {occ:<22}  {po.get('rho',float('nan')):>+6.3f}  "
                      f"{po.get('p_rho',float('nan')):>7.4f}  "
                      f"{po.get('diff_hi_lo',float('nan')):>+10.3f}  "
                      f"{po.get('p_diff',float('nan')):>7.4f}  "
                      f"{po.get('n_jds',0):>6}")
            if not delta_sub:
                print("  no occupation has both changes and densities of the "
                      "same length; this test is left out")
                continue
            tab, pool = study2_core.jd_language_test(
                delta_sub, dens_sub, n_perm=N_PERM, seed=SEED)
            key = f"{_tag}_{delta_key}_arm={arm}_jd={jd_dim}"
            jd_test_results[key] = {"tab": tab if isinstance(tab, pd.DataFrame)
                                             else pd.DataFrame(tab),
                                     "pool": pool}
            print(f"  {'POOLED':<22}  {pool.get('rho',float('nan')):>+6.3f}  "
                  f"{pool.get('p_rho',float('nan')):>7.4f}  "
                  f"{pool.get('diff_hi_lo',float('nan')):>+10.3f}  "
                  f"{pool.get('p_diff',float('nan')):>7.4f}  "
                  f"{pool.get('n_jds',0):>6}")

def _delta_rank_j(occ, arm, tag, idx):
    if tag == "pool":
        so = study2_core.jd_similarity(emb_orig_pool[occ]["mean"],   emb_jd[occ], how=JD_REPR)
        sm = study2_core.jd_similarity(emb_manip_pool[occ][arm]["mean"], emb_jd[occ], how=JD_REPR)
        ppo = study2_core.pool_pct_per_jd(so)
        ppm = study2_core.indiv_cf_pct_per_jd(so, sm)
    else:
        if emb_focal_orig[occ][arm] is None: return None
        sp  = study2_core.jd_similarity(emb_orig_pool[occ]["mean"],   emb_jd[occ], how=JD_REPR)
        sfo = study2_core.jd_similarity(emb_focal_orig[occ][arm]["mean"],  emb_jd[occ], how=JD_REPR)
        sfm = study2_core.jd_similarity(emb_focal_manip[occ][arm]["mean"], emb_jd[occ], how=JD_REPR)
        ppo = study2_core.focal_pct_per_jd(sp, sfo)
        ppm = study2_core.focal_pct_per_jd(sp, sfm)
    return (ppm - ppo)[idx].mean(axis=0)

def _delta_z_j(occ, arm, tag, idx):
    if tag == "pool":
        so = study2_core.jd_similarity(emb_orig_pool[occ]["mean"],   emb_jd[occ], how=JD_REPR)
        sm = study2_core.jd_similarity(emb_manip_pool[occ][arm]["mean"], emb_jd[occ], how=JD_REPR)
        dz_mat = study2_core.delta_z_per_jd(so, so, sm)
    else:
        if emb_focal_orig[occ][arm] is None: return None
        sp  = study2_core.jd_similarity(emb_orig_pool[occ]["mean"],   emb_jd[occ], how=JD_REPR)
        sfo = study2_core.jd_similarity(emb_focal_orig[occ][arm]["mean"],  emb_jd[occ], how=JD_REPR)
        sfm = study2_core.jd_similarity(emb_focal_manip[occ][arm]["mean"], emb_jd[occ], how=JD_REPR)
        dz_mat = study2_core.delta_z_per_jd(sp, sfo, sfm)
    return dz_mat[idx].mean(axis=0)

for samp_label, df_s, tag in [("POOL", df_pool, "pool"), ("FOCAL", df_focal, "focal")]:
    _run_jd_lang(samp_label, df_s, tag, "delta_rank", _delta_rank_j)
    _run_jd_lang(samp_label, df_s, tag, "delta_z",    _delta_z_j)

print(f"\nJD language test elapsed: {time.time()-_t_jdtest:.0f}s")


=== POOL  arm=communion  JD-dim=communion (MATCHED)  [delta_rank] ===
  Occ                        rho    p_rho  diff_hi_lo   p_diff   n_jds
  -----------------------------------------------------------------
  accountant              -0.062   0.3225      -0.024   0.5997     250
  architect               +0.006   0.9250      -0.005   0.8978     250
  attorney                -0.098   0.1218      -0.090   0.1074     250
  dentist                 -0.309   0.0002      -0.260   0.0002     250
  dietitian               -0.233   0.0452      -0.231   0.0226      77
  nurse                   -0.159   0.0108      -0.426   0.0018     250
  paralegal               -0.160   0.0128      -0.128   0.0108     250
  physician               -0.089   0.1540      -0.133   0.2218     250
  professor               -0.218   0.0100      -0.383   0.0006     148
  psychologist            -0.161   0.0088      -0.285   0.0116     250
  software_engineer       -0.272   0.0002      -0.271   0.0004     250
  surgeon

### 14.5 Data per job description

One row per job description, with its densities and the change it produced, so the tests above can be retraced.

In [37]:
# Save per-JD data including dj vectors
print("Building study2_jd_level.csv ...")
_jdlev_rows = []
for occ in KEPT_13:
    _so = study2_core.jd_similarity(
        emb_orig_pool[occ]["mean"], emb_jd[occ], how=JD_REPR)   # (n_bio, n_jd)
    _nc = emb_jd[occ]["n_chunks"]
    _nt_arr = emb_jd[occ]["n_tokens"]
    for j in range(len(_nc)):
        _start = int(_nc[:j].sum())
        _ntot  = int(_nt_arr[_start: _start + _nc[j]].sum())
        _row = {
            "occ": occ, "jd_idx": j,
            "comm_dens": float(jd_comm_dens[occ][j]),
            "ag_dens":   float(jd_ag_dens[occ][j]),
            "n_words":   int(jd_n_words[occ][j]),
            "n_tokens":  _ntot,
            "n_chunks":  int(_nc[j]),
        }
        for sample, tag, get_dj_fn in [
            ("pool",  "pool",  _delta_rank_j),
            ("focal", "focal", _delta_rank_j),
        ]:
            for arm in ("communion", "agency"):
                dg = _gated(df_pool if sample == "pool" else df_focal, PRIMARY_GATE)
                dg = dg[(dg["arm"] == arm) & (dg["n_subs"] >= 1) & (dg["occ"] == occ)]
                if dg.empty:
                    raise AssertionError(
                        f"No changed biography of {occ} in {sample}/{arm} "
                        f"passes the gate of {PRIMARY_GATE}, so the table "
                        f"would hold a column without values.")
                idx = dg["bio_idx"].values.astype(int) if sample == "pool" \
                      else dg["focal_idx"].values.astype(int)
                _dj_r = get_dj_fn(occ, arm, tag, idx)
                _dj_z = _delta_z_j(occ, arm, tag, idx)
                _row[f"dj_{sample}_{arm}_rank"] = float(_dj_r[j]) if _dj_r is not None else float("nan")
                _row[f"dj_{sample}_{arm}_z"]    = float(_dj_z[j]) if _dj_z is not None else float("nan")
        _jdlev_rows.append(_row)
df_jd_level = pd.DataFrame(_jdlev_rows)
# Carry the variant densities over from so and can use them without
# recomputing: the densities without the D1 function words, that part on its own,
# and the densities under A u B.
_variant_columns = [c for c in ("comm_dens_cw", "ag_dens_cw", "balance_cw",
                                "comm_dens_fw", "ag_dens_fw", "balance_fw",
                                "comm_dens_AB", "ag_dens_AB", "balance_AB",
                                "balance")
                    if c in df_jd_eda.columns]
if _variant_columns:
    df_jd_level = df_jd_level.merge(
        df_jd_eda[["occ", "jd_idx"] + _variant_columns],
        on=["occ", "jd_idx"], how="left")
    print(f"  variant density columns carried over: {_variant_columns}")
print(f"  {len(df_jd_level)} rows  ({df_jd_level['occ'].nunique()} occ, "
      f"{df_jd_level.groupby('occ').size().mean():.0f} avg JDs/occ)")
print(df_jd_level.head(3).to_string(index=False))

Building study2_jd_level.csv ...
  variant density columns carried over: ['comm_dens_cw', 'ag_dens_cw', 'balance_cw', 'comm_dens_fw', 'ag_dens_fw', 'balance_fw', 'comm_dens_AB', 'ag_dens_AB', 'balance_AB', 'balance']
  2975 rows  (13 occ, 229 avg JDs/occ)
  occ  jd_idx  comm_dens  ag_dens  n_words  n_tokens  n_chunks  dj_pool_communion_rank  dj_pool_communion_z  dj_pool_agency_rank  dj_pool_agency_z  dj_focal_communion_rank  dj_focal_communion_z  dj_focal_agency_rank  dj_focal_agency_z  comm_dens_cw  ag_dens_cw  balance_cw  comm_dens_fw  ag_dens_fw  balance_fw  comm_dens_AB  ag_dens_AB  balance_AB   balance
nurse       0   8.235294 8.039216      510       647         2                1.144809             0.038735            -2.076391         -0.057697                   2.1728              0.089764              1.068826           0.041170      6.274510    5.490196    0.784314      1.960784    2.549020   -0.588235      8.627451    9.803922   -1.176471  0.196078
nurse       1   3.426791 4

### 14.6 Joint model of the two densities

- **Question:** Does the relation with the communal density hold when the
  agentic density and the length are in the model as well?
- **Test:** job description; OLS of the change on the standardised densities and
  the log token count with occupation effects, HC1.
- **Status:** exploratory, fixed 2026-10-05.
- **Output:** `study2_jd_joint_model.csv`

In [38]:
# Joint model of the two densities and the length, per outcome.
import statsmodels.formula.api as smf_f7

print(f"control job description table rows: {len(df_jd_level)} "
      f"(expected more than 0)")
if df_jd_level.empty:
    raise AssertionError(
        "The job description table holds no rows, so the joint model has no "
        "data; the job description section has to run before this one.")
_jdj = df_jd_level.copy()
# Z-standardise within occupation
for _v in ("comm_dens", "ag_dens", "comm_dens_cw", "ag_dens_cw"):
    if _v not in _jdj.columns:
        continue
    _jdj[f"{_v}_z"] = _jdj.groupby("occ")[_v].transform(
        lambda x: (x - x.mean()) / x.std(ddof=1) if x.std(ddof=1) > 0 else 0.0)
_jdj["n_tokens_safe"] = _jdj["n_tokens"].clip(lower=1)
_jdj["logtok_raw"] = np.log(_jdj["n_tokens_safe"])
_jdj["logtok_z"] = _jdj.groupby("occ")["logtok_raw"].transform(
    lambda x: (x - x.mean()) / x.std(ddof=1) if x.std(ddof=1) > 0 else 0.0)

_outcomes = [c for c in _jdj.columns
             if c.startswith("dj_") or c == "gap_f_minus_m"]
# The main rows use the full densities; the rows marked robustness use the
# densities without the function words of D1.
_density_sets = [("primary", "comm_dens_z", "ag_dens_z")]
if "comm_dens_cw_z" in _jdj.columns and "ag_dens_cw_z" in _jdj.columns:
    _density_sets.append(("robustness", "comm_dens_cw_z", "ag_dens_cw_z"))

_jm_rows = []
print(f"occupations: {_jdj['occ'].nunique()}")
print(f"  {'outcome':<34} {'status':<11} {'n':>4} {'comm_z':>8} {'p':>7} "
      f"{'ag_z':>8} {'p':>7} {'logtok_z':>9} {'p':>7} {'R2':>6}")
for _status, _comm, _ag in _density_sets:
    for _y in _outcomes:
        _sub = _jdj[["occ", _y, _comm, _ag, "logtok_z"]].dropna()
        # The model holds the occupation fixed, so it needs several occupations
        # and enough job descriptions with all four values.
        if len(_sub) < 10 or _sub["occ"].nunique() < 2:
            print(f"  {_y:<34} {_status:<11} left out: {len(_sub)} rows, "
                  f"{_sub['occ'].nunique()} occupations")
            continue
        _res = smf_f7.ols(f"{_y} ~ {_comm} + {_ag} + logtok_z + C(occ)",
                          data=_sub).fit(cov_type="HC1")
        _bc = float(_res.params.get(_comm, float("nan")))
        _pbc = float(_res.pvalues.get(_comm, float("nan")))
        _ba = float(_res.params.get(_ag, float("nan")))
        _pba = float(_res.pvalues.get(_ag, float("nan")))
        _bl = float(_res.params.get("logtok_z", float("nan")))
        _pbl = float(_res.pvalues.get("logtok_z", float("nan")))
        print(f"  {_y:<34} {_status:<11} {int(_res.nobs):>4} {_bc:>+8.3f} "
              f"{_pbc:>7.4f} {_ba:>+8.3f} {_pba:>7.4f} {_bl:>+9.3f} "
              f"{_pbl:>7.4f} {float(_res.rsquared):>6.3f}")
        _jm_rows.append({"outcome": _y, "status": _status,
                         "x_communion": _comm, "x_agency": _ag,
                         "n": int(_res.nobs),
                         "b_comm_z": _bc, "p_comm_z": _pbc,
                         "b_ag_z": _ba, "p_ag_z": _pba,
                         "b_logtok_z": _bl, "p_logtok_z": _pbl,
                         "r2": float(_res.rsquared)})
df_jd_joint_model = pd.DataFrame(_jm_rows)
print(f"  {len(df_jd_joint_model)} models fitted")

control job description table rows: 2975 (expected more than 0)
occupations: 13
  outcome                            status         n   comm_z       p     ag_z       p  logtok_z       p     R2
  dj_pool_communion_rank             primary     2975   -0.065  0.0000   -0.004  0.7940    +0.096  0.0000  0.740
  dj_pool_communion_z                primary     2975   -0.002  0.0000   -0.000  0.6150    +0.004  0.0000  0.730
  dj_pool_agency_rank                primary     2975   -0.116  0.0000   -0.019  0.2800    +0.057  0.0004  0.801
  dj_pool_agency_z                   primary     2975   -0.004  0.0000   -0.001  0.1597    +0.002  0.0036  0.769
  dj_focal_communion_rank            primary     2975   -0.152  0.0000   -0.033  0.0785    +0.306  0.0000  0.607
  dj_focal_communion_z               primary     2975   -0.005  0.0000   -0.001  0.0636    +0.011  0.0000  0.594
  dj_focal_agency_rank               primary     2975   -0.117  0.0000   -0.084  0.0000    +0.096  0.0000  0.660
  dj_focal_agenc

### 14.7 Comparison with Study 1

- **Question:** Do the occupations where Study 1 finds the larger gap in
  wording also show the larger change in ranking?
- **Test:** occupation; Pearson and Spearman with a leave-one-out range.
- **Status:** robustness, fixed 2026-10-05. Dictionary A is the main analysis,
  A $\cup$ B the variant.
- **Output:** `study2_loo_crossval.csv`

In [39]:
# Cross-validation against Study 1. The betas per occupation come from the
# saved Study 1 results, so this notebook uses exactly the numbers Study 1
# reports: dictionary A is the main analysis, A u B the robustness variant.
_s1_path = Path("outputs/study1_results/study1_lexicon_comparison_by_occupation.csv")
if not _s1_path.exists():
    raise FileNotFoundError(
        f"{_s1_path} not found. This section compares against Study 1; run "
        f"notebook 04 first.")
_s1 = pd.read_csv(_s1_path)
_s1 = _s1[_s1["quantity"].isin(["communion_rate", "agency_rate"])]
print(f"Study 1 betas read: {len(_s1)} rows, {_s1['occupation'].nunique()} occupations")

# Beta_s1[version][occ][dimension]
beta_s1_by_version = {"A": {}, "AB": {}}
for _, _row in _s1.iterrows():
    _dim = "communion" if _row["quantity"] == "communion_rate" else "agency"
    for _version, _col in (("A", "beta_A"), ("AB", "beta_AB")):
        beta_s1_by_version[_version].setdefault(_row["occupation"], {})[_dim] = \
            float(_row[_col])
beta_s1 = beta_s1_by_version[LEX_MAIN]   # the main analysis keeps the old name
_missing = [o for o in KEPT_13 if o not in beta_s1]
print(f"occupations of Study 2 without a Study 1 beta (expected none): {_missing}")

print()
print(f"{'Dim':<12}  {'Sample':<7}  {'Var':<12}  {'lex':<4}  {'n':>3}  "
      f"{'Pearson_r':>10}  {'p':>7}  {'Spearman_r':>11}  {'p':>7}  status")
print("-" * 104)
loo_rows = []
for arm in ("communion", "agency"):
    for label, df_src in [("POOL", df_pool), ("FOCAL", df_focal)]:
        dg = _gated(df_src, PRIMARY_GATE)
        if dg.empty:
            raise AssertionError(
                f"No biography of {label} passes the gate of {PRIMARY_GATE}, "
                f"so this table cannot be formed.")
        for variant, variant_df in [("all", dg), ("changed", dg[dg["n_subs"] >= 1])]:
            mean_d = variant_df[variant_df["arm"] == arm].groupby("occ")["delta"].mean()
            for _version in (LEX_MAIN, LEX_ROBUST):
                _betas = beta_s1_by_version[_version]
                x, y, labs = [], [], []
                for occ in KEPT_13:
                    if (occ in _betas and arm in _betas[occ]
                            and occ in mean_d.index):
                        x.append(_betas[occ][arm]); y.append(mean_d[occ]); labs.append(occ)
                if len(x) < 3:
                    continue
                rp, pp = pearsonr(x, y)
                rs, ps = spearmanr(x, y)
                _status = "primary" if _version == LEX_MAIN else "robustness"
                print(f"  {arm:<12}  {label:<7}  {variant:<12}  {_version:<4}  "
                      f"{len(x):>3}  {rp:>+10.3f}  {pp:>7.4f}  {rs:>+11.3f}  "
                      f"{ps:>7.4f}  {_status}")
                loo = study2_core.loo_correlation(x, y, labs)
                _pr_min = float(loo["pearson_r"].min())
                _pr_max = float(loo["pearson_r"].max())
                _pp_min = float(loo["pearson_p"].min()) \
                          if "pearson_p" in loo.columns else float("nan")
                _pp_max = float(loo["pearson_p"].max()) \
                          if "pearson_p" in loo.columns else float("nan")
                print(f"    LOO range: r in [{_pr_min:+.3f}, {_pr_max:+.3f}]  "
                      f"p in [{_pp_min:.4f}, {_pp_max:.4f}]")
                loo = loo.copy()
                loo.insert(0, "status", _status)
                loo.insert(0, "lexicon", _version)
                loo.insert(0, "variant", variant)
                loo.insert(0, "arm", arm)
                loo.insert(0, "sample", label.lower())
                loo_rows.append(loo)
df_loo_all = pd.concat(loo_rows, ignore_index=True) if loo_rows else pd.DataFrame()

Study 1 betas read: 54 rows, 27 occupations
occupations of Study 2 without a Study 1 beta (expected none): []

Dim           Sample   Var           lex     n   Pearson_r        p   Spearman_r        p  status
--------------------------------------------------------------------------------------------------------
  communion     POOL     all           A      13      -0.434   0.1386       -0.324   0.2799  primary
    LOO range: r in [-0.564, -0.241]  p in [0.0563, 0.4512]
  communion     POOL     all           AB     13      -0.421   0.1520       -0.341   0.2547  robustness
    LOO range: r in [-0.535, -0.206]  p in [0.0733, 0.5214]
  communion     POOL     changed       A      13      -0.435   0.1376       -0.335   0.2629  primary
    LOO range: r in [-0.576, -0.240]  p in [0.0500, 0.4525]
  communion     POOL     changed       AB     13      -0.426   0.1468       -0.352   0.2387  robustness
    LOO range: r in [-0.550, -0.210]  p in [0.0638, 0.5117]
  communion     FOCAL    all        

### 14.8 Share of women and mean change

- **Question:** Is the change larger in occupations with more women?
- **Test:** occupation; Spearman.
- **Status:** exploratory, fixed 2026-10-05.
- **Output:** `study2_pct_female.csv`

In [40]:
# Share of women against the mean change, per occupation.
print(f"{'Arm':<12}  {'Sample':<7}  {'Pearson_r':>10}  {'p':>7}  {'Spearman_r':>11}  {'p':>7}")
print("-" * 60)
for arm in ("communion", "agency"):
    for label, df_src in [("POOL", df_pool), ("FOCAL", df_focal)]:
        dg = _gated(df_src, PRIMARY_GATE)
        if dg.empty:
            raise AssertionError(
                f"No biography of {label} passes the gate of {PRIMARY_GATE}, "
                f"so this table cannot be formed.")
        mean_d = dg[dg["arm"] == arm].groupby("occ")["delta"].mean()
        occs_avail = [o for o in KEPT_13 if o in mean_d.index and o in PCT_FEMALE]
        if len(occs_avail) < 3:
            continue
        pf  = np.array([PCT_FEMALE[o] for o in occs_avail])
        md_ = np.array([mean_d[o]     for o in occs_avail])
        rp, pp = pearsonr(pf, md_)
        rs, ps = spearmanr(pf, md_)
        print(f"  {arm:<12}  {label:<7}  {rp:>+10.3f}  {pp:>7.4f}  {rs:>+11.3f}  {ps:>7.4f}")

Arm           Sample    Pearson_r        p   Spearman_r        p
------------------------------------------------------------
  communion     POOL         -0.069   0.8236       -0.049   0.8725
  communion     FOCAL        -0.293   0.3313       -0.159   0.6031
  agency        POOL         -0.047   0.8780       +0.154   0.6158
  agency        FOCAL        +0.333   0.2657       +0.302   0.3156


### 14.9 Sensitivity to the gate and the representation

- **Question:** Does the result hold at other gates and with the truncated
  representation?
- **Test:** biography; the same medians and tests under each setting.
- **Status:** robustness, fixed 2026-10-05.
- **Output:** `study2_sensitivity.csv`,
  `study2_sensitivity_gate_by_sample.csv`

In [41]:
# The same result at other gates and with the truncated representation.
_t_sens = time.time()
# Gate x JD_REPR table (pool only, existing)
_sim_cache = {}
for occ in KEPT_13:
    _sim_cache[occ] = {}
    for arm in ("communion", "agency"):
        _sim_cache[occ][arm] = {}
        for jd_repr in ("mean", "max"):
            s_o = study2_core.jd_similarity(emb_orig_pool[occ]["mean"], emb_jd[occ], how=jd_repr)
            s_m = study2_core.jd_similarity(emb_manip_pool[occ][arm]["mean"], emb_jd[occ], how=jd_repr)
            p_o = study2_core.percentile_ranks(s_o)
            p_i = study2_core.individual_cf_ranks(s_o, s_m)
            _sim_cache[occ][arm][jd_repr] = p_i - p_o
        _sim_cache[occ][arm]["trunc"] = delta_trunc[occ][arm]

def _sensitivity_row(gate, jd_repr):
    n_excl = 0;  all_d = {"communion": [], "agency": []}
    for occ in KEPT_13:
        for arm in ("communion", "agency"):
            deltas_all = _sim_cache[occ][arm][jd_repr]
            mask = cp_sims_pool[occ][arm] >= gate if gate is not None else np.ones(len(deltas_all), bool)
            n_excl += (~mask).sum()
            changed = mask & np.array([r.n_replaced + r.n_deleted > 0
                                       for r in manip_pool[occ][arm]])
            all_d[arm].extend(deltas_all[changed].tolist())
    result = {"gate": str(gate), "jd_repr": jd_repr, "n_excl": int(n_excl)}
    for arm in ("communion", "agency"):
        d = np.array(all_d[arm])
        # A gate can leave no changed biography at all. The row then holds a
        # count of zero and no test, and the reason is printed.
        if len(d) == 0:
            print(f"  gate {gate}, {jd_repr}, {arm}: no changed biography, "
                  f"no test")
            result[f"n_changed_{arm}"] = 0
            continue
        # The signed rank test needs changes that are not all zero.
        if not (d != 0).any():
            raise AssertionError(
                f"All {len(d)} changes under gate {gate} for {arm} are exactly "
                f"zero, so no biography moved and the signed rank test is not "
                f"defined.")
        _, p = wilcoxon(d, alternative="two-sided")
        result[f"n_changed_{arm}"] = len(d)
        result[f"med_{arm}"]  = float(np.median(d))
        result[f"mean_{arm}"] = float(d.mean())
        result[f"pct_pos_{arm}"] = float((d > 0).mean())
        result[f"pct_neg_{arm}"] = float((d < 0).mean())
        result[f"p_{arm}"]    = float(p)
    return result

sens_rows = []
for gate in GATES:
    for jd_repr in ("mean", "max", "trunc"):
        sens_rows.append(_sensitivity_row(gate, jd_repr))
df_sensitivity = pd.DataFrame(sens_rows)

print(f"{'Gate':<6}  {'JD':>5}  {'n_excl':>6}  "
      f"{'n_ch_c':>7}  {'med_c':>7}  {'p_c':>7}  "
      f"{'n_ch_a':>7}  {'med_a':>7}  {'p_a':>7}")
print("-" * 72)
for _, row in df_sensitivity.iterrows():
    print(f"  {row['gate']:<6}  {row['jd_repr']:>5}  {row['n_excl']:>6}  "
          f"{row.get('n_changed_communion',0):>7}  "
          f"{row.get('med_communion',np.nan):>+7.3f}  {row.get('p_communion',np.nan):>7.4f}  "
          f"{row.get('n_changed_agency',0):>7}  "
          f"{row.get('med_agency',np.nan):>+7.3f}  {row.get('p_agency',np.nan):>7.4f}")

# Gate x Sample table (POOL + FOCAL)
gate_sample_rows = []
for sample_label, df_src in [("pool", df_pool), ("focal", df_focal)]:
    n_total_all = {arm: len(_gated(df_src[df_src["arm"]==arm], None)) for arm in ("communion","agency")}
    for gate in GATES:
        dg = _gated(df_src, gate)
        # A strict gate can leave no biography in a sample; that row is then
        # left out and the reason is printed.
        if dg.empty:
            print(f"  {sample_label}, gate {gate}: no biography passes, "
                  f"left out")
            continue
        t = study2_core.treatment_table(dg, ["arm"])
        for _, row in t.iterrows():
            arm = row.get("arm","")
            n_tot = n_total_all.get(arm, np.nan)
            n_excl_g = n_tot - row.get("n", 0) if not np.isnan(n_tot) else np.nan
            gate_sample_rows.append({
                "sample": sample_label, "gate": str(gate), "arm": arm,
                "n": row.get("n", np.nan),
                "n_total": n_tot, "n_excluded": n_excl_g,
                "n_treated": row.get("n_treated", np.nan),
                "mean_treated": row.get("mean_treated", np.nan),
                "median_treated": row.get("median_treated", np.nan),
                "share_delta_neg": row.get("share_delta_neg", np.nan),
                "share_delta_pos": row.get("share_delta_pos", np.nan),
                "p_wilcoxon_treated": row.get("p_wilcoxon_treated", np.nan),
            })
df_sensitivity_gate_sample = pd.DataFrame(gate_sample_rows)
print()
print("Gate x Sample (POOL + FOCAL):")
print(f"{'Sample':<7}  {'Gate':<6}  {'Arm':>9}  {'n':>5}  {'n_excl':>6}  "
      f"{'n_treat':>7}  {'med_treat':>9}  {'pct_neg':>7}  {'pct_pos':>7}  {'p_treat':>8}")
print("-" * 80)
for _, row in df_sensitivity_gate_sample.iterrows():
    print(f"  {row['sample']:<7}  {row['gate']:<6}  {row['arm']:>9}  "
          f"{row.get('n',np.nan):>5.0f}  {row.get('n_excluded',np.nan):>6.0f}  "
          f"{row.get('n_treated',np.nan):>7.0f}  {row.get('median_treated',np.nan):>+9.3f}  "
          f"{row.get('share_delta_neg',np.nan):>7.1%}  {row.get('share_delta_pos',np.nan):>7.1%}  "
          f"{row.get('p_wilcoxon_treated',np.nan):>8.4f}")

print(f"\nSensitivity elapsed: {time.time()-_t_sens:.0f}s")

Gate       JD  n_excl   n_ch_c    med_c      p_c   n_ch_a    med_a      p_a
------------------------------------------------------------------------
  None     mean       0     1878   +0.347   0.0000     1093   +0.185   0.0000
  None      max       0     1878   +0.338   0.0000     1093   +0.157   0.0000
  None    trunc       0     1878   +0.338   0.0000     1093   +0.173   0.0000
  0.85     mean       1     1878   +0.347   0.0000     1092   +0.186   0.0000
  0.85      max       1     1878   +0.338   0.0000     1092   +0.161   0.0000
  0.85    trunc       1     1878   +0.338   0.0000     1092   +0.174   0.0000
  0.9      mean      17     1870   +0.345   0.0000     1084   +0.186   0.0000
  0.9       max      17     1870   +0.335   0.0000     1084   +0.161   0.0000
  0.9     trunc      17     1870   +0.337   0.0000     1084   +0.174   0.0000
  0.95     mean      84     1829   +0.363   0.0000     1058   +0.186   0.0000
  0.95      max      84     1829   +0.345   0.0000     1058   +0.161   

### 14.10 Rater validation of the list

- **Question:** Do the rewritten passages keep the meaning and stay
  grammatical?
- **Test:** occurrence; two raters, agreement and the rates by the joint
  judgement with a Wilson interval.
- **Status:** control, D7, fixed 2026-10-11.
- **Output:** `validation_agreement_v2.csv`, `validation_rates_v2.csv`,
  `substitution_rules_v2_validation.csv`

In [42]:
# Rater validation of the substitution list: agreement, the rates by the joint
# judgement, and the rules that D7 switched off.
_agree_path = Path(f"outputs/study2_{RUN_TAG}/validation_agreement_v2.csv")
_rates_path = Path(f"outputs/study2_{RUN_TAG}/validation_rates_v2.csv")
_decision_path = Path(VALIDATION_FILE)
for _path in (_agree_path, _rates_path, _decision_path):
    if not _path.exists():
        raise FileNotFoundError(
            f"{_path} not found. Run scripts/apply_validation_v2.py on the rated "
            f"sample before this notebook; the validation of Run 5 has no "
            f"fallback file.")

df_validation_agreement = pd.read_csv(_agree_path)
df_validation_rates = pd.read_csv(_rates_path)
df_validation_decisions = pd.read_csv(_decision_path)

_n_rated = int(df_validation_rates.loc[df_validation_rates["subset"] == "all",
                                       "n_rated"].iloc[0])
print(f"cases with a joint judgement: {_n_rated}")
print(f"cases rated by both raters  : "
      f"{int(df_validation_agreement['n_double_rated'].iloc[0])}")
print()
print("agreement between the two raters")
print(f"  {'criterion':<14} {'raw':>8} {'kappa':>8} {'AC1':>8}")
for _, _row in df_validation_agreement.iterrows():
    print(f"  {_row['criterion']:<14} {_row['percent_agreement']:>8.3f} "
          f"{_row['cohen_kappa']:>8.3f} {_row['gwet_ac1']:>8.3f}")
print("  AC1 is reported next to kappa because almost every case passes, which "
      "depresses kappa.")
print()
print("rates by the joint judgement, with a 95 per cent Wilson interval")
print(f"  {'subset':<10} {'criterion':<18} {'n':>5} {'rate':>8} {'interval':>20}")
for _, _row in df_validation_rates.iterrows():
    print(f"  {_row['subset']:<10} {_row['criterion']:<18} {_row['n_rated']:>5} "
          f"{_row['rate']:>8.3f} "
          f"[{_row['ci_low']:.3f}, {_row['ci_high']:.3f}]")

_failed_rules = sorted(df_validation_decisions.loc[
    ~df_validation_decisions["passed"].astype(bool), "rule_id"])
print()
print(f"rules that fail D7: {len(_failed_rules)} {_failed_rules}")
for _, _row in df_validation_decisions[
        ~df_validation_decisions["passed"].astype(bool)].iterrows():
    print(f"  {_row['rule_id']} {_row['arm']:<10} "
          f"{_row['n_passing_both']} of {_row['n_rated']} passing -> "
          f"{_row['consequence']}")

# The rules switched off in the engine have to be exactly those the validation
# file marks as failed.
_switched_off = sorted(r.rule_id for r in rules_v2 if not r.validated)
print()
print(f"control rules switched off against the validation file: "
      f"{'identical' if _switched_off == _failed_rules else 'DIFFERENT'} "
      f"(expected identical)")
if _switched_off != _failed_rules:
    raise AssertionError(
        f"The engine has {_switched_off} switched off, the validation file lists "
        f"{_failed_rules}.")

cases with a joint judgement: 179
cases rated by both raters  : 88

agreement between the two raters
  criterion           raw    kappa      AC1
  meaning           0.864    0.380    0.825
  grammatical       0.875    0.457    0.838
  AC1 is reported next to kappa because almost every case passes, which depresses kappa.

rates by the joint judgement, with a 95 per cent Wilson interval
  subset     criterion              n     rate             interval
  all        meaning_preserved    179    0.899 [0.847, 0.935]
  all        grammatical          179    0.883 [0.827, 0.922]
  all        both                 179    0.872 [0.815, 0.913]
  communion  meaning_preserved     89    0.955 [0.890, 0.982]
  communion  grammatical           89    0.910 [0.833, 0.954]
  communion  both                  89    0.910 [0.833, 0.954]
  agency     meaning_preserved     90    0.844 [0.756, 0.905]
  agency     grammatical           90    0.856 [0.768, 0.914]
  agency     both                  90    0.833 [

### 14.11 Word-level overlap

- **Question:** Does the effect of a substitution depend on whether the
  replaced word or the new word carries more weight?
- **Test:** biography; the word sets come from the protocol, so a deleted
  occurrence has no target word.
- **Status:** exploratory, fixed 2026-10-05.
- **Output:** `study2_word_overlap.csv`, `study2_word_overlap_bio.csv`

In [43]:
# Word-level overlap. The sets come from the logged substitutions of the
# manipulation result, not from a set difference of the texts, so a source word
# that stays elsewhere and a target word that was already there are both handled.
# A deleted occurrence has no target word: only its source word counts.
print(f"control occurrence protocol rows: {len(df_occurrences)} "
      f"(expected more than 0)")
if df_occurrences.empty:
    raise AssertionError(
        "The occurrence protocol is empty, so the word sets cannot be formed; "
        "the manipulation section has to run first.")
_treated = df_occurrences[
    (df_occurrences["sample"] == "pool")
    & (df_occurrences["action_taken"].isin(["replaced", "deleted"]))].copy()
_treated["source_lower"] = _treated["token"].str.lower()
_treated["target_lower"] = _treated["target"].fillna("").str.lower()

_bio_rows = []
for (occ, arm, bio_idx), group in _treated.groupby(["occ", "arm", "bio_idx"]):
    _sources = set(group["source_lower"])
    # Deletions contribute no target word.
    _targets = {t for t, action in zip(group["target_lower"], group["action_taken"])
                if action == "replaced" and t}
    _bio_rows.append({
        "occ": occ, "arm": arm, "bio_idx": bio_idx,
        "n_source_words": len(_sources), "n_target_words": len(_targets),
        "n_replaced": int((group["action_taken"] == "replaced").sum()),
        "n_deleted": int((group["action_taken"] == "deleted").sum()),
        "overlap": len(_sources & _targets),
        "source_words": "|".join(sorted(_sources)),
        "target_words": "|".join(sorted(_targets)),
    })
df_word_overlap_bio = pd.DataFrame(_bio_rows)
df_word_overlap_bio = df_word_overlap_bio.merge(
    df_pool[["occ", "bio_idx", "arm", "delta", "sim_cp", "n_subs"]],
    on=["occ", "bio_idx", "arm"], how="left")

print("Word-level overlap over the treated occurrences of the pool")
print(f"{'arm':<10} {'bios':>6} {'src/bio':>8} {'tgt/bio':>8} "
      f"{'deleted/bio':>12} {'overlap':>8}")
print("-" * 56)
_rows = []
for arm in ("communion", "agency"):
    _s = df_word_overlap_bio[df_word_overlap_bio["arm"] == arm]
    if _s.empty:
        raise AssertionError(
            f"No biography was treated for {arm}, so no word sets exist.")
    print(f"{arm:<10} {len(_s):>6} {_s['n_source_words'].mean():>8.2f} "
          f"{_s['n_target_words'].mean():>8.2f} {_s['n_deleted'].mean():>12.2f} "
          f"{_s['overlap'].mean():>8.3f}")
    _rows.append({"arm": arm, "n_bios": len(_s),
                  "mean_source_words": float(_s["n_source_words"].mean()),
                  "mean_target_words": float(_s["n_target_words"].mean()),
                  "mean_deleted": float(_s["n_deleted"].mean()),
                  "mean_overlap": float(_s["overlap"].mean()),
                  "n_bios_deleted_only": int((_s["n_replaced"] == 0).sum())})

# The same table per occupation.
print()
print("Per occupation:")
print(f"{'occ':<20} {'arm':<10} {'bios':>6} {'src/bio':>8} {'tgt/bio':>8}")
print("-" * 56)
# An occupation and arm without a single treated biography carries no row. That
# is possible when no source word of the list occurs there, so the pair is left
# out and the pairs left out are named below the table.
_no_rows = []
for occ in KEPT_13:
    for arm in ("communion", "agency"):
        _s = df_word_overlap_bio[(df_word_overlap_bio["occ"] == occ)
                                 & (df_word_overlap_bio["arm"] == arm)]
        if _s.empty:
            _no_rows.append(f"{occ}/{arm}")
            continue
        print(f"{occ:<20} {arm:<10} {len(_s):>6} "
              f"{_s['n_source_words'].mean():>8.2f} "
              f"{_s['n_target_words'].mean():>8.2f}")
        _rows.append({"occ": occ, "arm": arm, "n_bios": len(_s),
                      "mean_source_words": float(_s["n_source_words"].mean()),
                      "mean_target_words": float(_s["n_target_words"].mean()),
                      "mean_deleted": float(_s["n_deleted"].mean()),
                      "mean_overlap": float(_s["overlap"].mean()),
                      "n_bios_deleted_only": int((_s["n_replaced"] == 0).sum())})
if _no_rows:
    print(f"occupation and arm without a treated biography, left out: "
          f"{_no_rows}")
df_word_overlap = pd.DataFrame(_rows)

# The sets come from the protocol. Taking the set difference of the texts
# instead loses a source word that stays elsewhere in the text and a target
# word that was already there, so the two differ for some biographies.
_differs = 0
_compared = 0
for occ in KEPT_13:
    for arm in ("communion", "agency"):
        _logged = df_word_overlap_bio[
            (df_word_overlap_bio["occ"] == occ)
            & (df_word_overlap_bio["arm"] == arm)]
        for _, _row in _logged.iterrows():
            _idx = int(_row["bio_idx"])
            _result = manip_pool[occ][arm][_idx]
            _before = set(str(_result.original).lower().split())
            _after = set(str(_result.manipulated).lower().split())
            _source_diff = {w.strip(".,;:()") for w in _before - _after}
            _target_diff = {w.strip(".,;:()") for w in _after - _before}
            _source_log = set(str(_row["source_words"]).split("|")) - {""}
            _target_log = set(str(_row["target_words"]).split("|")) - {""}
            _compared += 1
            if _source_log != _source_diff or _target_log != _target_diff:
                _differs += 1
print()
print(f"biographies whose word sets differ from the set difference of the "
      f"texts: {_differs} of {_compared}")
print("  the protocol is the source of the sets, so these are the cases the "
      "set difference would have got wrong")

control occurrence protocol rows: 48411 (expected more than 0)
Word-level overlap over the treated occurrences of the pool
arm          bios  src/bio  tgt/bio  deleted/bio  overlap
--------------------------------------------------------
communion    1878     1.64     1.63         0.02    0.000
agency       1093     1.31     1.27         0.04    0.000

Per occupation:
occ                  arm          bios  src/bio  tgt/bio
--------------------------------------------------------
nurse                communion     174     1.91     1.91
nurse                agency         47     1.32     1.34
software_engineer    communion      92     1.37     1.36
software_engineer    agency         49     1.14     1.08
accountant           communion     114     1.42     1.45
accountant           agency         86     1.43     1.38
architect            communion     122     1.34     1.35
architect            agency         80     1.36     1.32
psychologist         communion     164     1.89     1.87
ps

### 14.12 By word class

- **Question:** Do stylistic, neutral-factual and content words differ in
  how far they move the ranking?
- **Test:** biography; median change per class, and split by what happened to
  the occurrence.
- **Status:** exploratory, fixed 2026-10-05.
- **Output:** `study2_by_word_class.csv`

In [44]:
# By word class. The classes come from the column word_class of the v2 rules
# (content, neutral_factual, style), carried through the occurrence protocol,
# and the results are also split by what happened to the occurrence.
print(f"control occurrence protocol rows: {len(df_occurrences)} "
      f"(expected more than 0)")
if df_occurrences.empty:
    raise AssertionError(
        "The occurrence protocol is empty, so the word classes cannot be "
        "assigned; the manipulation section has to run first.")
_treated = df_occurrences[
    (df_occurrences["sample"] == "pool")
    & (df_occurrences["action_taken"].isin(["replaced", "deleted"]))]
_classes = sorted(c for c in _treated["word_class"].dropna().unique() if c)
print(f"word classes in the v2 rules: {_classes}")

# Counts per bio and class, and per bio and action.
_by_class = (_treated.groupby(["occ", "bio_idx", "arm", "word_class"])
             .size().unstack(fill_value=0).reset_index())
_by_action = (_treated.groupby(["occ", "bio_idx", "arm", "action_taken"])
              .size().unstack(fill_value=0).reset_index())
_df_wc = df_pool.merge(_by_class, on=["occ", "bio_idx", "arm"], how="left")
_df_wc = _df_wc.merge(_by_action, on=["occ", "bio_idx", "arm"], how="left")
for _column in _classes + ["replaced", "deleted"]:
    if _column not in _df_wc.columns:
        _df_wc[_column] = 0
    _df_wc[_column] = _df_wc[_column].fillna(0).astype(int)
_df_wc = _df_wc[_df_wc["n_subs"] >= 1]

# How the occurrence was treated: replaced only, deleted only, or both.
_df_wc["action_kind"] = np.where(
    _df_wc["deleted"] == 0, "replaced only",
    np.where(_df_wc["replaced"] == 0, "deleted only", "both"))

BINS    = [(-999, 0.90), (0.90, 0.95), (0.95, 0.98), (0.98, 0.99), (0.99, 999)]
BIN_LBL = ["<0.90", "0.90-0.95", "0.95-0.98", "0.98-0.99", ">=0.99"]
_wc_rows = []

print()
print("By word class, over the pool")
print("1. Delta by sim_cp class, over all changed bios (no gate):")
print(f"  {'arm':<10} {'sim_cp':<12} {'n':>6} {'median':>9} {'share>0':>9}")
# A similarity bin that no biography falls into has no row of its own. The bins
# are fixed in advance, so the empty ones are named after the table.
_empty_bins = []
for arm in ("communion", "agency"):
    _sub = _df_wc[_df_wc["arm"] == arm]
    for _label, (lo, hi) in zip(BIN_LBL, BINS):
        _b = _sub[(_sub["sim_cp"] >= lo) & (_sub["sim_cp"] < hi)]
        if _b.empty:
            _empty_bins.append(f"{arm}/{_label}")
            continue
        print(f"  {arm:<10} {_label:<12} {len(_b):>6} "
              f"{_b['delta'].median():>+9.3f} {(_b['delta'] > 0).mean():>9.1%}")
        _wc_rows.append({"part": "sim_cp_class", "arm": arm, "level": _label,
                         "n": len(_b), "median_delta": float(_b["delta"].median()),
                         "share_above_zero": float((_b["delta"] > 0).mean())})

if _empty_bins:
    print(f"  bins that no biography falls into, left out: {_empty_bins}")

print()
print("2. Bios that carry only one word class, without a gate and at two gates:")
# A word class can be absent on its own, that is without another class in the
# same biography, and then has no row.
_empty_classes = []
print(f"  {'arm':<10} {'class':<16} {'gate':<6} {'n':>6} {'median':>9} "
      f"{'share>0':>9}")
for arm in ("communion", "agency"):
    for _gate_label, _gate in (("none", None), ("0.90", 0.90), ("0.95", 0.95)):
        _sub = _df_wc[_df_wc["arm"] == arm]
        if _gate is not None:
            _sub = _sub[_sub["sim_cp"] >= _gate]
        for _cl in _classes:
            _others = [c for c in _classes if c != _cl]
            _only = _sub[(_sub[_cl] >= 1) & (_sub[_others].sum(axis=1) == 0)] \
                    if _others else _sub[_sub[_cl] >= 1]
            if _only.empty:
                _empty_classes.append(f"{arm}/{_cl}/{_gate_label}")
                continue
            print(f"  {arm:<10} {_cl:<16} {_gate_label:<6} {len(_only):>6} "
                  f"{_only['delta'].median():>+9.3f} "
                  f"{(_only['delta'] > 0).mean():>9.1%}")
            _wc_rows.append({"part": "single_class", "arm": arm, "level": _cl,
                             "gate": _gate_label, "n": len(_only),
                             "median_delta": float(_only["delta"].median()),
                             "share_above_zero": float((_only["delta"] > 0).mean())})

if _empty_classes:
    print(f"  arm, class and gate without a biography of that class alone, "
          f"left out: {_empty_classes}")

print()
print("3. Split by what happened to the occurrences:")
# A kind of change can be absent in an arm, for instance when nothing was
# deleted there.
_empty_kinds = []
print(f"  {'arm':<10} {'action':<15} {'n':>6} {'median':>9} {'share>0':>9}")
for arm in ("communion", "agency"):
    _sub = _df_wc[_df_wc["arm"] == arm]
    for _kind in ("replaced only", "deleted only", "both"):
        _k = _sub[_sub["action_kind"] == _kind]
        if _k.empty:
            _empty_kinds.append(f"{arm}/{_kind}")
            continue
        print(f"  {arm:<10} {_kind:<15} {len(_k):>6} "
              f"{_k['delta'].median():>+9.3f} {(_k['delta'] > 0).mean():>9.1%}")
        _wc_rows.append({"part": "action_taken", "arm": arm, "level": _kind,
                         "n": len(_k), "median_delta": float(_k["delta"].median()),
                         "share_above_zero": float((_k["delta"] > 0).mean())})

if _empty_kinds:
    print(f"  arm and kind of change without a biography, left out: "
          f"{_empty_kinds}")

print()
print("4. Word class and action together:")
# A combination of class and kind of change under five biographies is not
# reported; how many combinations that concerns is printed after the table.
_too_small = []
for arm in ("communion", "agency"):
    _sub = _df_wc[_df_wc["arm"] == arm]
    for _cl in _classes:
        for _kind in ("replaced only", "deleted only", "both"):
            _k = _sub[(_sub[_cl] >= 1) & (_sub["action_kind"] == _kind)]
            if len(_k) < 5:
                _too_small.append(f"{arm}/{_cl}/{_kind}")
                continue
            print(f"  {arm:<10} {_cl:<16} {_kind:<15} n={len(_k):>5} "
                  f"median={_k['delta'].median():>+8.3f}")
            _wc_rows.append({"part": "class_and_action", "arm": arm,
                             "level": f"{_cl} / {_kind}", "n": len(_k),
                             "median_delta": float(_k["delta"].median()),
                             "share_above_zero": float((_k["delta"] > 0).mean())})
if _too_small:
    print(f"  combinations of class and kind of change under five "
          f"biographies, left out: {len(_too_small)} of "
          f"{len(_classes) * 6}")
df_word_class = pd.DataFrame(_wc_rows)

control occurrence protocol rows: 48411 (expected more than 0)
word classes in the v2 rules: ['content', 'neutral_factual', 'style']

By word class, over the pool
1. Delta by sim_cp class, over all changed bios (no gate):
  arm        sim_cp            n    median   share>0
  communion  <0.90             8    +7.165     62.5%
  communion  0.90-0.95        41    -0.627     43.9%
  communion  0.95-0.98       168    +0.993     58.9%
  communion  0.98-0.99       245    +0.761     59.2%
  communion  >=0.99         1416    +0.288     58.9%
  agency     <0.90             9    -1.001     44.4%
  agency     0.90-0.95        26    +0.067     50.0%
  agency     0.95-0.98       107    +4.051     67.3%
  agency     0.98-0.99       139    +1.661     65.5%
  agency     >=0.99          812    +0.040     51.5%

2. Bios that carry only one word class, without a gate and at two gates:
  arm        class            gate        n    median   share>0
  communion  content          none      822    +0.085    

### 14.13 Gender gap in the original rankings

- **Question:** Do biographies by women sit lower in the original rankings,
  and does that go with the wording of the advertisement?
- **Expectation:** the gap grows with the communal wording of the advertisement.
- **Test:** job description; Spearman within occupation with a pooled
  permutation test.
- **Status:** pre-specified test T1 (fixed 2026-10-05), with the published main
  dictionary A; A $\cup$ B and the rows without the function words of D1 are
  variants.
- **Output:** `study2_gender_gap.csv`, `study2_gender_gap_ols_post.csv`

In [45]:
# + Gender gap in original rankings
import statsmodels.formula.api as smf12l
from scipy.stats import mannwhitneyu as _mwu12l
from pathlib import Path as _P12l

print("Gender gap in the original rankings of the pool")

_gap_rows, _gap_bio_rows = [], []
_pct_orig_all, _P_mat_all, _gap_j_all = {}, {}, {}

for occ in KEPT_13:
    _so = study2_core.jd_similarity(
        emb_orig_pool[occ]["mean"], emb_jd[occ], how=JD_REPR)
    _ppo = study2_core.percentile_ranks(_so)
    _pct_orig_all[occ] = _ppo
    _pool_df = bio_pools[occ].reset_index(drop=True)
    _f_col = _pool_df["female"] if "female" in _pool_df.columns else \
             (_pool_df.get("gender_label","") == "Female").astype(int)
    for i in range(len(_pool_df)):
        _gap_bio_rows.append({"occ":occ,"bio_idx":i,
                               "female":int(_f_col.iloc[i]),"pct_orig":float(_ppo[i])})

df_gap_bio = pd.DataFrame(_gap_bio_rows)

# Bio-level per occ
print(f"\n1. Bio-level: mean pct_orig by gender")
print(f"  {'Occ':<22}  {'mean_F':>7}  {'mean_M':>7}  {'diff_FM':>8}  {'n_f':>4}  {'n_m':>4}  {'MW_p':>8}")
print("  " + "-" * 72)
for occ in KEPT_13:
    _sub = df_gap_bio[df_gap_bio["occ"]==occ]
    _f = _sub[_sub["female"]==1]["pct_orig"].values
    _m = _sub[_sub["female"]==0]["pct_orig"].values
    if len(_f) < 3 or len(_m) < 3:
        print(f"  {occ:<22}  (insufficient data)"); continue
    _U, _p = _mwu12l(_f, _m, alternative="two-sided")
    _diff = float(_f.mean() - _m.mean())
    print(f"  {occ:<22}  {_f.mean():>7.2f}  {_m.mean():>7.2f}  "
          f"{_diff:>+8.2f}  {len(_f):>4}  {len(_m):>4}  {_p:>8.4f}")
    _gap_rows.append({"occ":occ,"mean_pct_orig_f":float(_f.mean()),
                       "mean_pct_orig_m":float(_m.mean()),"diff_f_minus_m":_diff,
                       "n_f":len(_f),"n_m":len(_m),"mw_p":float(_p)})

print(f"\n  Pooled OLS: pct_orig ~ female + C(occ), HC1")
_n_women = int((df_gap_bio["female"] == 1).sum())
_n_men   = int((df_gap_bio["female"] == 0).sum())
_n_occ   = int(df_gap_bio["occ"].nunique())
print(f"  control biographies: women {_n_women}, men {_n_men}, "
      f"occupations {_n_occ} (at least 2, 2 and 2 expected)")
if _n_women < 2 or _n_men < 2 or _n_occ < 2:
    raise AssertionError(
        f"The pooled regression needs biographies of both genders in at least "
        f"two occupations; women {_n_women}, men {_n_men}, occupations "
        f"{_n_occ}.")
_ols_res = smf12l.ols("pct_orig ~ female + C(occ)", data=df_gap_bio).fit(cov_type="HC1")
_b_f = float(_ols_res.params["female"])
_se_f = float(_ols_res.bse["female"])
_p_f  = float(_ols_res.pvalues["female"])
_n_f  = int(_ols_res.nobs)
print(f"  b_female = {_b_f:>+.3f}  SE={_se_f:.3f}  p={_p_f:.3f}  n={_n_f}")

# 2. JD-level gap
print(f"\n2. JD-level gap analysis")
for occ in KEPT_13:
    _so  = study2_core.jd_similarity(
        emb_orig_pool[occ]["mean"], emb_jd[occ], how=JD_REPR)
    _P   = study2_core.pool_pct_per_jd(_so)
    _f_col = (bio_pools[occ].reset_index(drop=True)
               .get("female", pd.Series(dtype=int)).values)
    _fidx = _f_col == 1; _midx = _f_col == 0
    if _fidx.sum() < 2 or _midx.sum() < 2:
        raise AssertionError(
            f"control biographies per gender in {occ}: women {int(_fidx.sum())}, "
            f"men {int(_midx.sum())} (expected at least 2 of each). The gap "
            f"cannot be formed, and a vector of zeros would enter the test as a "
            f"result.")
    _gap_j_all[occ] = _P[_fidx].mean(axis=0) - _P[_midx].mean(axis=0)
    _P_mat_all[occ] = _P

# The balance of the job descriptions comes from the density section. Without it
# the gap cannot be related to the language of the advertisement, so its absence
# stops the run rather than leaving the test out.
print(f"control density table rows: {len(df_jd_eda)}  "
      f"balance column present: {'balance' in df_jd_eda.columns}")
if df_jd_eda.empty or "balance" not in df_jd_eda.columns:
    raise AssertionError(
        "The density table of the job descriptions has no balance column, so "
        "the gender gap cannot be related to their language; the density "
        "section has to run before this one.")
_balance_j_all, _balance_j_all_cw = {}, {}
_balance_j_all_fw, _balance_j_all_ab = {}, {}
for occ in KEPT_13:
    _sorted = df_jd_eda[df_jd_eda["occ"] == occ].sort_values("jd_idx")
    _balance_j_all[occ] = _sorted["balance"].values
    for _column, _target in (("balance_cw", _balance_j_all_cw),
                             ("balance_fw", _balance_j_all_fw),
                             ("balance_AB", _balance_j_all_ab)):
        if _column in _sorted.columns:
            _target[occ] = _sorted[_column].values

_fit_test_rows = []

def _run_gap_jd_test(dens_label, dens_dict, expected_sign,
                     test_id="T1_gap_jd", status="primary", lexicon=None):
    print(f"\n  JD language test: gap_j ~ {dens_label}")
    print(f"  {'Occ':<22}  {'rho':>6}  {'p_rho':>7}  {'diff_hi_lo':>10}  {'p_diff':>7}")
    print("  " + "-" * 60)
    _delta_sub = {o: _gap_j_all[o] for o in KEPT_13
                  if o in _gap_j_all and o in dens_dict
                  and len(_gap_j_all[o]) == len(dens_dict[o])}
    _dens_sub  = {o: dens_dict[o] for o in _delta_sub}
    # A rank correlation needs at least three job descriptions per occupation.
    _too_few = [o for o in _delta_sub if len(_delta_sub[o]) < 3]
    if _too_few:
        print(f"  occupations with fewer than 3 job descriptions, left out: "
              f"{_too_few}")
    for occ in sorted(o for o in _delta_sub if o not in _too_few):
        _, _po = study2_core.jd_language_test(
            {occ: _delta_sub[occ]}, {occ: _dens_sub[occ]}, n_perm=N_PERM, seed=SEED)
        print(f"  {occ:<22}  {_po.get('rho',float('nan')):>+6.3f}  "
              f"{_po.get('p_rho',float('nan')):>7.4f}  "
              f"{_po.get('diff_hi_lo',float('nan')):>+10.3f}  "
              f"{_po.get('p_diff',float('nan')):>7.4f}")
        _fit_test_rows.append({"test_id":test_id,"status":status,
                                "lexicon":lexicon or LEX_MAIN,
                                "expected_sign":expected_sign,
                                "unit":"JD","y":"gap_f_minus_m","x":dens_label,"occ":occ,
                                "n":len(_delta_sub[occ]),
                                "rho":float(_po.get("rho",float("nan"))),
                                "p_rho":float(_po.get("p_rho",float("nan"))),
                                "diff_hi_lo":float(_po.get("diff_hi_lo",float("nan"))),
                                "p_diff":float(_po.get("p_diff",float("nan")))})
    if len(_delta_sub) >= 2:
        _tab, _pool = study2_core.jd_language_test(
            _delta_sub, _dens_sub, n_perm=N_PERM, seed=SEED)
        print(f"  {'POOLED':<22}  {_pool.get('rho',float('nan')):>+6.3f}  "
              f"{_pool.get('p_rho',float('nan')):>7.4f}  "
              f"{_pool.get('diff_hi_lo',float('nan')):>+10.3f}  "
              f"{_pool.get('p_diff',float('nan')):>7.4f}")
        _fit_test_rows.append({"test_id":"T1_gap_jd","status":"primary",
                                "expected_sign":expected_sign,
                                "unit":"JD","y":"gap_f_minus_m","x":dens_label,"occ":"POOLED",
                                "n":len(_delta_sub),"rho":float(_pool.get("rho",float("nan"))),
                                "p_rho":float(_pool.get("p_rho",float("nan"))),
                                "diff_hi_lo":float(_pool.get("diff_hi_lo",float("nan"))),
                                "p_diff":float(_pool.get("p_diff",float("nan"))),
                                "test_id":test_id,"status":status,
                                "lexicon":lexicon or LEX_MAIN})
    else:
        print(f"  pooled test left out: {len(_delta_sub)} occupations, 2 needed")

# Main test with the published main dictionary A, then the variants. The main
# test is the row with status "primary"; everything else is a variant.
_run_gap_jd_test("comm_dens", jd_comm_dens, "+")
_run_gap_jd_test("ag_dens",   jd_ag_dens,   "-")
if _balance_j_all:
    _run_gap_jd_test("balance", _balance_j_all, "+")

# A u B as the robustness variant.
_run_gap_jd_test("comm_dens_AB", jd_comm_dens_ab, "+",
                 test_id="T1_gap_jd", status="robustness", lexicon=LEX_ROBUST)
_run_gap_jd_test("ag_dens_AB", jd_ag_dens_ab, "-",
                 test_id="T1_gap_jd", status="robustness", lexicon=LEX_ROBUST)
if _balance_j_all_ab:
    _run_gap_jd_test("balance_AB", _balance_j_all_ab, "+",
                     test_id="T1_gap_jd", status="robustness", lexicon=LEX_ROBUST)

# The same test without the D1 function words (_cw) and on
# the part that was taken out (_fw).
_run_gap_jd_test("comm_dens_cw", jd_comm_dens_cw, "+",
                 test_id="T1_cw", status="robustness")
_run_gap_jd_test("ag_dens_cw", jd_ag_dens_cw, "-",
                 test_id="T1_cw", status="robustness")
if _balance_j_all_cw:
    _run_gap_jd_test("balance_cw", _balance_j_all_cw, "+",
                     test_id="T1_cw", status="robustness")
if _balance_j_all_fw:
    _run_gap_jd_test("balance_fw", _balance_j_all_fw, "+",
                     test_id="T1_fw", status="robustness")

# Reading fixed in advance: if the main test and _cw agree in sign
# and _cw is significant on its own, the result holds without pronouns and modal
# verbs. If only _fw carries it, the relation is reported as an effect of how the
# advertisement addresses the reader, not of agentic or communal word choice.
_t1 = pd.DataFrame(_fit_test_rows)
if not _t1.empty and "test_id" in _t1.columns:
    _pooled = _t1[_t1["occ"].isna()] if "occ" in _t1.columns else _t1
    for _label in ("T1_gap_jd", "T1_cw", "T1_fw"):
        _rows = _t1[(_t1["test_id"] == _label) & (_t1["x"].astype(str).str.startswith("balance"))]
        if _rows.empty:
            print(f"  {_label:<10} no balance rows; that variant of the "
                  f"densities is not in the table")
            continue
        print(f"  {_label:<10} balance rows: {len(_rows)}  "
              f"median rho {_rows['rho'].median():+.3f}")

# Occ-level Spearman: mean gap_j vs mean balance (n=13)
if len(KEPT_13) >= 10:
    from scipy.stats import spearmanr as _sp_gap
    _mg = [float(_gap_j_all[o].mean()) for o in KEPT_13 if o in _gap_j_all]
    _mb = [float(_balance_j_all[o].mean()) for o in KEPT_13 if o in _balance_j_all]
    if len(_mg) == len(_mb) and len(_mg) >= 5:
        _rho_g, _p_g = _sp_gap(_mg, _mb)
        print(f"\n  Occ-level Spearman: mean gap_j ~ mean balance (n={len(_mg)}): "
              f"rho={_rho_g:>+.3f}  p={_p_g:.4f}")

# The gap per job description is added to the job description table, which is
# written to disk after this section.
if df_jd_level.empty:
    raise AssertionError(
        "The job description table holds no rows, so the gap cannot be added to "
        "it; the job description section has to run before this one.")
_gap_flat = [{"occ":occ,"jd_idx":j,"gap_f_minus_m":float(g)}
             for occ in KEPT_13
             for j, g in enumerate(_gap_j_all[occ])]
_gap_df2 = pd.DataFrame(_gap_flat)
if "gap_f_minus_m" in df_jd_level.columns:
    df_jd_level = df_jd_level.drop(columns=["gap_f_minus_m"])
df_jd_level = df_jd_level.merge(_gap_df2, on=["occ","jd_idx"], how="left")
_no_gap = int(df_jd_level["gap_f_minus_m"].isna().sum())
print(f"\n  gap_f_minus_m added to the job description table: "
      f"{len(df_jd_level)} rows, {_no_gap} without a gap (expected 0)")
if _no_gap:
    raise AssertionError(
        f"{_no_gap} job descriptions have no gap. The two tables do not cover "
        f"the same job descriptions.")

# 3. Post-manipulation OLS
print(f"\n3. Post-manipulation: OLS pct_indiv ~ female + C(occ), HC1")
_ols_post_rows = []
if "female" not in df_pool.columns:
    raise AssertionError(
        "The pool table has no gender column, so the regression after the "
        "manipulation cannot be fitted.")
for arm in ("communion","agency"):
    _df_a = df_pool[df_pool["arm"]==arm].copy()
    _n_women_a = int((_df_a["female"] == 1).sum())
    _n_men_a   = int((_df_a["female"] == 0).sum())
    _n_occ_a   = int(_df_a["occ"].nunique())
    print(f"  control {arm:<10}: women {_n_women_a}, men {_n_men_a}, "
          f"occupations {_n_occ_a}")
    if _n_women_a < 2 or _n_men_a < 2 or _n_occ_a < 2:
        raise AssertionError(
            f"The regression for {arm} needs biographies of both genders in at "
            f"least two occupations; women {_n_women_a}, men {_n_men_a}, "
            f"occupations {_n_occ_a}.")
    _res = smf12l.ols("pct_indiv ~ female + C(occ)", data=_df_a).fit(cov_type="HC1")
    _b = float(_res.params["female"]); _se = float(_res.bse["female"])
    _p = float(_res.pvalues["female"]); _n = int(_res.nobs)
    print(f"  {arm:<10}: b_female={_b:>+.3f}  SE={_se:.3f}  p={_p:.3f}  n={_n}")
    _ols_post_rows.append({"arm":arm,"b_female":_b,"se":_se,"p":_p,"n":_n})

df_gender_gap     = pd.DataFrame(_gap_rows)
df_gender_gap_ols_post = pd.DataFrame(_ols_post_rows)
df_fit_tests      = pd.DataFrame(_fit_test_rows)

Gender gap in the original rankings of the pool

1. Bio-level: mean pct_orig by gender
  Occ                      mean_F   mean_M   diff_FM   n_f   n_m      MW_p
  ------------------------------------------------------------------------
  nurse                     49.69    53.93     -4.24   232    18    0.5436
  software_engineer         48.03    50.40     -2.37    42   208    0.6474
  accountant                51.56    49.11     +2.45    91   159    0.4937
  architect                 44.00    51.90     -7.90    60   190    0.0359
  psychologist              51.09    48.34     +2.74   151    99    0.4128
  attorney                  52.13    48.82     +3.31    89   161    0.3698
  teacher                   55.19    43.28    +11.91   141   109    0.0006
  surgeon                   50.52    49.92     +0.60    32   218    0.7884
  physician                 46.56    52.49     -5.92   105   145    0.0458
  professor                 53.73    47.07     +6.65   110   140    0.0318
  dentist    

### 14.14 Fit between biography and advertisement

- **Question:** Do communally written biographies rank relatively better
  against communally worded advertisements?
- **Expectation:** the slope over the advertisements grows with the balance of
  the biography.
- **Test:** biography; Spearman within occupation with a pooled permutation
  test.
- **Status:** pre-specified test T2 (fixed 2026-10-05), with the published main
  dictionary A; A $\cup$ B and the rows without the function words of D1 are
  variants.
- **Output:** `study2_fit_slopes.csv`, `study2_fit_tests.csv`

In [46]:
# Fit between biography and advertisement: the slope over the job
# descriptions, which is the pre-specified test T2.
print("Fit between biography and advertisement")

_slope_rows = []
for occ in KEPT_13:
    _so = study2_core.jd_similarity(
        emb_orig_pool[occ]["mean"], emb_jd[occ], how=JD_REPR)
    _P  = study2_core.pool_pct_per_jd(_so)           # (n_bios, n_jds)
    _pool_df = bio_pools[occ].reset_index(drop=True)
    if "female" not in _pool_df.columns:
        _pool_df["female"] = (_pool_df.get("gender_label","")=="Female").astype(int)
    if "communion_rate" not in _pool_df.columns:
        _pool_df["communion_rate"] = _pool_df["communion_count"]/_pool_df["word_count"]*100
    if "agency_rate" not in _pool_df.columns:
        _pool_df["agency_rate"] = _pool_df["agency_count"]/_pool_df["word_count"]*100
    _P_mat_all[occ] = _P

    # Get JD v_j vectors from EDA
    def _get_vj(name):
        _sub = df_jd_eda[df_jd_eda["occ"]==occ].sort_values("jd_idx")
        return _sub[name].values if name in _sub.columns else None

    _v_bal = _get_vj("balance")
    _v_com = _get_vj("comm_dens")
    _v_ag  = _get_vj("ag_dens")
    if _v_bal is None:
        _v_bal = jd_comm_dens[occ] - jd_ag_dens[occ]
        _v_com = jd_comm_dens[occ]
        _v_ag  = jd_ag_dens[occ]
    # Variants: without the D1 function words, only that part, and under A u B.
    _v_bal_cw = _get_vj("balance_cw")
    _v_bal_fw = _get_vj("balance_fw")
    _v_bal_ab = _get_vj("balance_AB")

    def _slope_v(v):
        vc = v - v.mean()
        denom = vc @ vc
        if denom < 1e-12: return np.zeros(_P.shape[0])
        return (_P - _P.mean(axis=1, keepdims=True)) @ vc / denom

    _s_bal = _slope_v(_v_bal)
    _s_com = _slope_v(_v_com)
    _s_ag  = _slope_v(_v_ag)
    _s_bal_cw = _slope_v(_v_bal_cw) if _v_bal_cw is not None else None
    _s_bal_fw = _slope_v(_v_bal_fw) if _v_bal_fw is not None else None
    _s_bal_ab = _slope_v(_v_bal_ab) if _v_bal_ab is not None else None

    # The slopes are centred, so each one has to average to zero.
    for _nm, _sv in [("s_bal",_s_bal),("s_comm",_s_com),("s_ag",_s_ag)]:
        if abs(_sv.mean()) > 1e-7:
            raise AssertionError(
                f"control mean slope {occ}/{_nm}: {_sv.mean():.2e} "
                f"(expected below 1e-7)")

    for i in range(len(_pool_df)):
        _r = _pool_df.iloc[i]
        _slope_rows.append({"occ":occ,"bio_idx":i,
                             "female":int(_r.get("female",float("nan"))),
                             "word_count":float(_r.get("word_count",float("nan"))),
                             "communion_rate":float(_r.get("communion_rate",float("nan"))),
                             "agency_rate":float(_r.get("agency_rate",float("nan"))),
                             "bio_balance":float(_r.get("communion_rate",0))-float(_r.get("agency_rate",0)),
                             "s_bal":float(_s_bal[i]),"s_comm":float(_s_com[i]),"s_ag":float(_s_ag[i]),
                             "communion_rate_AB":float(_r.get("communion_rate_AB",float("nan"))),
                             "agency_rate_AB":float(_r.get("agency_rate_AB",float("nan"))),
                             "bio_balance_AB":float(_r.get("bio_balance_AB",float("nan"))),
                             "s_bal_cw":float(_s_bal_cw[i]) if _s_bal_cw is not None else float("nan"),
                             "s_bal_fw":float(_s_bal_fw[i]) if _s_bal_fw is not None else float("nan"),
                             "s_bal_AB":float(_s_bal_ab[i]) if _s_bal_ab is not None else float("nan")})

df_fit_slopes = pd.DataFrame(_slope_rows)
# The biography rates without the D1 function words come from the function word
# section. They carry the robustness tests, so they are required here.
print(f"control rates without the function words: {len(df_bio_fw)} rows")
if df_bio_fw.empty:
    raise AssertionError(
        "The biography rates without the D1 function words are missing; the "
        "function word section has to run before this one.")
df_fit_slopes = df_fit_slopes.merge(df_bio_fw, on=["occ", "bio_idx"], how="left")

# Tests: y=s_bal, x=bio_balance (T2 main); also s_comm~comm_rate, s_ag~ag_rate, crossed
print(f"\n  T2 Main: y=s_bal ~ x=bio_balance (expected rho>0)")
_test_specs = [
    # Main test with the published main dictionary A (fixed on 2026-10-05;
    # Dictionary specified on 2026-10-10, before any result with A).
    ("T2_main",   "primary",  "s_bal",  "bio_balance",  "+"),
    ("T2_s_comm", "secondary","s_comm", "communion_rate","+"),
    ("T2_s_ag",   "secondary","s_ag",   "agency_rate",  "+"),
    ("T2_cross1", "secondary","s_ag",   "communion_rate","0"),
    ("T2_cross2", "secondary","s_comm", "agency_rate",  "0"),
    # A u B as the robustness variant.
    ("T2_main",   "robustness","s_bal_AB","bio_balance_AB","+"),
    # Without the D1 function words, and only that part.
    ("T2_cw",     "robustness","s_bal_cw","bio_balance_cw","+"),
    ("T2_fw",     "robustness","s_bal_fw","bio_balance_fw","+"),
]
print(f"  {'test':<12}  {'status':<10}  {'y':<8}  {'x':<16}  "
      f"{'rho':>6}  {'p_rho':>7}  {'diff':>8}  {'p_diff':>7}  {'exp':>4}")
print("  " + "-" * 85)
for _tid, _stat, _y, _x, _exp in _test_specs:
    _dy = {occ: df_fit_slopes[df_fit_slopes["occ"]==occ][_y].values for occ in KEPT_13}
    _dx = {occ: df_fit_slopes[df_fit_slopes["occ"]==occ][_x].values for occ in KEPT_13}
    _dy = {o: v for o,v in _dy.items() if len(v)>0}; _dx = {o:v for o,v in _dx.items() if o in _dy}
    if len(_dy) < 2:
        print(f"  {_tid:<12}  left out: {len(_dy)} occupations with slopes, "
              f"2 needed")
        continue
    _tab2, _pool2 = study2_core.jd_language_test(_dy, _dx, n_perm=N_PERM, seed=SEED)
    _rho2 = float(_pool2.get("rho",float("nan")))
    _p2   = float(_pool2.get("p_rho",float("nan")))
    _dhl  = float(_pool2.get("diff_hi_lo",float("nan")))
    _pdhl = float(_pool2.get("p_diff",float("nan")))
    print(f"  {_tid:<12}  {_stat:<10}  {_y:<8}  {_x:<16}  "
          f"{_rho2:>+6.3f}  {_p2:>7.4f}  {_dhl:>+8.3f}  {_pdhl:>7.4f}  {_exp:>4}")
    df_fit_tests = pd.concat([df_fit_tests,
        pd.DataFrame([{"test_id":_tid,"status":_stat,"unit":"Bio",
                       "expected_sign":_exp,
                       "y":_y,"x":_x,"occ":"POOLED","n":len(_dy),
                       "rho":_rho2,"p_rho":_p2,
                       "diff_hi_lo":_dhl,"p_diff":_pdhl}])],
        ignore_index=True)

# Robustness: residualize on log_wc before testing T2
print(f"\n  T2 Robustness: residualize on log_wc first")
import statsmodels.formula.api as smf12m
_res_slopes = df_fit_slopes.copy()
_res_slopes["log_wc"] = np.log(_res_slopes["word_count"].clip(lower=1))
# A residual needs at least five biographies and a length that varies. Where
# that does not hold the occupation has no residual, and how many biographies
# that concerns is printed.
for _v in ("s_bal","bio_balance"):
    _res = []
    _left_out = []
    for occ in KEPT_13:
        _sub = _res_slopes[_res_slopes["occ"]==occ]
        if len(_sub) < 5 or _sub["log_wc"].nunique() < 2:
            _left_out.append((occ, len(_sub), int(_sub["log_wc"].nunique())))
            _res.extend([float("nan")]*len(_sub))
            continue
        _fit = smf12m.ols(f"{_v} ~ log_wc", data=_sub).fit()
        _res.extend(list(_fit.resid))
    if _left_out:
        print(f"  {_v}: occupations without residuals "
              f"(occupation, biographies, different lengths): {_left_out}")
    _res_slopes[f"{_v}_res"] = _res
_dy_r = {occ: _res_slopes[_res_slopes["occ"]==occ]["s_bal_res"].dropna().values for occ in KEPT_13}
_dx_r = {occ: _res_slopes[_res_slopes["occ"]==occ]["bio_balance_res"].dropna().values for occ in KEPT_13}
_dy_r = {o:v for o,v in _dy_r.items() if len(v)>3}; _dx_r = {o:v for o,v in _dx_r.items() if o in _dy_r}
if len(_dy_r) >= 2:
    _, _pool_r = study2_core.jd_language_test(_dy_r, _dx_r, n_perm=N_PERM, seed=SEED)
    print(f"  T2_resid: rho={_pool_r.get('rho',float('nan')):>+.3f}  "
          f"p_rho={_pool_r.get('p_rho',float('nan')):.4f}")
else:
    print(f"  T2_resid left out: {len(_dy_r)} occupations with residuals, "
          f"2 needed")
print(f"\n  df_fit_slopes: {len(df_fit_slopes)} rows")

Fit between biography and advertisement
control rates without the function words: 3250 rows

  T2 Main: y=s_bal ~ x=bio_balance (expected rho>0)
  test          status      y         x                    rho    p_rho      diff   p_diff   exp
  -------------------------------------------------------------------------------------
  T2_main       primary     s_bal     bio_balance       +0.123   0.0002    +0.186   0.0002     +
  T2_s_comm     secondary   s_comm    communion_rate    +0.175   0.0002    +0.324   0.0002     +
  T2_s_ag       secondary   s_ag      agency_rate       +0.132   0.0002    +0.238   0.0002     +
  T2_cross1     secondary   s_ag      communion_rate    +0.022   0.2132    +0.040   0.3195     0
  T2_cross2     secondary   s_comm    agency_rate       +0.047   0.0060    +0.131   0.0004     0
  T2_main       robustness  s_bal_AB  bio_balance_AB    +0.112   0.0002    +0.132   0.0002     +
  T2_cw         robustness  s_bal_cw  bio_balance_cw    +0.132   0.0002    +0.246   0.00

### 14.15 Fit by gender

- **Question:** Does the fit differ between biographies of women and of men,
  and how much of that sits in the word shares?
- **Test:** biography; OLS with occupation effects, HC1.
- **Status:** pre-specified side analysis, fixed 2026-10-05.
- **Output:** `study2_fit_gender.csv`

In [47]:
# Does the fit differ between women and men? OLS as a side analysis.
import statsmodels.formula.api as smf12n
print("Fit by gender")
print(f"control slope table rows: {len(df_fit_slopes)} (expected more than 0)")
if df_fit_slopes.empty:
    raise AssertionError(
        "The slope table is empty, so the fit cannot be split by gender; the "
        "fit section has to run first.")
_n_women_fs = int((df_fit_slopes["female"] == 1).sum())
_n_men_fs   = int((df_fit_slopes["female"] == 0).sum())
_n_occ_fs   = int(df_fit_slopes["occ"].nunique())
print(f"control biographies with a slope: women {_n_women_fs}, men {_n_men_fs}, "
      f"occupations {_n_occ_fs} (at least 2 of each expected)")
if _n_women_fs < 2 or _n_men_fs < 2 or _n_occ_fs < 2:
    raise AssertionError(
        f"The models need biographies of both genders in at least two "
        f"occupations; women {_n_women_fs}, men {_n_men_fs}, occupations "
        f"{_n_occ_fs}.")

_fs = df_fit_slopes.copy()
_fs["log_wc"] = np.log(_fs["word_count"].clip(lower=1))
_ols12n_rows = []

# Model 1: s_bal ~ female + C(occ)
_m1 = smf12n.ols("s_bal ~ female + C(occ)", data=_fs).fit(cov_type="HC1")
_b1 = float(_m1.params["female"]); _se1 = float(_m1.bse["female"])
_ci1 = _m1.conf_int().loc["female"]
print(f"  Model 1 (s_bal ~ female + C(occ)): "
      f"b={_b1:>+.4f}  SE={_se1:.4f}  95%CI=[{float(_ci1[0]):>+.4f}, {float(_ci1[1]):>+.4f}]  "
      f"p={float(_m1.pvalues['female']):.4f}  n={int(_m1.nobs)}")
_ols12n_rows.append({"model":"m1","y":"s_bal","predictors":"female+C(occ)",
                      "b_female":_b1,"se":_se1,"ci_lo":float(_ci1[0]),"ci_hi":float(_ci1[1]),
                      "p":float(_m1.pvalues["female"]),"n":int(_m1.nobs)})

# Model 2: s_bal ~ female + communion_rate + agency_rate + log_wc + C(occ)
_no_rate = [_c for _c in ("communion_rate", "agency_rate")
            if _c not in _fs.columns or _fs[_c].isna().all()]
if _no_rate:
    raise AssertionError(
        f"The second model needs the biography rates; these are missing or "
        f"empty: {_no_rate}.")
_m2 = smf12n.ols("s_bal ~ female + communion_rate + agency_rate + log_wc + C(occ)",
                  data=_fs).fit(cov_type="HC1")
_b2 = float(_m2.params["female"]); _se2 = float(_m2.bse["female"])
_ci2 = _m2.conf_int().loc["female"]
_b_comm = float(_m2.params["communion_rate"])
_b_ag   = float(_m2.params["agency_rate"])
if _b1 == 0.0:
    raise AssertionError(
        "The coefficient of the first model is exactly zero, so the share by "
        "which the second model reduces it is not defined.")
_reduction = 1.0 - _b2 / _b1
print(f"  Model 2 (+ rates + log_wc): "
      f"b_female={_b2:>+.4f}  SE={_se2:.4f}  p={float(_m2.pvalues['female']):.4f}")
print(f"  b_communion_rate={_b_comm:>+.4f}  b_agency_rate={_b_ag:>+.4f}")
print(f"  1 - b_female(M2)/b_female(M1) = {_reduction:>.3f}")
_ols12n_rows.append({"model":"m2","y":"s_bal",
                      "predictors":"female+comm_rate+ag_rate+log_wc+C(occ)",
                      "b_female":_b2,"se":_se2,"ci_lo":float(_ci2[0]),"ci_hi":float(_ci2[1]),
                      "p":float(_m2.pvalues["female"]),"n":int(_m2.nobs),
                      "b_communion_rate":_b_comm,"b_agency_rate":_b_ag,
                      "reduction_b_female":_reduction})

# Model 3: after the manipulation, on the changed biographies of the gate.
print(f"\n  Post-manipulation s_bal (gate={PRIMARY_GATE}):")
for arm in ("communion","agency"):
    _dg = df_pool[(df_pool["arm"]==arm) & (df_pool["n_subs"]>=1)]
    if PRIMARY_GATE is not None:
        _dg = _dg[_dg["sim_cp"] >= PRIMARY_GATE]
    _no_emb = [occ for occ in KEPT_13 if emb_manip_pool[occ].get(arm) is None]
    if _no_emb:
        raise AssertionError(
            f"No manipulated embeddings for {arm} in {_no_emb}; the embedding "
            f"section has to run before this one.")
    _s_cf_col = []
    for occ in KEPT_13:
        _so = study2_core.jd_similarity(
            emb_orig_pool[occ]["mean"], emb_jd[occ], how=JD_REPR)
        _sm = study2_core.jd_similarity(
            emb_manip_pool[occ][arm]["mean"], emb_jd[occ], how=JD_REPR)
        _Pcf = study2_core.indiv_cf_pct_per_jd(_so, _sm)
        _vr = df_jd_eda[df_jd_eda["occ"]==occ].sort_values("jd_idx")
        _v_bal = _vr["balance"].values
        _vc = _v_bal - _v_bal.mean(); _dn = _vc @ _vc
        if _dn < 1e-12:
            raise AssertionError(
                f"The balance of the job descriptions of {occ} does not vary, "
                f"so no slope can be formed on it.")
        _s_cf = (_Pcf - _Pcf.mean(axis=1,keepdims=True)) @ _vc / _dn
        for bi in _dg[_dg["occ"]==occ]["bio_idx"].astype(int):
            _s_cf_col.append({"occ":occ,"bio_idx":bi,"arm":arm,
                               "s_bal_cf":float(_s_cf[bi])})
    _cf_df = pd.DataFrame(_s_cf_col)
    if _cf_df.empty:
        raise AssertionError(
            f"No biography of {arm} passes the gate of {PRIMARY_GATE}, so the "
            f"fit after the manipulation cannot be formed.")
    _merged = _dg[["occ","bio_idx","female"]].merge(_cf_df, on=["occ","bio_idx"])
    _merged_all = _merged.merge(df_fit_slopes[["occ","bio_idx","s_bal"]], on=["occ","bio_idx"])
    _n_women_m = int((_merged["female"] == 1).sum())
    _n_men_m   = int((_merged["female"] == 0).sum())
    _n_occ_m   = int(_merged["occ"].nunique())
    print(f"  control {arm:<10}: women {_n_women_m}, men {_n_men_m}, "
          f"occupations {_n_occ_m}")
    if _n_women_m < 2 or _n_men_m < 2 or _n_occ_m < 2:
        raise AssertionError(
            f"The model for {arm} needs biographies of both genders in at least "
            f"two occupations; women {_n_women_m}, men {_n_men_m}, occupations "
            f"{_n_occ_m}.")
    _m3 = smf12n.ols("s_bal_cf ~ female + C(occ)", data=_merged).fit(cov_type="HC1")
    _m3b = smf12n.ols("s_bal ~ female + C(occ)", data=_merged_all).fit(cov_type="HC1")
    print(f"  {arm}: s_bal_cf: b_female={float(_m3.params['female']):>+.4f}  "
          f"p={float(_m3.pvalues['female']):.4f}  n={int(_m3.nobs)}")
    print(f"  {arm}: s_bal (same sel.): b_female={float(_m3b.params['female']):>+.4f}  "
          f"p={float(_m3b.pvalues['female']):.4f}")
    # Median change per gender. The signed rank test needs changes that are not
    # all zero; where they are, the test is left out and the reason is printed.
    _merged_all["delta_s"] = (_merged.set_index(["occ","bio_idx"])["s_bal_cf"]
                                .reindex(_merged_all.set_index(["occ","bio_idx"]).index)
                                .values - _merged_all["s_bal"].values)
    for _f_val, _f_lbl in [(1,"women"),(0,"men")]:
        _vals = _merged_all[_merged_all["female"]==_f_val]["delta_s"].dropna()
        if len(_vals) < 5:
            print(f"    {arm}/{_f_lbl}: test left out, {len(_vals)} changes, "
                  f"5 needed")
            continue
        if not (_vals != 0).any():
            print(f"    {arm}/{_f_lbl}: test left out, all {len(_vals)} changes "
                  f"are exactly zero")
            continue
        _, _wp = wilcoxon(_vals)
        print(f"    {arm}/{_f_lbl}: median deltas_bal={float(_vals.median()):>+.4f}  "
              f"p_wilcoxon={_wp:.4f}  n={len(_vals)}")
    _ols12n_rows.append({"model":f"m3_{arm}","y":"s_bal_cf",
                          "predictors":"female+C(occ)",
                          "b_female":float(_m3.params["female"]),
                          "se":float(_m3.bse["female"]),
                          "p":float(_m3.pvalues["female"]),"n":int(_m3.nobs)})

# Control: the slope of the gap on the balance has to equal the difference of
# the mean slopes of women and men, occupation by occupation.
if "gap_f_minus_m" not in df_jd_level.columns:
    raise AssertionError(
        "The job description table has no gap column, so the control cannot be "
        "run; the gender gap section has to run before this one.")
print(f"\n  Control 2: polyfit gap_j ~ balance_j vs occ mean(s_bal_F - s_bal_M):")
_max_ctrl2 = 0.0
_ctrl2_left_out = []
for occ in KEPT_13:
    _jdl_occ = df_jd_level[df_jd_level["occ"]==occ].sort_values("jd_idx")
    _gj = _jdl_occ["gap_f_minus_m"].values
    _bj = df_jd_eda[df_jd_eda["occ"]==occ].sort_values("jd_idx")["balance"].values
    if len(_gj) != len(_bj) or len(_gj) < 3:
        _ctrl2_left_out.append((occ, len(_gj), len(_bj)))
        continue
    _slope_pf = float(np.polyfit(_bj, _gj, 1)[0])
    _fs_occ = df_fit_slopes[df_fit_slopes["occ"]==occ]
    _diff_means = float(_fs_occ[_fs_occ["female"]==1]["s_bal"].mean() -
                          _fs_occ[_fs_occ["female"]==0]["s_bal"].mean())
    _diff_ctrl = abs(_slope_pf - _diff_means)
    if _diff_ctrl > _max_ctrl2: _max_ctrl2 = _diff_ctrl
if _ctrl2_left_out:
    print(f"    occupations left out (occupation, gaps, balances): "
          f"{_ctrl2_left_out}")
print(f"    max |polyfit - diff_means| over occs: {_max_ctrl2:.2e}  (expected <1e-8)")
if _max_ctrl2 > 1e-8:
    raise AssertionError(
        f"The slope of the gap on the balance and the difference of the mean "
        f"slopes differ by up to {_max_ctrl2:.2e}; the two ways of forming the "
        f"same quantity do not agree.")

df_fit_gender_ols = pd.DataFrame(_ols12n_rows)
print(f"\n  df_fit_gender_ols: {len(df_fit_gender_ols)} rows")

Fit by gender
control slope table rows: 3250 (expected more than 0)
control biographies with a slope: women 1587, men 1663, occupations 13 (at least 2 of each expected)
  Model 1 (s_bal ~ female + C(occ)): b=+0.0971  SE=0.0308  95%CI=[+0.0367, +0.1576]  p=0.0016  n=3250
  Model 2 (+ rates + log_wc): b_female=+0.0829  SE=0.0303  p=0.0062
  b_communion_rate=+0.0384  b_agency_rate=+0.0091
  1 - b_female(M2)/b_female(M1) = 0.147

  Post-manipulation s_bal (gate=0.9):
  control communion : women 994, men 876, occupations 13
  communion: s_bal_cf: b_female=+0.0380  p=0.3797  n=1870
  communion: s_bal (same sel.): b_female=+0.0443  p=0.3066
    communion/women: median deltas_bal=-0.0102  p_wilcoxon=0.0000  n=994
    communion/men: median deltas_bal=-0.0030  p_wilcoxon=0.4523  n=876
  control agency    : women 490, men 594, occupations 13
  agency: s_bal_cf: b_female=+0.0840  p=0.1317  n=1084
  agency: s_bal (same sel.): b_female=+0.0781  p=0.1559
    agency/women: median deltas_bal=-0.0034  p

## 15. Arm 2

- **Question:** Does the ranking effect hold with a search encoder that
  takes a task instruction instead of a general sentence encoder?
- **Test:** biography; the same medians and tests on embeddings produced on a
  GPU, with the gate of Arm 1 so the same biographies are compared.
- **Status:** robustness, fixed 2026-10-11. The embeddings come from
  `06a_arm2_embeddings_kaggle.ipynb`; Arm 1 runs without them.
- **Output:** `study2_delta_r5_pool_arm2.csv`, `study2_arm2_tests.csv`,
  `study2_arm2_rank_correlation.csv`

In [48]:
# Arm 2: read the embeddings from Kaggle and rank the same bios again.
# The bio selection follows the mpnet gate, so the two arms compare the same
# biographies and only the ranking changes.
_arm2_dir = Path(ARM2_DIR)
_arm2_bio = _arm2_dir / "arm2_bio_embeddings.parquet"
_arm2_jd  = _arm2_dir / "arm2_jd_embeddings.parquet"
_arm2_meta = _arm2_dir / "arm2_metadata.json"

# Arm 2 is a second encoder, not a part of the main result. Whether its
# embeddings are in place is recorded here, and the saving cell writes the three
# Arm 2 tables only when they are.
ARM2_PRESENT = bool(_arm2_bio.exists() and _arm2_jd.exists())
print(f"control Arm 2 embeddings in {_arm2_dir}: {ARM2_PRESENT}")
if not ARM2_PRESENT:
    print("Run 06a_arm2_embeddings_kaggle.ipynb on Kaggle and copy "
          "arm2_bio_embeddings.parquet, arm2_jd_embeddings.parquet and "
          "arm2_metadata.json into that folder. Arm 1 is complete without them.")
else:
    _meta = json.loads(_arm2_meta.read_text(encoding="utf-8")) \
            if _arm2_meta.exists() else {}
    print(f"model          : {_meta.get('model')}")
    print(f"revision       : {_meta.get('revision')}")
    print(f"substitute used: {_meta.get('substitute_used')}")
    print(f"weight dtype   : {_meta.get('weight_dtype')}  "
          f"stored dtype: {_meta.get('stored_dtype')}")
    print(f"max_seq_length : {_meta.get('max_seq_length')}  "
          f"longest text: {_meta.get('longest_text_tokens')} tokens")
    print(f"truncated texts: {_meta.get('n_truncated')} (expected 0)")
    if _meta.get("n_truncated"):
        raise AssertionError(
            f"{_meta['n_truncated']} texts were truncated in Arm 2; the "
            f"requires whole job descriptions.")

    _bio_emb = pd.read_parquet(_arm2_bio)
    _jd_emb  = pd.read_parquet(_arm2_jd)
    print(f"biography embeddings: {len(_bio_emb)}  job description embeddings: "
          f"{len(_jd_emb)}")

    _bio_lookup = {(r.occ, int(r.bio_idx), r.version): np.asarray(r.embedding)
                   for r in _bio_emb.itertuples()}

    # The embeddings were produced for the full pool. Every biography of the
    # pool used here has to be among them, matched by occupation and position.
    _not_embedded = [(occ, i) for occ in KEPT_13
                     for i in range(len(bio_pools[occ]))
                     if (occ, i, "original") not in _bio_lookup]
    print(f"control pool biographies without an embedding: "
          f"{len(_not_embedded)} (expected 0)")
    if _not_embedded:
        raise AssertionError(
            f"{len(_not_embedded)} biographies of the pool have no Arm 2 "
            f"embedding, for example {_not_embedded[:5]}. The embeddings belong "
            f"to a different pool; produce them again for this one.")
    _arm2_rows, _corr_rows = [], []
    for occ in KEPT_13:
        _jd_occ = _jd_emb[_jd_emb["occ"] == occ].sort_values("jd_idx")
        if _jd_occ.empty:
            raise AssertionError(
                f"The Arm 2 file holds no job description of {occ}, so the two "
                f"arms would not compare the same job descriptions.")
        _jd_matrix = np.vstack([np.asarray(e) for e in _jd_occ["embedding"]])
        _n_bio = len(bio_pools[occ])
        _orig = np.vstack([_bio_lookup[(occ, i, "original")]
                           for i in range(_n_bio)])
        _sim_orig = _orig @ _jd_matrix.T
        _pct_orig = study2_core.percentile_ranks(_sim_orig)
        for arm in ("communion", "agency"):
            _manip = []
            _present = []
            for i in range(_n_bio):
                key = (occ, i, arm)
                if key in _bio_lookup:
                    _manip.append(_bio_lookup[key]); _present.append(i)
                else:
                    _manip.append(_orig[i]); _present.append(i)
            _manip = np.vstack(_manip)
            _sim_manip = _manip @ _jd_matrix.T
            _pct_indiv = study2_core.individual_cf_ranks(_sim_orig, _sim_manip)
            _delta = _pct_indiv - _pct_orig
            _dz = study2_core.delta_z_per_jd(_sim_orig, _sim_orig,
                                             _sim_manip).mean(axis=1)
            for i in range(_n_bio):
                _arm2_rows.append({
                    "occ": occ, "bio_idx": i, "arm": arm, "encoder": "arm2",
                    "female": int(bio_pools[occ].iloc[i]["female"]),
                    "delta": float(_delta[i]), "delta_z": float(_dz[i]),
                    "pct_orig": float(_pct_orig[i]),
                    "pct_indiv": float(_pct_indiv[i]),
                    "sim_cp": float(cp_sims_pool[occ][arm][i]),
                })
            # Rank correlation of the original ranks and of the deltas between arms.
            _a1 = pd.DataFrame(records_r5_pool)
            _a1 = _a1[(_a1["occ"] == occ) & (_a1["arm"] == arm)].sort_values("bio_idx")
            if len(_a1) == _n_bio:
                _rho_orig, _p_orig = spearmanr(_a1["pct_orig"], _pct_orig)
                _rho_delta, _p_delta = spearmanr(_a1["delta"], _delta)
                _corr_rows.append({"occ": occ, "arm": arm,
                                   "rho_pct_orig": float(_rho_orig),
                                   "p_pct_orig": float(_p_orig),
                                   "rho_delta": float(_rho_delta),
                                   "p_delta": float(_p_delta)})
    df_arm2_delta = pd.DataFrame(_arm2_rows)
    df_arm2_rank_corr = pd.DataFrame(_corr_rows)

    print()
    print(f"{'arm':<10} {'n':>6} {'median_delta':>13} {'share_up':>9} {'p_wilcoxon':>11}")
    print("-" * 54)
    _test_rows = []
    for arm in ("communion", "agency"):
        _s = df_arm2_delta[df_arm2_delta["arm"] == arm]
        _gated = _s[_s["sim_cp"] >= PRIMARY_GATE]
        for label, frame in (("gate", _gated), ("all", _s)):
            if frame.empty:
                print(f"{arm:<10} no biography in the subset {label}, left out")
                continue
            # The helper returns the p value alone, and not a number when
            # fewer than ten biographies moved at all.
            _p = study2_core._safe_wilcoxon(frame["delta"])
            if not np.isfinite(_p):
                print(f"{arm:<10} subset {label}: fewer than 10 biographies "
                      f"moved, so the signed rank test is not defined")
            print(f"{arm:<10} {len(frame):>6} {frame['delta'].median():>13.3f} "
                  f"{(frame['delta'] > 0).mean():>9.1%} {_p:>11.4g}  ({label})")
            _test_rows.append({"encoder": "arm2", "arm": arm, "subset": label,
                               "n": len(frame),
                               "median_delta": float(frame["delta"].median()),
                               "share_up": float((frame["delta"] > 0).mean()),
                               "median_delta_z": float(frame["delta_z"].median()),
                               "p_wilcoxon": float(_p)})
    df_arm2_tests = pd.DataFrame(_test_rows)

    if not df_arm2_rank_corr.empty:
        print()
        print("Rank correlation between Arm 1 and Arm 2, per occupation:")
        print(df_arm2_rank_corr.to_string(index=False))

control Arm 2 embeddings in outputs\study2_run5\arm2: True
model          : Qwen/Qwen3-Embedding-0.6B
revision       : 97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3
substitute used: False
weight dtype   : torch.bfloat16  stored dtype: float32
max_seq_length : 3294  longest text: 3230 tokens
truncated texts: 0 (expected 0)
biography embeddings: 6221  job description embeddings: 2975
control pool biographies without an embedding: 0 (expected 0)

arm             n  median_delta  share_up  p_wilcoxon
------------------------------------------------------
communion    3242         0.000     26.5%   1.341e-05  (gate)
communion    3250         0.000     26.6%   2.783e-05  (all)
agency       3241         0.000     16.4%      0.1717  (gate)
agency       3250         0.000     16.4%      0.1543  (all)

Rank correlation between Arm 1 and Arm 2, per occupation:
              occ       arm  rho_pct_orig   p_pct_orig  rho_delta      p_delta
            nurse communion      0.621950 3.703254e-28   0.45571

## 16. Run 5 against Run 4

- **Question:** How far do the results move between dictionary A with list
  v2 and the A $\cup$ B variant with list v1?
- **Test:** biography; pooled medians and the share that moves up or down, per
  arm and sample.
- **Status:** descriptive, fixed 2026-10-11. Run 4 is read from its saved files
  and never overwritten.
- **Output:** `study2_run5_vs_run4.csv`

In [49]:
# Run 5 against Run 4: pooled medians, share up and down, per arm and sample.
_rows = []
for sample, run5_records, run4_file in (
        ("pool", records_r5_pool, f"{RUN4_BASE}/study2_delta_r4_pool.csv"),
        ("focal", records_r5_focal, f"{RUN4_BASE}/study2_delta_r4_focal.csv")):
    _path = Path(run4_file)
    if not _path.exists():
        raise FileNotFoundError(
            f"{run4_file} is missing, so Run 5 cannot be compared with Run 4. "
            f"The comparison belongs to the run, so this is not skipped.")
    _r4s = pd.read_csv(_path)
    _r5s = pd.DataFrame(run5_records)
    for arm in ("communion", "agency"):
        _a5 = _r5s[_r5s["arm"] == arm]
        _a4 = _r4s[_r4s["arm"] == arm]
        for label, f5, f4 in (("gate", _a5[_a5["sim_cp"] >= PRIMARY_GATE],
                               _a4[_a4["sim_cp"] >= PRIMARY_GATE]),
                              ("all", _a5, _a4)):
            if f5.empty or f4.empty:
                print(f"  {sample}/{arm}/{label}: Run 5 has {len(f5)} rows, "
                      f"Run 4 has {len(f4)}; left out")
                continue
            _rows.append({
                "sample": sample, "arm": arm, "subset": label,
                "n_run5": len(f5), "n_run4": len(f4),
                "median_delta_run5": float(f5["delta"].median()),
                "median_delta_run4": float(f4["delta"].median()),
                "share_up_run5": float((f5["delta"] > 0).mean()),
                "share_up_run4": float((f4["delta"] > 0).mean()),
                "share_down_run5": float((f5["delta"] < 0).mean()),
                "share_down_run4": float((f4["delta"] < 0).mean()),
                "median_n_subs_run5": float(f5["n_subs"].median())
                                      if "n_subs" in f5.columns else np.nan,
                "median_n_subs_run4": float(f4["n_subs"].median())
                                      if "n_subs" in f4.columns else np.nan,
            })
df_run5_vs_run4 = pd.DataFrame(_rows)
if df_run5_vs_run4.empty:
    raise AssertionError(
        "The comparison with Run 4 holds no rows although both files were "
        "read; the two runs have no subset in common.")
print(df_run5_vs_run4[["sample", "arm", "subset", "n_run5", "n_run4",
                       "median_delta_run5", "median_delta_run4",
                       "share_up_run5", "share_up_run4"]].to_string(index=False))
print()
print("Run 5 uses dictionary A and substitution list v2; Run 4 uses A u B and "
      "list v1, so the two differ in both the dictionary and the manipulation.")

sample       arm subset  n_run5  n_run4  median_delta_run5  median_delta_run4  share_up_run5  share_up_run4
  pool communion   gate    3242    3185           0.000000             0.0000       0.338063       0.377394
  pool communion    all    3250    3250           0.000000             0.0000       0.338769       0.377231
  pool    agency   gate    3241    3147           0.000000             0.0000       0.183277       0.306641
  pool    agency    all    3250    3250           0.000000             0.0000       0.184000       0.301231
 focal communion   gate    3169    3091           0.591892             0.0880       0.590092       0.515044
 focal communion    all    3250    3250           0.617891             0.0392       0.592000       0.505846
 focal    agency   gate    3195    3050           0.092800             0.0000       0.522379       0.459344
 focal    agency    all    3250    3250           0.073600             0.0000       0.518154       0.442154

Run 5 uses dictionary A and

## 17. Provenance of the embeddings

How many texts of each part of the analysis were read from the cache and how many were computed in this run. The cache key holds the text, the model, the prefix and the width of the embeddings, so an entry is only read when all four match.

In [50]:
# Where the embeddings of this run come from. A cache entry is only read when
# the text, the model, the prefix and the width of the stored key all match, so
# A text that was read from the cache rests on the same arrays as before and a
# text that was computed rests on this run.
_cache_log = pd.DataFrame(CACHE_LOG)
if _cache_log.empty:
    raise AssertionError(
        "No embedding was read or computed in this run, so no result rests on "
        "any array; the embedding sections have to run before this one.")

def _cache_part(tag):
    """The part of the analysis that a cache tag belongs to."""
    if tag.startswith("cachetest"):
        return "cache test"
    if tag.startswith("regress_v1_ab"):
        return "regression test against Run 4"
    if tag.endswith("_pool_orig"):
        return "pool, original"
    if "_focal_orig" in tag:
        return "focal sample, original"
    if "manip_pool" in tag or "pool_manip" in tag:
        return "pool, manipulated"
    if "manip_focal" in tag or "focal_manip" in tag:
        return "focal sample, manipulated"
    if "_jds" in tag:
        return "job descriptions"
    return "not assigned"

_cache_log["part"] = [_cache_part(t) for t in _cache_log["tag"]]
_cache_sum = (_cache_log.groupby(["part", "source"])
                        .agg(calls=("tag", "size"), texts=("n_texts", "sum"))
                        .reset_index()
                        .sort_values(["part", "source"]))
print("Provenance of the embeddings")
print(f"  {'part of the analysis':<32} {'source':<9} {'calls':>6} {'texts':>8}")
print("  " + "-" * 60)
for _, _row in _cache_sum.iterrows():
    print(f"  {_row['part']:<32} {_row['source']:<9} "
          f"{int(_row['calls']):>6} {int(_row['texts']):>8}")

_from_cache = int(_cache_log[_cache_log["source"] == "cache"]["n_texts"].sum())
_computed = int(_cache_log[_cache_log["source"] == "computed"]["n_texts"].sum())
print()
print(f"texts read from the cache: {_from_cache}")
print(f"texts computed in this run: {_computed}")
print(f"models written to: {sorted(_cache_log['model'].unique())}")

# A tag that no part claims would be left out of the table above, so it is named.
_not_assigned = sorted(set(_cache_log[_cache_log["part"] == "not assigned"]["tag"]))
print(f"tags that no part of the analysis claims: {len(_not_assigned)}")
for _tag in _not_assigned:
    print(f"  {_tag}")

Provenance of the embeddings
  part of the analysis             source     calls    texts
  ------------------------------------------------------------
  cache test                       computed       2       10
  focal sample, manipulated        cache          6     1500
  focal sample, manipulated        computed      46    11500
  focal sample, original           cache          6     1500
  focal sample, original           computed      46    11500
  job descriptions                 cache         28     6277
  job descriptions                 computed      11     2648
  pool, manipulated                cache          6     1500
  pool, manipulated                computed      46    11500
  pool, original                   cache          2      500
  pool, original                   computed      24     6000

texts read from the cache: 11277
texts computed in this run: 43158
models written to: ['sentence-transformers/all-mpnet-base-v2', 'sentence-transformers/all-mpnet-base-v2_plai

## 18. Save outputs

Every table and figure of this run is written to the output folder.

In [51]:
# Write every table of this run. The tables are listed by name: a name that no
# section has filled stops the run, an empty table stops it too, and at the end
# A control lists every file that was written with its number of rows.
from pathlib import Path
import json as _json, platform
from sentence_transformers import __version__ as _st_ver
import numpy as _nv, pandas as _pdv, scipy as _scv, spacy as _spv

Path(OUT_BASE).mkdir(parents=True, exist_ok=True)

# One result lives in a dictionary rather than in a table; it becomes one here.
df_pct_female = pd.DataFrame({"occ": list(PCT_FEMALE.keys()),
                              "pct_female": list(PCT_FEMALE.values())})

# File, object, and when the file belongs to a run:
#   always  every run writes it
#   full    the full run only; the test run does not fill that section
#   arm2    only when the Arm 2 embeddings are in place
TABLES = [
    ("study2_delta_r5_pool.csv",            "df_pool",                    "always"),
    ("study2_delta_r5_focal.csv",           "df_focal",                   "always"),
    ("study2_pool_focal_comparison.csv",    "df_pool_focal_cmp",          "always"),
    ("study2_sensitivity.csv",              "df_sensitivity",             "always"),
    ("study2_sensitivity_gate_by_sample.csv", "df_sensitivity_gate_sample", "always"),
    ("study2_ols.csv",                      "df_ols",                     "always"),
    ("study2_cp_gate.csv",                  "df_cp_gate",                 "always"),
    ("study2_pct_female.csv",               "df_pct_female",              "always"),
    ("study2_jd_lengths.csv",               "df_jd_lengths",              "always"),
    ("study2_loo_crossval.csv",             "df_loo_all",                 "always"),
    ("study2_trunc_vs_full.csv",            "df_trunc_vs_full",           "always"),
    ("study2_word_overlap.csv",             "df_word_overlap",            "always"),
    ("study2_word_overlap_bio.csv",         "df_word_overlap_bio",        "always"),
    ("study2_by_word_class.csv",            "df_word_class",              "always"),
    ("study2_jd_origin.csv",                "df_jd_origin",               "always"),
    ("study2_jd_eda.csv",                   "df_jd_eda",                  "always"),
    ("study2_jd_eda_by_occupation.csv",     "df_jd_eda_by_occ",           "always"),
    ("study2_jd_eda_tests.csv",             "df_jd_eda_tests",            "always"),
    ("study2_jd_top_lexicon_words.csv",     "df_jd_top_words",            "always"),
    ("study2_jd_joint_model.csv",           "df_jd_joint_model",          "always"),
    ("study2_fit_bio_slopes.csv",           "df_fit_slopes",              "always"),
    ("study2_fit_gender_ols.csv",           "df_fit_gender_ols",          "always"),
    ("study2_fit_tests.csv",                "df_fit_tests",               "always"),
    ("study2_gender_gap.csv",               "df_gender_gap",              "always"),
    ("study2_gender_gap_ols_post.csv",      "df_gender_gap_ols_post",     "always"),
    # The job description table carries gap_f_minus_m from the gender gap
    # section, so it is written once, after that section has run.
    ("study2_jd_level.csv",                 "df_jd_level",                "always"),
    # Substitution list v2.
    ("coverage_v2.csv",                     "df_coverage_v2",             "always"),
    ("substitution_log_v2.csv",             "df_occurrences",             "always"),
    ("focal_selection_v2.csv",              "df_focal_selection",         "always"),
    ("manipulation_check.csv",              "df_manip_check",             "always"),
    ("manip_quintiles.csv",                 "df_manip_quintiles",         "always"),
    ("manip_spearman.csv",                  "df_manip_spearman",          "always"),
    ("study2_function_words_bios.csv",      "df_function_words_summary",  "always"),
    ("control_original_ranks.csv",          "df_ctrl_ranks",              "full"),
    ("study2_run5_vs_run4.csv",             "df_run5_vs_run4",            "always"),
    ("control_jd_densities.csv",            "df_ctrl_jd_dens",            "full"),
    ("study2_delta_r5_pool_arm2.csv",       "df_arm2_delta",              "arm2"),
    ("study2_arm2_tests.csv",               "df_arm2_tests",              "arm2"),
    ("study2_arm2_rank_correlation.csv",    "df_arm2_rank_corr",          "arm2"),
]

_in_this_run = {"always": True, "full": not SMOKE, "arm2": ARM2_PRESENT}
_wanted = [(_file, _name) for _file, _name, _group in TABLES if _in_this_run[_group]]
print(f"tables that belong to this run: {len(_wanted)} of {len(TABLES)}  "
      f"(test run: {SMOKE}, Arm 2 present: {ARM2_PRESENT})")

_never_built = [_name for _, _name in _wanted if _name not in globals()]
if _never_built:
    raise AssertionError(
        f"These tables were never built: {_never_built}. The sections that fill "
        f"them have to run before this cell.")
_no_rows = [_name for _, _name in _wanted if globals()[_name].empty]
if _no_rows:
    raise AssertionError(
        f"These tables hold no rows: {_no_rows}. An empty table is not a result, "
        f"so the run stops here rather than writing an empty file.")

written = {}
for _file, _name in _wanted:
    globals()[_name].to_csv(f"{OUT_BASE}/{_file}", index=False, encoding="utf-8")
    written[_file] = len(globals()[_name])

# The treatment and the gender tables come as a dictionary, one file per key.
for _label, _frames in (("treatment", treatment_tables), ("gender", gender_frames)):
    if not _frames:
        raise AssertionError(
            f"No {_label} tables; the section that builds them has to run first.")
    for _key, _frame in _frames.items():
        if _frame.empty:
            raise AssertionError(f"The {_label} table {_key} holds no rows.")
        _file = f"study2_{_label}_{_key}.csv"
        _frame.to_csv(f"{OUT_BASE}/{_file}", index=False, encoding="utf-8")
        written[_file] = len(_frame)

# The job description language test: the pooled summary and the table per
# occupation of every test.
if not jd_test_results:
    raise AssertionError(
        "No job description language tests; the section that runs them has to "
        "run first.")
_pool_rows, _tab_rows = [], []
for _key, _result in jd_test_results.items():
    _row = {"test_key": _key}
    _row.update(_result["pool"])
    _pool_rows.append(_row)
    _tab = _result["tab"]
    if _tab.empty:
        raise AssertionError(
            f"The table per occupation of the test {_key} holds no rows.")
    _tab = _tab.copy()
    _tab.insert(0, "test_key", _key)
    _tab_rows.append(_tab)
_pooled = pd.DataFrame(_pool_rows)
_pooled.to_csv(f"{OUT_BASE}/study2_jd_language_test.csv", index=False,
               encoding="utf-8")
written["study2_jd_language_test.csv"] = len(_pooled)
_per_occ = pd.concat(_tab_rows, ignore_index=True)
_per_occ.to_csv(f"{OUT_BASE}/study2_jd_language_test_per_occ.csv", index=False,
                encoding="utf-8")
written["study2_jd_language_test_per_occ.csv"] = len(_per_occ)

# The manifest records the settings of the run and the versions of the libraries.
manifest = {
    "smoke": SMOKE, "primary_model": PRIMARY_MODEL, "run_tag": RUN_TAG,
    "max_seq_length": int(main_model.max_seq_length),
    "encode_long_window": int(main_model.max_seq_length - 2 - 4),
    "jd_repr": JD_REPR, "primary_gate": PRIMARY_GATE, "gates": GATES,
    "focal_n": FOCAL_N, "focal_min_count": FOCAL_MIN_COUNT,
    "focal_rate_quantile": FOCAL_RATE_QUANTILE, "n_perm": N_PERM, "seed": SEED,
    "n_bios": N_BIOS_FULL, "n_jds": N_JDS_FULL, "random_state": RANDOM_STATE,
    "kept_13": KEPT_13,
    "lexicon_main": LEX_MAIN, "lexicon_robustness": LEX_ROBUST,
    "manipulation": MANIP, "rules_file": RULES_FILE,
    "fixed_terms_file": FIXED_TERMS_FILE, "validation_file": VALIDATION_FILE,
    "arm2_present": ARM2_PRESENT,
    "pool_records_total": len(df_pool), "focal_records_total": len(df_focal),
    "python_version": platform.python_version(),
    "packages": {"sentence_transformers": _st_ver, "numpy": _nv.__version__,
                 "pandas": _pdv.__version__, "scipy": _scv.__version__,
                 "spacy": _spv.__version__},
}
with open(f"{OUT_BASE}/run_manifest.json", "w", encoding="utf-8") as _f:
    _json.dump(manifest, _f, indent=2)

# Control: every file that was written lies on disk and holds rows.
print()
print(f"  {'file':<50} {'rows':>7}  {'KB':>8}")
print("  " + "-" * 70)
_failed = []
for _file in sorted(written):
    _path = Path(OUT_BASE) / _file
    if not _path.exists() or _path.stat().st_size == 0 or written[_file] == 0:
        _failed.append(_file)
        continue
    print(f"  {_file:<50} {written[_file]:>7} {_path.stat().st_size / 1024:>8.1f}")
_manifest_path = Path(OUT_BASE) / "run_manifest.json"
print(f"  {'run_manifest.json':<50} {'':>7} "
      f"{_manifest_path.stat().st_size / 1024:>8.1f}")
if _failed or not _manifest_path.exists():
    raise AssertionError(
        f"Missing or empty after writing: {_failed}; manifest present: "
        f"{_manifest_path.exists()}.")
print()
print(f"files written: {len(written)} tables and the manifest, in {OUT_BASE}/")

# Files that this cell did not write. They come from an earlier run or from a
# script, and they are listed so that the folder is not read as one run.
_other = sorted(p.name for p in Path(OUT_BASE).glob("*.csv") if p.name not in written)
print(f"other CSV files in the folder: {len(_other)}")
for _name in _other:
    print(f"  {_name}")

tables that belong to this run: 39 of 39  (test run: False, Arm 2 present: True)

  file                                                  rows        KB
  ----------------------------------------------------------------------
  control_jd_densities.csv                                13      1.1
  control_original_ranks.csv                              13      1.3
  coverage_v2.csv                                          2      0.5
  focal_selection_v2.csv                                  26      1.0
  manip_quintiles.csv                                     10      0.8
  manip_spearman.csv                                       2      0.1
  manipulation_check.csv                               13000   1731.0
  study2_arm2_rank_correlation.csv                        26      2.6
  study2_arm2_tests.csv                                    4      0.4
  study2_by_word_class.csv                                46      3.6
  study2_cp_gate.csv                                      52      2.8
  st